# Same-Time Sky-to-Science Coefficient Transfer with a Physically Split Continuum

**Variant status.** This notebook is the split-zodi branch of the deployed
symmetric dual-encoder group-head regressor. The corpus has been re-decomposed
with `sky_decomp.SkyDecompLSFSurfaceIterative(split_zodi=True)`, and the ML
model learns to predict the resulting two-family continuum representation.
See the identifiability notebook
`notebooks/moon_zodi_split_identifiability.ipynb` for the decomposition-side
design and validation.

## Problem statement

Every science exposure taken with LVM at Las Campanas is accompanied by two
simultaneous sky-arm measurements — a SKY_NEAR fibre bundle pointed a few
degrees from the science IFU and a SKY_FAR bundle pointed several degrees
further away.  The two auxiliary pointings exist because almost every LVM
science target is a low-surface-brightness Galactic diffuse emitter for
which the sky is a first-order contaminant, and the goal of the sky-subtraction
pipeline is to remove that contaminant at the pixel level.

The sky is however not spatially uniform on the two-arm-to-science baseline of
five to ten degrees.  Scattered moonlight has strong angular structure driven
by the Krisciunas–Schaefer scattering phase function; the zodiacal continuum
swings by tens of percent across the ecliptic longitude range spanned by three
pointings on the same night; the OH airglow bands are modulated by mesospheric
gravity waves on scales of tens of arcminutes.  A raw copy-of-nearest-arm
sky subtraction therefore leaves systematics of order one to ten percent of
the sky flux — enough to bury the extended emission that motivates the
survey.

This notebook trains an ML transfer that takes the two sky-arm spectra
(already decomposed by the physics-driven pipeline of Chapter 1 into per-emitter
coefficient vectors) plus the row's astrometry, moon geometry, ecliptic
geometry, solar-activity indices and observing time, and predicts what the
sky spectrum would have looked like at the science IFU location at the
exposure's midpoint.  Reconstructing that predicted sky and subtracting it
pixel-by-pixel from the science exposure is what closes the sky-subtraction
loop.

---

# Chapter 1 — Decompositions

Chapter 1 turns each row's raw sky spectrum into a compact, physics-grounded
per-row coefficient vector that Chapter 3's ML transfer consumes.  The
decomposition solves one QP per row against a fixed physical dictionary of
moon, zodi, airglow and continuum templates, and persists both the
coefficients and their joint covariance.

## 1.1 Goal and pipeline

LVM's science fibres and its two sky arms (SKY_NEAR, SKY_FAR) sample three
lines of sight simultaneously. Each row (fibre × exposure) is fit
**independently** by the same physical model to yield a coefficient vector
$\mathbf{c}\in\mathbb{R}^{P}$ with $P=388$ (see §1.3). The fit is a
constrained quadratic program (QP) driven by the observed flux, its photon
inverse-variance and curvature regularisers — the weighting became real only on
2026-09-18, and §1.7 corrects the record on what it was before.  The
deployed corpus is `skysub/gaia-stars-mask-telluric-chi2-1.3.2/` (DRP 1.3.2,
18 668 rows, 13 688 after the §1.11 filters) with suffix
`_palace_aijc_vnf_split_zodi_lsf_spline2d`, set by `cfg.data.decomp_data_root`
and `cfg.data.decomp_suffix`; earlier runs used `skysub/gaia-stars-mask-telluric/`
(DRP 1.2.1), `skysub/gaia-stars-mask-cont2/` (same continuum model, no telluric
term), and before that `skysub/new-oh-3/` and `skysub/moon_zodi_spline/`.

The corpus we consume here is produced by

```
python skysub/decompose_parallel.py \
    lvmsframe_median_stack_1.3.2_gaia1over100.fits \
    sky_decomp/data \
    --fit-model palace-aijc-vnf-split-zodi-lsf-spline2d \
    --n-spline-knots 11 --n-zodi-spline-knots 1 \
    --n-refinement-cycles 5 --n-workers 32 \
    --output-dir gaia-stars-mask-telluric-chi2-1.3.2/
```

**Defaults changed on 2026-09-24, for the next corpus.** Run without options,
`decompose_parallel` now fits the ridge-corrected OH line strengths
(`--fit-model palacecorr-aijc-vnf-split-zodi-lsf-spline2d`, §1.2.3) and anchors
the zodi to the empirically corrected Leinert model
(`SPLIT_ZODI_ZODI_CORRECTION = "lvm-ecl-2026-09"`, §1.4.3 (c)). The command
above names its `--fit-model` explicitly. Reproducing the deployed corpus also
needs `LVMSKY_ZODI_CORRECTION=none` in the environment.

**Two fit variants are live**, and `mlp_predictor.data.DECOMP_VARIANTS` holds
both.  The original non-telluric `lsf-surface-iterative-split-zodi` fit (discrete
11-tap LSF surface, suffix `_lsf_surface_iterative_split_zodi`) was retired on
2026-09-18 and is no longer reachable from the CLI or the ML side.

| `--fit-model` | suffix | what it adds |
|---|---|---|
| `palace-aijc-vnf-split-zodi-lsf-spline2d` | `_palace_aijc_vnf_split_zodi_lsf_spline2d` | **telluric absorption fitted simultaneously** (§1.2.3), continuous M-spline 2-D LSF (§1.6) |
| `palacecorr-aijc-vnf-split-zodi-lsf-spline2d` | `_palacecorr_aijc_vnf_split_zodi_lsf_spline2d` | the telluric fit with **ridge-corrected OH line strengths** (§1.2.3); `decompose_parallel`'s default from 2026-09-24. `DECOMP_VARIANT = 'telluric-palacecorr'`. |

Both produce the SAME 388 coefficient names in the same order, so the ML stage
is drop-in — but they differ in their OH line file, so every reconstruction
must use the variant's own (the variant entry carries it; see §1.2.3).  Two behaviours that were added on 2026-09-18 are ON by default and can be
turned off for reproducing an older corpus: the photon-noise weighting
(`--no-fit-pixel-weights`, §1.7) and retry-on-reversal (`--no-reversal-retry`,
§1.12).

**Identifiability constraints.** Every split-zodi fit model applies five
constraints, set in `decompose_parallel.SPLIT_ZODI_*` and requiring no CLI
flags: adjacent-knot ratio bounds on both spline families
(`moon_ratio_bound = zodi_ratio_bound = 0.7`), a bracket on the moon's share
of the moon+zodi continuum (`amp_prior_tol = 3`), and an absolute bracket on
$\int\!{\rm zodi}$ against the Leinert $B_{500}$ prediction
(`zodi_amp_bound = 2`, recentred by `SPLIT_ZODI_ZODI_PRIOR_CALIBRATION = 1.6`
and, from 2026-09-24, reshaped by the empirical ecliptic correction
`SPLIT_ZODI_ZODI_CORRECTION`, §1.4.3 (c)), plus the two diffuse-block
constraints of §1.4.4 (species-ratio bracket and moon-gated cap against OH).
The moon-share and zodi brackets are per-row geometry predictions installed by
`set_amplitude_prior`, computed from moon altitude and airmass, moon–target
separation, lunar phase and the target's helioecliptic coordinates via
`sky_decomp.moon_zodi_model.geometry_amplitude_prior`.  Section 1.4.3 gives
the formulation; the changelog records the measurements that set the values.

## 1.2 Physical model

The decomposition treats the sky as a non-negative linear combination of a
handful of physical carriers whose spectral shapes are fixed by physics or
by pre-fit lookup tables; only the per-row amplitudes are free parameters.

The decomposition writes each row's spectrum $y(\lambda)$ as a non-negative
sum of physical templates:

$$
y(\lambda) \;\approx\; \sum_k c_k\; T_k(\lambda),\qquad c_k\ge 0,
$$

where $T_k$ is the template of family $k$ convolved with the row's fitted
2-D LSF kernel (§1.6), attenuated as appropriate for its emitting layer and
divided by the DRP transmission (§1.2.3).

### 1.2.1 Moon carrier (15 basis functions)

Scattered moonlight is the dominant broadband contaminant on bright-moon
nights.  Its spectrum is the solar SED reddened through the lunar-albedo
envelope; the free parameters are the 15 B-spline amplitudes that let the
overall shape drift smoothly with the row's specific scattering geometry.

$$
T_{\rm moon,\,j}(\lambda) \;=\; I_\odot(\lambda)\; \alpha_{\rm ROLO}(\lambda,\,\phi_0{=}30°)\; B_j^{(K_{\rm m}=11)}(\lambda),
$$

with $I_\odot$ the solar SED (Meftah 2018 UV–NIR reference), $\alpha_{\rm ROLO}$
the disk-integrated lunar albedo at a fiducial phase angle, and
$B_j^{(K_{\rm m})}$ a cubic B-spline basis with $K_{\rm m}=11$ interior knots
(15 columns).  This resolution eliminates all
adjacent-knot |corr| > 0.99 pairs. The ROLO envelope carries sharp mineral
absorption bands, giving the moon carrier a distinctive spectral shape.

### 1.2.2 Zodi carrier (5 basis functions)

The zodiacal light is a smooth reddened solar continuum; separating it from
the moon carrier lets the ML head learn its ecliptic-geometry dependence
independently from moon geometry.  Pre-split, the smooth zodi component was
silently absorbed into `Moon_bs` on moon-down + low-$|\beta_{\rm ecl}|$ rows.

$$
T_{\rm zodi,\,j}(\lambda) \;=\; I_\odot(\lambda)\; (\lambda/5000\,\text{\AA})^{0.26}\; B_j^{(K_{\rm z}=1)}(\lambda),
$$

with $K_{\rm z}=1$ interior knot (5 columns) and $0.26$ the Leinert-style
zodiacal reddening exponent. The 1-knot spline is heavily
curvature-penalised ($\lambda_{\rm z}=0.1$, §1.4) so the zodi carrier is
nearly monotone in $\lambda$. Introducing this family (`split_zodi=True`)
is the central decomposition-side change of this variant: pre-split, the
smooth zodi
component was silently absorbed into `Moon_bs` on moon-down + low-$|\beta_{\rm ecl}|$
rows, giving `c_moon` a mixed identity that the ML head could not learn.

### 1.2.3 OH Meinel bands

OH nightglow is the dominant narrow-line airglow signal at all wavelengths
beyond ~6000 Å.  LVM's resolution resolves individual rotational
transitions, and each carries its own free amplitude so that the fit can track
mesospheric temperature and gravity-wave-driven line-ratio changes.

OH transitions from the PALACE population model (`pmd_popmodel_OH*`) are
grouped by upper state $(v_{\rm upper}, N_{\rm upper}, F_{\rm upper})$ into
357 stick spectra $L_i(\lambda)$, convolved with the row LSF. Coefficients are
`OH_000`–`OH_356`. Together with the single O$_2$ b-band coefficient they form
the 358-coefficient mesospheric group.

**Telluric absorption in the fit** (`palace-aijc-vnf-split-zodi-lsf-spline2d`).
OH emits at 85–90 km, so every OH line is seen THROUGH the whole lower
atmosphere and is attenuated before it reaches the fibre — mostly by H$_2$O and
O$_2$ bands, which sit on top of the Meinel bands in the NIR and vary with
airmass and precipitable water vapour.  Fitting undimmed templates leaves the
line amplitudes to absorb that transmission, which is exactly the quantity the
ML stage is asked to transfer: the OH block then carries atmospheric
attenuation dressed as mesospheric emission, and the corruption is
airmass-dependent, so it does not average out.

The variant fixes this by dividing EVERY family matrix by that row's DRP
transmission before the solve, so the QP sees the templates as they actually
arrive and the coefficients stay physical:

$$\mathbf{A}^{\rm row} = \mathbf{A} \, / \, T_{\rm drp}(\lambda \,;\,
X_{\rm row}, {\rm PWV}_{\rm row}).$$

The OH sticks are grouped by $(v_{\rm upper}, N_{\rm upper}, F_{\rm
upper})$ rather than by rotational level alone.  Consequences, all measured:

* **Decomposition $\chi^2$ falls ~30% full-band**, with the blue essentially
  flat — as expected, since the telluric bands the correction removes are all
  in the NIR where OH lives.  All 357 OH centroids are NIR in both variants.
* The design matrix becomes **per-row** and can no longer be hoisted out of
  the row loop (~0.17 s/row/arm to rebuild; ~4 min for a 500-row sample across
  three arms), and the LSF is stored as a continuous density rather than an
  11-tap surface, so the iterative class cannot read it at all (§1.6).
* Integrated amplitudes $A_g = \mathbf{c}_g\cdot\mathbf{v}_g$ keep the
  TELLURIC-FREE template integrals in both flavours: the coefficient multiplies
  the same physical template either way, and the transmission is a per-row
  correction on top.
* **Coefficient-space metrics are NOT comparable across the variants** even
  though the names are identical — `mean_eRMSE` is an absolute per-coefficient
  RMSE and the mesospheric-coefficient gain can flip sign against the same
  errors measured in flux space.  A/B the two variants in FLUX space.

**Ridge-corrected line strengths**
(`palacecorr-aijc-vnf-split-zodi-lsf-spline2d`, the default from 2026-09-24).
Same telluric fit, same $(v, N, F)$ grouping and the same 388 coefficients.
Only the relative PALACE line strengths inside each group change. They come
from `pmd_popmodel_OH_skyfar_linear_ridge_0p1_v1.dat`, a linear correction
fitted with ridge $\lambda = 0.1$ on SkyFar spectra. The primary header
records `OHFILE` and `OHRIDGE`. On a 1000-row A/B against the deployed corpus:

* **Decomposition $\chi^2$ on OH-dominated pixels falls ×0.030 in r and
  ×0.28 in z**, and every row improves. Band-wide it is ×0.51 (r), ×0.65 (z)
  and ×0.69 (full); blue and OH-free pixels are unchanged.
* The systematic residual at the OH lines, stacked over rows, falls from 1.1%
  to 0.14% of the line peak in r and from 0.39% to 0.11% in z.
* **The coefficients barely move**: OH coefficients change by a median of 0
  and 0.8% at $p_{90}$. The template flux moves instead: OH flux is +0.11 dex
  in r, +1% in b and −1% in z. The old r-band lines share coefficients with
  the brighter z-band lines, so the fit could not correct their strengths.
* The continuum partition and every reliability flag are unchanged.

A reconstruction must use the same OH file as the fit. The coefficients are
nearly identical, so a mismatch does not show up in coefficient space. It
shows up only in flux.

### 1.2.4 Diffuse continuum templates

Three fixed diffuse templates $D_1,D_2,D_3$ (HO2, FeO, O2Ac) come from
`pmd_refcont`. They carry the flat, wavelength-smooth part of the continuum
so `Moon_bs` and `Zodi_bs` compete only for curvature-sensitive shape.

**Which `pmd_refcont`, and why (2026-09-10).** The deployed table is
`pmd_refcont_canonhyb_v1.dat`: canonical PALACE v1.0 `fcHO2` and `fcFeO`
interpolated onto the native LVM grid, with `fcO2Ac` taken verbatim from the
earlier native-LVM refit. PALACE v1.0 is Noll et al. (2025, GMD 18, 4353) and
its continuum components are quantified in Noll et al. (2024, ACP 24, 1143).

The previous table refit all three shapes on LVM sky
(`_joint_native_adam_invsky_p2_10000iter`, and its own header calls it
"experimental … not a full PALACE table"). That refit had moved HO2 from the
blue tail of a 1.51 µm feature to a 595 nm peak, i.e. onto FeO's own
signature, so the two species were effectively swapped relative to the
published identification. Restoring the canonical pair fixes the attribution:
measured on every10, the 580–610 nm peak as a fraction of airglow in
500–720 nm goes HO2 6.84% → **0.22%** and FeO 2.73% → **8.06%**, and FeO now
peaks at 5966 Å exactly where Noll puts FeO(VIS).

`fcO2Ac` is deliberately *not* canonical. Canonical O2Ac peaks at 3220 Å,
outside the LVM band, so only its tail is in range and it runs ≈2× high
through 4200–5900 Å. On ten far-arm dark off-ecliptic rows the fully canonical
table costs ×1.43 in blue $\chi^2$ and biases the median residual to
$-0.26\sigma$; the hybrid recovers that to $-0.02\sigma$ and has the best
full-band $\chi^2$ of the three variants.

**A caveat that is not resolved.** PALACE is calibrated for Cerro Paranal and
LVM observes from LCO, ~1000 km away, so PALACE's *absolute* levels are
indicative rather than binding here. In dark time — where there is no moon to
leak — the diffuse block still sits at 7.78% of the 500–720 nm airglow against
Noll's 3.3 ± 0.8%, a factor 2.4 that no basis choice and no constraint below
removes. Either LCO differs, or something that is not airglow lives in this
block.

### 1.2.5 Molecular O2 and atomic lines

The O$_2$ atmospheric b-band (8645 Å) is pre-fitted on its own; the resulting
per-row band shape (`VECTOR_O2`) then enters the main fit as a single template
with one free amplitude, `O2_b01`. Seven atomic lines enter as single
templates: the mesopause lines K I 7699 (`ATOM_K`), Na I D (`ATOM_Na`) and
[O I] 5577 (`ATOM_Og`), and the F-region lines N I 5199 (`ATOM_N`),
[O I] 6300/6364 (`ATOM_Or`), O I 7774 (`ATOM_Orc_OI0777`) and O I 8446
(`ATOM_Orc_OI0845`).

## 1.3 Coefficient block and design matrix

Concatenating the templates gives the row-level design matrix
$\mathbf{A}\in\mathbb{R}^{N_\lambda\times P}$:

$$
\mathbf{A} \;=\; [\, \mathbf{A}_{\rm oh}\ |\ \mathbf{A}_{\rm moon}\ |\ \mathbf{A}_{\rm zodi}\ |\ \mathbf{A}_{\rm diff}\ |\ \mathbf{A}_{\rm atom}\ |\ \mathbf{A}_{\rm orc}\ |\ \mathbf{A}_{\rm o2}\,].
$$

For this variant $N_\lambda=12401$ pixels on 3600–9800 Å at $\Delta\lambda=0.5$ Å,
and $P=388$ coefficients, of which the OH block is 357 columns:

| block          | count | code family    | physical driver                           |
|:---------------|------:|:---------------|:------------------------------------------|
| OH Meinel      |   357 | `OH_ddd`        | mesospheric temperature, gravity waves     |
| O2 b band      |     1 | `O2_b01`        | O2 atmospheric band (per-row pre-fit shape) |
| Moon spline    |    15 | `Moon_bs\d+`    | $g_{\rm moon}(\phi, h, r_{\rm sep})$      |
| Zodi spline    |     5 | `Zodi_bs\d+`    | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$·airmass |
| Diffuse        |     3 | HO2, FeO, O2Ac  | mesospheric chemistry residual            |
| Mesopause atoms|     3 | `ATOM_K`, `ATOM_Na`, `ATOM_Og` | K I 7699, Na I D, [O I] 5577 |
| F-region atoms |     4 | `ATOM_N`, `ATOM_Or`, `ATOM_Orc_OI0777`, `ATOM_Orc_OI0845` | N I 5199, [O I] 6300/6364, O I 7774, O I 8446 |

Deployed knot counts: `n_spline_knots=11` for moon and
`n_zodi_spline_knots=1` for zodi, auto-inferred from `coef_names` by the
`infer-spline-knots` notebook cell.

## 1.4 QP formulation

For each row, the fit solves a **weighted non-negative regularised
least-squares** problem:

$$
\boxed{\;\;\mathbf{c}^\star \;=\; \arg\min_{\mathbf{c}\ge 0}\;\; \underbrace{\tfrac{1}{2}\big\lVert \operatorname{diag}(\sqrt{\mathbf{w}})\big(\mathbf{A}\mathbf{c}-\mathbf{y}\big)\big\rVert_2^2}_{\text{data likelihood}} \;+\; \underbrace{\lambda_{\rm m}\lVert \mathbf{D}^{(2)}_{\rm m}\mathbf{c}_{\rm m}\rVert_2^2 \;+\;\lambda_{\rm z}\lVert \mathbf{D}^{(2)}_{\rm z}\mathbf{c}_{\rm z}\rVert_2^2}_{\text{spline curvature}}.\;\;}
$$

Symbols:

- $\mathbf{y}\in\mathbb{R}^{N_\lambda}$: the observed spectrum, scaled by
  `FACTOR = 1e14` (to bring counts to O(1)).
- $\mathbf{w} = 1/\sigma_{\rm pix}^2$: pixel inverse-variance from the
  reduction pipeline (`IVAR`, `SKY_IVAR`), with a relative floor
  `PIXEL_SIGMA_FLOOR_REL` per HDU.
- $\mathbf{D}^{(2)}$: second-difference operator on adjacent spline
  coefficients; suppresses ripples/oscillations in the moon and zodi splines.
- $\mathbf{c}\ge 0$: hard non-negativity constraint on every column
  (every family is physically non-negative).

The deployed curvature strengths (per unit scaled flux $y'=y/d$, see §1.5) are

$$
\lambda_{\rm m}=\lambda_{\rm z}=10^{-1}
$$

(`SPLIT_ZODI_CONTINUUM_DEFAULTS`; the bare `SkyDecomp` default of
$\lambda_{\rm m}=10^{-3}$ is not what the corpus uses). With a single interior
zodi knot the penalty keeps the zodi multiplier a smooth, nearly monotone
departure from the power law.

### 1.4.1 Solver

The problem is convex and is delegated to **Clarabel** (interior-point QP,
sparse triangular Hessian). The active-set structure at the optimum is used
downstream to compute coefficient uncertainties (§1.9).

### 1.4.2 Interline reweighting

The fit optionally reweights pixels near strong OH lines by
`moon_interline_weight ∈ [0, 1]` to prevent line residuals from stealing
signal from `Moon_bs`. This is disabled in the deployed split-zodi
configuration but the option remains in `fit.py`.

### 1.4.3 Identifiability constraints on the moon/zodi split

The two continuum families are nearly degenerate. Both are built on the same
solar template, so the fit can trade flux between them almost freely, and the
data term alone does not pin the split: the *sum* is well constrained while the
*division* is not. Measured on 200 lunation-stratified sky spectra, the
unconstrained fit put the two colours in the **wrong order on 164 of 168
moon-up spectra** — fitted moon log-log slope $+0.21$ where the physics gives
$-3.7$, fitted zodi $-4.04$ where the physics gives $-0.3$. It also gave the
moon family 45% of the continuum with the moon 37° *below* the horizon, and
produced a "zodi" whose amplitude tracked lunar illumination at
$\rho=+0.95$ while retaining only $\rho=0.09$ of its Leinert $B_{500}$
dependence. The downstream ML was therefore being handed moon-labelled zodi
spectra and vice versa.

Three families of constraint fix this. All are **hard linear inequalities**
appended to the same non-negative cone as $\mathbf{c}\ge 0$, not penalties:
they never bid against the likelihood, and the first two are homogeneous in
$\mathbf{c}$ and so carry no dependence on the flux scale.

**(a) Adjacent-knot ratio bounds — fix the colours.** For each family
$f\in\{\rm m, z\}$ and each adjacent pair of spline coefficients,

$$
\beta_f\,c^{(f)}_k \;\le\; c^{(f)}_{k+1} \;\le\; \beta_f^{-1} c^{(f)}_k ,
\qquad \beta_{\rm m}=\beta_{\rm z}=0.7 .
$$

This caps how fast a family's multiplier may vary with wavelength, so neither
family can manufacture the other's colour. With $K$ basis functions the total
permitted swing across the band is $\beta^{-(K-1)}$, so the *same* $\beta$ is
looser the more knots there are — these values are calibrated for 15 moon and
5 zodi basis functions and must be re-validated if the knot counts change.
The bounds also forbid a spline going to zero mid-band and re-emerging, which
is how the unconstrained fit reached a lower residual: it used the moon family
as piecewise scratch space, collapsing it to $\sim 0$ across 5100–6900 Å.
Note the interaction with non-negativity: since $c\ge0$ already holds, a ratio
bound makes each block either all-positive or identically zero.

**(b) Moon-share bracket — fixes dark time.** With
$u=\int\!{\rm moon}\,{\rm d}\lambda$ and $v=\int\!{\rm zodi}\,{\rm d}\lambda$,
both linear in $\mathbf{c}$,

$$
f_{\rm lo} \;\le\; \frac{u}{u+v} \;\le\; f_{\rm hi},
\qquad\Longleftrightarrow\qquad
\begin{cases}
(1-f_{\rm hi})\,u - f_{\rm hi}\,v \le 0\\[2pt]
f_{\rm lo}\,v - (1-f_{\rm lo})\,u \le 0
\end{cases}
$$

centred on the geometry-predicted fraction $f_{\rm pred}$ with a
multiplicative tolerance $\kappa_f=3$, clipped into
$[\epsilon,\,1-\epsilon]$ with $\epsilon=0.02$. Constraining the *fraction*
rather than either amplitude is what makes this calibration-free: both
predictions leave the physical model through the same solid-angle and flux
conversion, so the ratio is independent of throughput. The $\epsilon$ floor
guards the two places the prediction is least trustworthy — the below-horizon
suppression is a learned extrapolation fitted only to $-7°$ altitude, and ROLO
phase coverage runs out on thin crescents.

**(c) Absolute Leinert anchor — fixes the amplitude geometry.**

$$
\kappa_z^{-1} Z_{\rm pred} \;\le\; v \;\le\; \kappa_z\,Z_{\rm pred},
\qquad \kappa_z = 2 .
$$

The bracket in (b) is *relative*, so it can only divide whatever total the two
splines hold; it cannot stop both growing with the moon. Only an absolute
anchor reaches that. Its empirical licence is dark time: with the shape bounds
on and no moon, fitted/predicted $\int\!{\rm zodi}$ is $0.94$ with $0.16$ dex
scatter, so the literature scale is good to ~6% median with a factor-1.4
spread — do not tighten $\kappa_z$ below that spread. These are the only rows
with a non-zero right-hand side, hence stated in native flux units:
`set_amplitude_prior` must be given $Z_{\rm pred}$ on the same scale as the
flux being fitted.

**Empirical correction to the Leinert profile (2026-09-24, on by default).**
$Z_{\rm pred}$ now carries a multiplicative correction:

$$
Z_{\rm pred} = Z_{\rm Leinert}\;10^{f(u,v)},\qquad
f = \sum_{(i,j)} c_{ij}\,P_i(u)\,P_j(v),\qquad
u = \frac{2\,|\lambda-\lambda_\odot|}{180^\circ} - 1,\quad v = 2\sin|\beta| - 1,
$$

with $P$ the Legendre polynomials of degree $\le 2$ (8 terms,
`ZODI_LEINERT_CORRECTIONS["lvm-ecl-2026-09"]` in
`sky_decomp/moon_zodi_model.py`). $|\lambda-\lambda_\odot|$ is the true
geocentric-ecliptic longitude from the Sun, computed inside the physics model.

* **What was wrong.** For LVM the Leinert table falls off too steeply with
  ecliptic latitude: it is too bright on the ecliptic and too faint toward the
  poles. It therefore overstated the zodi contrast between neighbouring
  pointings. The slope of fitted against model sci-arm differences was
  0.45 (near) and 0.43 (far).
* **How it was fitted.** On 14 979 dark pointings (all three arms pooled),
  where the fit sets `Zodi_bs` freely. Rows on the $\pm\log_{10}\kappa_z$
  bracket were treated as censored (Tobit likelihood). The ecliptic shape was
  fitted jointly with a Galactic-latitude term and an airmass term, and only
  the ecliptic part is deployed. `Zodi_bs` also absorbs Galactic continuum
  (−0.15 dex plane-to-pole, with no template in the basis), and the airmass
  term looks like airglow leaking into it.
* **Validation**, with whole nights held out: dark-time scatter
  0.124 → 0.102 dex; sci-arm contrast slope 0.45 → 0.74 (near) and
  0.43 → 0.94 (far). Degree 5 fits the levels better but drops the near-arm
  slope to 0.37, i.e. it learns structure that does not carry over to a
  neighbouring pointing.
* **Size.** Over the observed pointings it applies −0.13 to +0.26 dex,
  median +0.06; the pole is +0.30 dex relative to the ecliptic.
* **Decomposition A/B** on 1000 rows stratified in moon state, $|\beta|$ and
  $|\Delta\lambda|$:
  * The dark-time trend of fitted/anchor zodi with $|\beta|$
    (+0.03 → +0.25 dex from ecliptic to pole) goes flat (+0.04 to +0.07).
  * Dark rows pinned by the anchor fall from 12% to 1%; the free dark rows'
    scatter falls from 0.114 to 0.108 dex.
  * On moon-up rows about 0.07 dex of continuum moves from the moon to the
    zodi, with the total conserved to $4\times10^{-5}$. 84% of those rows
    stay on the ceiling, as intended.
  * Median $\chi^2$ is unchanged in every band, and 61–67% of rows improve
    in blue.

The correction is also applied to $f_{\rm pred}$, so constraints (b) and (c)
stay consistent. It is selected by `SPLIT_ZODI_ZODI_CORRECTION`, which the
environment variable `LVMSKY_ZODI_CORRECTION` overrides; `"none"` reproduces
every earlier corpus bit for bit. The chosen name is written to the primary
header of every product as `ZODICORR` and folded into the run fingerprint.

> **One correction, three consumers.** The decomposition anchor, the
> `moon_model_cache` and inference must all use the same correction. The cache
> matters because its physics-only `zodi_total_po` feeds `zodi_po_log10` and
> the zodi ceiling (§3.9.1); inference reads `ensemble["zodi_correction"]`.
> The cache reads `ZODICORR` from the `_decomp_*.fits` files, records it, and
> `load()` refuses a cache built with a different one. The trainer stamps the
> name into the saved ensemble. Changing it therefore means re-decompose,
> rebuild the cache, retrain. The raw context feature `zodi_log10_v` stays the
> uncorrected Leinert value.

Both predictions come from a single call to the frozen physical model with its
*learned* scale factors set to unity, keeping only the below-horizon taper.
The other ten parameters were fitted with `CORRECTION_SCOPE='moon_plus_zodi'`,
i.e. only the sum was ever constrained, so the split they imply is not
independently calibrated; and `zodi_target_airmass_log = 1.28` (zodi growing
as $X^{1.28}$, where extinction should make it *fall*) looks like it absorbed
the very moon-into-zodi leakage these priors exist to remove.

> **Use the physics-only variant for anything that compares against these
> brackets.** `moon_model_cache` stores the *learned*-parameter prediction as
> well, and the two differ by 1.5–1.7× on the absolute zodi. Comparing fitted
> zodi against the learned value made the anchor look almost never binding
> (1.7% of moon-up rows at the ceiling, the ratio spanning 3.7 dex and 16.5% of
> rows *below the floor*, which an enforced bound makes impossible). The
> physics-only value gives 87.5%. Cache v2 stores both; the tell that you have
> the wrong one is a ratio wider than the $2\log_{10}\kappa_z = 0.602$ dex the
> bracket permits.

**What the constraints actually do, measured.** On `gaia-stars-mask` with the
exact per-row design rebuilt from the stored LSF surface, each half of the
corpus has exactly one continuum family set by a constraint rather than by the
data — a symmetry worth internalising before reading any moon or zodi metric:

| rows | zodi anchor | | | moon share | | |
|---|---|---|---|---|---|---|
| | ceiling | floor | interior | ceiling | floor | interior |
| moon up | **87.5%** | 1.2% | 11.2% | 1.2% | 5.0% | 93.8% |
| moon down | 5.0% | 12.5% | **82.5%** | **87.5%** | 1.2% | 11.2% |

The dark-time anchor is now well calibrated — median
$\log_{10}(v / c\,Z_{\rm pred}) = -0.057$ at $c = 1.6$, i.e. within 14% —
so the recentring did its job there. What it did *not* do is unpin bright
moon: the ceiling moved up by $\log_{10}1.6$ and the QP followed it straight
back up.

**The ceiling is correct and must not be widened.** Releasing $\kappa_z$ to
100 on 294 lunation-stratified spectra (100 FLI-stratified exposures over 91
nights × 3 arms, refitted through `decompose_parallel.init_worker` itself so
only `zodi_amp_bound` differs) moves the freed zodi up 0.193 dex while the
moon falls 0.107 and the diffuse block falls 0.435 — the total continuum is
conserved to 0.5% and the median rms improves 0.6%. It is re-partitioning
among degenerate families, not new information. And the excess tracks the
*moon*, not the ecliptic: partial $\rho({\rm excess},{\rm FLI}\mid B_{500})
= +0.733$ against partial $\rho({\rm excess}, B_{500}\mid{\rm FLI}) = -0.322$,
which has the wrong sign for zodiacal light. It is moon-into-zodi leakage, and
the bracket is what stops it. This is also the measurement the older note
called impossible while the anchor binds: releasing it is precisely what makes
$\rho({\rm zodi},{\rm FLI})$ and $\rho({\rm zodi},B_{500})$ interpretable.

**The $\epsilon$ floor makes the dark-time moon a ghost.** With the moon down
$f_{\rm pred}\sim3\times10^{-5}$, so $\kappa_f f_{\rm pred}\ll\epsilon$ and
the bracket collapses to $[\,\cdot\,,\epsilon]$; the QP then sits on that
ceiling on 87.5% of such rows, giving $\int\!{\rm moon} =
R\int\!{\rm zodi}$ with $R = \epsilon/(1-\epsilon)$ in basis units. The
component is 1.04% of the fitted continuum there (p90 1.75%) against 54% with
the moon up, and its *shape* is a corner of the feasible polytope — 93.9% of
the 14 adjacent `Moon_bs` knot pairs sit exactly on a $\beta$ ratio bound,
against 44.0% with the moon up. §3.9 explains what the ML side does about it.

**Measured effect** (200 spectra, 168 moon-up, 28 dark):

| quantity | before | after | target |
|---|---|---|---|
| colour reversals, moon-up | 164/168 | **2/168** | 0 |
| separation margin $s_z-s_{\rm m}$, p10 | $-7.71$ | $+0.98$ | $>0$ |
| moon log-log slope | $+0.21$ | $-4.10$ | $-3.7$ |
| zodi log-log slope | $-4.04$ | $-1.64$ | $-0.3$ |
| moon share, moon $>5°$ below horizon | 0.45 | **0.02** | $\sim0$ |
| $\rho(\int\!{\rm zodi},\,B_{500})$ | 0.09 | **0.90** | $\sim1$ |
| $\rho(\int\!{\rm zodi},\,{\rm FLI})$ | 0.94 | **0.12** | 0.10 |
| near–far moon disagreement, $p_{50}$ | 0.0554 | **0.0045** | — |

The cost is a median residual ratio of 1.01, concentrated entirely at bright
moon (×1.49 median for FLI > 0.8, ≈1% of the continuum, blue-weighted). Part
of that is the old fit overfitting through the spline hole described in (a),
so the comparison flatters the unconstrained version.

**Residual weaknesses.** The 2/168 remaining reversals are near-ties, not
confident inversions: over the full 17k-row corpus the worst separation is
$-1.68$ against $-9.95$ before, and 100% lie within 2.0 of the boundary. Their
rate rises with decreasing lunar illumination (0.1–0.7% at FLI > 0.85, 9–15%
below 0.3) and with decreasing moon altitude (2.2% above 40°, **19.4% below
5°**), and the science arm reverses 3.6× more often than the far sky arm
(4.7% vs 1.3%) — consistent with faint-source contamination of the science
continuum. Rows reversed in any arm are dropped from training and from the
every10 diagnostic samples (§3.8).

### 1.4.4 Identifiability constraints on the diffuse block

The moon/zodi constraints above leave the *inside* of the diffuse block free,
and it is not identifiable either. Two further hard linear inequalities, in
the same cone, fix it. Both were sized on the blue ($\lambda<6000$ Å), because
OH owns the red pixel budget and a full-band $\chi^2$ hides everything that
happens to the continuum.

**(d) Species-ratio bracket.** On the canonical basis the free fit spreads
$\log_{10}(A_{\rm FeO}/A_{\rm HO2})$ over **10.2 dex** and drives HO2 to
exactly zero on 23% of rows, because canonical HO2 carries only 4.8% of its
emission below 9800 Å and is a featureless red riser there. That spread is not
information: the **three arms of one exposure**, looking at the same sky a few
degrees apart, disagree about the ratio by **0.633 dex** at the median, and
airglow does not vary by ×4 in a species ratio over 5°.

So each of $\log_{10}(A_{\rm FeO}/A_{\rm HO2})$ and
$\log_{10}(A_{\rm O2Ac}/A_{\rm HO2})$ is bracketed to $\pm0.2$ dex about the
corpus geometric-median flux shares $(0.0396, 0.7026, 0.2578)$. A ratio bound
is linear in $\mathbf{c}$ — $A_k - R_{\rm hi}A_0 \le 0$ — so each side is one
inequality, exactly as for the moon share.

The centre comes from *our* corpus, not from PALACE: centring on PALACE's own
reference shares costs **25.7%** of the blue $\chi^2$ against 0.67% for the
corpus median, which is the Paranal-vs-LCO caveat of §1.2.4 in numbers. The
*width*, by contrast, is nearly free — the scatter is pure degeneracy, and
$\pm0.2$ dex costs 0.07% in the sizing scan. Validated over 500 stratified
rows: the 5–95 span collapses 10.25 → 0.400 dex (the bracket), HO2 near-zero
rows 23.2% → 0.2%, for +2.5% of sci blue $\chi^2$ and no measurable full-band
cost. It moves the split, not the sum: per-row $\log_{10}$ change is
zodi p90 0.029, diffuse p90 0.117, **zodi + diffuse p90 0.012**.

*Known failure mode.* With $\mathbf{c}\ge0$ a ratio bound makes a block
all-positive or all-zero, so a row whose fit wants HO2 $=0$ loses its entire
diffuse block. Measured at 0.2–0.4% of rows; `diffuse_zeroed_keep_mask`
catches them downstream.

**(e) Moon-gated cap on the block against OH.** The diffuse species are
mesospheric chemiluminescence and cannot depend on the moon, yet
$A_{\rm diffuse}/A_{\rm OH}$ rises with `moon_frac_po` at
$\rho=+0.716$ while airmass and van Rhijn give $+0.015$ and OH itself gives
$+0.088$. On full-moon rows ~74% of the fitted FeO is moonlight; the misplaced
flux is a median 3.3% of the fitted continuum and up to 81% on the worst rows.
Normalising by OH is what makes this measurable — OH is the best-determined
airglow amplitude and is itself moon-blind.

$$A_{\rm HO2}+A_{\rm FeO}+A_{\rm O2Ac} \;\le\; 10^{\,c + W}\,A_{\rm OH},
\qquad c = -0.6489,\; W = 0.15\ {\rm dex}$$

$A_{\rm OH}$ is a *known constant* in the continuum solve — OH lives in the
line model, which is held fixed there — so this is one inequality with a
right-hand side, re-installed on every continuum solve because $A_{\rm OH}$
moves as the LSF refines.

> **Which basis $A_{\rm OH}$ is in, and why it matters.** The OH coefficients
> stored in `COEF` live on the **convolved stick** basis, *not* on
> `SkyDecompLSFSurfaceIterative.matrix_oh`. Checked against the stored
> component spectrum, which is ground truth:
> `sum(COMP_OH) / (coef . stick_rowsum) = 1.0005` against
> `sum(COMP_OH) / (coef . matrix_oh.sum(axis=1)) = 0.19956`. The diffuse
> coefficients, by contrast, *do* live on `matrix_diffuse`
> (`sum(COMP_DIFFUSE)/(coef . matrix_diffuse.sum(axis=1)) = 1` exactly), so the
> two families in one `COEF` table use **different bases** and any
> cross-family ratio assembled from `matrix_*.sum(axis=1)` is wrong by 5.01x
> on the OH side. Reproduce $c$ and $W$ either from
> `model._convolve_matrix_channelwise(model.matrix_oh_stick).sum(axis=1)`, or
> — better — by integrating the stored `COMP_*` planes, which carry no basis
> ambiguity. Measured that way the cap binds at $\log_{10} = -0.4924$ against
> the specified $-0.4989$, a 0.0065 dex match. Getting this wrong once
> produced a fully self-consistent but false report of a 5.02x
> “mixed-normalisation bug” in this constraint, in which the stick/flux ratio
> reproduced the apparent offset to 0.004 dex — the same error on both sides
> of the comparison, read as confirmation.


Three design choices, each forced by measurement:

* **One-sided.** The leak is directional, and it is the *lower* bound that
  costs dark-time $\chi^2$.
* **Gated on `moon_frac_po` > 0.6**, as the zodi ceiling is. The dark-time
  scatter of the ratio is 0.216 dex and **real** — clipping it costs 15–32%
  of the blue $\chi^2$ — so an ungated bound is unaffordable.
* **The whole block, not FeO alone.** An FeO-only cap was implemented and
  validated first; it worked, but on ~40% of gated rows constraint (d) then
  became the binding one (lower-edge occupancy 25.8% → 40.5%) and 195 of 415
  gated rows stayed above their bound. Capping the block leaves **zero**,
  because the three species move together.

Validated over 1000 uniformly-drawn rows (gated fraction 41.5% against 41.1%
corpus-wide):

| quantity | no cap | $W=0.25$ | $W=0.15$ |
|---|---|---|---|
| gated rows above bound | 210 | 0 | **0** |
| binds on gated rows | — | 59.3% | **69.9%** |
| FeO / HO2 / block change | — | −43.9 / −24.2 / −34.3% | **−54.1 / −31.5 / −41.9%** |
| moon / zodi change | — | +3.9 / +0.0% | **+5.1 / +0.0%** |
| released flux → moon | — | 98.7% | **98.6%** |
| blue $\chi^2$, binding rows | — | +0.33% | **+1.27%** |
| blue $\chi^2$, gated not binding | — | −0.00% | **−0.03%** |
| blue $\chi^2$, dark rows | — | −0.00% | **−0.00%** |
| diffuse collapse rate | 2.40% | 2.40% | **2.40%** |
| 580–610 nm peak / airglow, gated | 11.34% | 9.67% | **8.61%** |

**The released flux goes to the moon and not to the zodi** — 98.6% and 0.0%,
with the continuum total conserved to 0.03%. That is the intended destination
(it is moonlight) and the Leinert anchor of (c) is what protects the zodi. The
moon only grows 5% because it is already ~14× the diffuse on those rows.
$W=0.15$ removes **77%** of the moon-attributable excess, defined as
gated-minus-dark in the last row of the table.

*Why not tighter.* The residual $\rho=+0.585$ is structural, not slack:
$A_{\rm diffuse}/A_{\rm OH}$ by gated moon quartile reads
0.2441 / 0.3193 / 0.3195 / 0.3197, so the top three sit exactly on the bound
and are flat in moon, and the whole residual is the lowest quartile at 0.2441
— *below* the bound and therefore untouched. A flat ceiling cannot remove a
rise that begins beneath it; doing so would need $W<0.036$ dex, far inside the
0.216 dex intrinsic scatter. Getting past it needs a moon-*sloped* bound.

*A surprise worth recording.* On binding rows the full-band $\chi^2$ is
**better** with the cap than without ($-0.25\%$), and at $W=0.25$ the blue was
better too. A constraint cannot improve a convex objective, so this is the
LSF ↔ continuum ↔ line iteration being non-convex: the constrained path lands
in a better basin. It also means the held-lines NNLS surrogate is unreliable
in *both* directions — it was optimistic by 35× on (d) and pessimistic on (e)
— and is good only for ranking options, never for sizing them.

#### Full-corpus outcome (`gaia-stars-mask-cont2`, 2026-09-11)

The cap now has a clean A/B: `gaia-stars-mask-cont2` differs from
`gaia-stars-mask-cont` *only* by constraint (e), the row gates are identical,
and the two filtered corpora come out at 9 943 and 9 951 rows, so the
row-set confound that invalidates a `mean_eRMSE` comparison is negligible
here. 41% of rows are gated; ungated rows are **bit-identical** between the
two corpora, as they must be.

*The leak is removed as designed.* $\rho(\log A_{\rm FeO}/A_{\rm OH},
A_{\rm moon})$ falls $+0.480 \to +0.208$, and FeO/OH by moon-amplitude
quartile goes from 0.125 / 0.167 / 0.146 / **0.428** to 0.124 / 0.168 /
0.146 / **0.165** — quartiles 1–3 untouched to three digits and only the
bright-moon excess removed, i.e. Q4/Q1 **3.43x** $\to$ **1.33x**. On gated
rows $A_{\rm diffuse}$ falls 32.3%, FeO 47.7%, HO2 24.1%, O2Ac 6.7%, the
moon rises 5.1% and the zodi moves $-0.00\%$. Decomposition
`reduced_chi2` median ratio is **1.0000** (p90 1.0085, p99 1.0797), the total
model amplitude is conserved to 0.0008 dex, and the diffuse-collapse rate
holds at 2.28% $\to$ 2.35%.

*It buys the continuum and costs OH.* Scale-free amplitude MADs all improve:
continuum **0.01868 $\to$ 0.01434** ($-23\%$), FeO 0.02355 $\to$ 0.01791,
HO2 0.02733 $\to$ 0.02150, zodi 0.00163 $\to$ **0.00022**, moon 0.01205
$\to$ 0.01163, OH total 0.00659 $\to$ 0.00648. Flux space improves slightly
too: reconstruction $\chi^2$ 3.984 $\to$ 3.889 against a self-fit floor
3.657 $\to$ 3.572, ratio 1.09x either way. But the OH **coefficient**
transfer regresses hard — mesospheric ML 38.94 $\to$ 45.0, its gain over
`B1_near_geo` **+1.3% $\to$ $-8.3\%$**, and GROUP-EQUAL **+3.0% $\to$
$-5.1\%$**; both now *lose* to the naive baseline. `mean_eRMSE` 7.956 $\to$
8.516 and the seed-to-seed std blows out 0.191 $\to$ **0.614**, so training
also became 3.2x less stable. Gains are same-test-row ratios, so unlike
`mean_eRMSE` the sign flip is not a row-set artefact.

The mechanism is straightforward: on 41% of rows the cap welds the diffuse
block to OH, which makes the diffuse trivially predictable (hence the 23%)
and pushes whatever broadband structure it used to absorb into the OH stick
distribution — OH *total* amplitude stays accurate (MAD 0.00648, median
$+0.00033$) while its 357-coefficient shape gets worse. Mesospheric is 358
of 388 coefficients, so GROUP-EQUAL follows it.

*On $W$, after a round trip.* $c = -0.6489$ is confirmed as the dark-time
median (measured $-0.6616$, 0.013 dex), so the centre is right. $W = 0.15$ is
~0.7x the dark-time robust sigma (one-sided $p_{84}-p_{50} = 0.305$ dex), so
the bound does sit inside the intrinsic spread and may clip legitimate
variation on gated rows. $W = 0.30$ was tried for exactly that reason
(`gaia-stars-mask-cont3`) and **reverted**: the OH regression it was meant to
buy back is a degenerate-metric artefact, and 0.30 gave back half the leak
suppression the constraint exists for. See the cont3 changelog entry. What
0.15 actually costs is small and measured: blue $\chi^2$ on
gated-but-not-binding rows $-0.03\%$, dark rows $-0.00\%$, collapse rate
2.28% $\to$ 2.35%, decomposition `reduced_chi2` median ratio 1.0000. The one
unresolved argument for a looser bound is training stability — seed-to-seed
`mean_eRMSE` std 0.614 at 0.15 against 0.354 at 0.30 and 0.191 uncapped —
though `mean_eRMSE` is itself the absolute coefficient-space metric and
inherits the same degeneracy, so that signal is not clean either.


## 1.5 Linear algebra: rescaling for stability

Direct QP on physical units suffers from column-scale imbalance
(order-of-magnitude differences between OH sticks and B-spline columns).
`SkyDecomp._solve_nonnegative_weighted` applies a **two-stage rescaling**:

1. **Data scale.** Let $d = \max\big(\sqrt{\overline{y_w^2}},\,1\big)$
   where $y_w = \sqrt{\mathbf{w}}\odot\mathbf{y}$. Set $\mathbf{y}' = \mathbf{y}/d$.
2. **Column scale.** Let $s_k = \lVert (\sqrt{\mathbf{w}}\odot\mathbf{A})_k\rVert_2$
   per column. Set $\mathbf{A}'_{:,k} = \mathbf{A}_{:,k}/(d\,s_k)$.
3. Solve $\mathbf{c}'^\star = \arg\min_{\mathbf{c}'\ge 0} \tfrac{1}{2}\lVert\sqrt{\mathbf{w}}(\mathbf{A}'\mathbf{c}'-\mathbf{y}')\rVert^2 + \text{(rescaled regularisers)}$
   using Clarabel.
4. **Undo the rescaling:** $c_k = c'_k / s_k$.

The regularisation operators are rescaled consistently:

- Curvature: $\mathbf{D}^{(2)}_{\rm scl} = \mathbf{D}^{(2)}\,\operatorname{diag}(1/s_{\rm m})$
  (and similarly for zodi).

## 1.6 LSF and the continuum / LSF / line refinement loop

The row's line-spread function varies with wavelength and between the three
spectrograph channels, and it sets the shape of every narrow template the QP
solves against.  Because the LSF and the coefficients constrain each other,
the pipeline updates them alternately.

**LSF representation** (`sky_decomp/lsf_spline2d.py`).  The kernel at
wavelength $\lambda$ is a continuous density in the offset
$\delta=\lambda'-\lambda$:

$$
K(\delta;\lambda)=\sum_{j=1}^{11}a_j(\lambda)\,M_j(\delta),\qquad
\delta\in[-3,3]\,\text{Å},\qquad \sum_j a_j(\lambda)=1,
$$

with $M_j$ eleven unit-integral cubic M-splines, so $K$ integrates to one.
The fit constrains the kernel to be non-negative and unimodal (one central
peak) at every wavelength.  The weights $a_j$ are constant across the b channel
and a six-function cubic B-spline in $\lambda$ across r and z.  Line templates
are rendered by integrating $K$ analytically over each pixel.  The fit carries
a roughness penalty on the weights (`roughness_fraction = 1e-4`) and a cubic
nuisance background per channel; a Gaussian of the DRP-reported width seeds
it and is kept for a channel with too little line information.

**Refinement sequence** (`SkyDecompLSFSurfaceIterative._run_iterations`):

1. **O$_2$ pre-fit** of the b-band shape (§1.2.5).
2. **Joint seed**: the full QP with all families at the nominal LSF.
3. **Refinement cycles** (`n_refinement_cycles = 5`), each of three stages:
   1. **continuum** — moon, zodi and diffuse refitted with the line model held
      fixed; pixels within ±2 Å of the lines carrying 99% of the line flux are
      down-weighted by `line_weight = 5e-4`, and outliers get a Huber weight
      beyond 3σ.  All continuum constraints of §1.4.3–§1.4.4 are imposed here;
   2. **LSF** — the kernel above refitted per channel to data − continuum,
      given the current line amplitudes;
   3. **lines** — the line templates re-rendered with the new LSF and the line
      block (OH, O$_2$, atoms) refitted to data − continuum.

   A cycle is committed only if all three stages succeed; otherwise the fit
   returns the last committed state.

The final coefficients, uncertainties and component spectra refer to the last
committed LSF.  The old 11-tap discrete surface (`native_grid_channel_median`)
belongs to the retired non-telluric variant; the telluric products store the
continuous density (`continuous_mspline_density`), which
`SkyDecompLSFSurfaceIterative` cannot read. Reconstruction therefore goes
through `mlp_predictor.data.make_reconstruction_decomposer`, which builds the
matching decomposer, and `reconstruct_with_lsf(..., telluric=...)`, which
supplies the row's transmission and OH line file.

## 1.7 Weights

**Correction (2026-09-18).** This section used to state that the QP weights
every pixel by its inverse variance, with a floor taken from the `IVAR` /
`SKY_IVAR` HDUs.  That was never what the code did.  The decomposition fit has
always been called with

```python
ivar_row = np.ones_like(flux_row)        # decompose_parallel._fit_ivar_row
ivar_row[science_line_windows] = 0.0     # the ONLY structure in it
```

— an unweighted MASK, not a variance.  No `IVAR` HDU was ever read on the
decomposition side, and the array was not even `isfinite`-filtered.  Every
pixel therefore counted equally, from the OH band heads to the faint interline
continuum, though their photon noise differs by more than an order of
magnitude.  Two things follow that matter elsewhere in this document: the
coefficient covariance of §1.9 is built from an unweighted normal matrix, and
`reduced_chi2` in the products is a residual-per-pixel, not a $\chi^2$ against
a noise model — which is why its values sit at 0.03–0.9 rather than near 1.

**Photon weights are now the default** (since 2026-09-18; disable with
`--no-fit-pixel-weights` to reproduce an older corpus).  They put the fit on the
same footing as the ML loss, which has used the absolute photon model since
2026-09-09:

$$\mathrm{var}(f) = \frac{f\,s(\lambda)}{t_{\rm exp}\,\Delta\lambda\,
N_{\rm eff}}, \qquad N_{\rm eff} = \tfrac{2}{\pi} n_{\rm fibres},$$

with $s$ the ABSOLUTE sensitivity and a 5%-of-median variance floor.  The
curves are vendored in `sky_decomp/data/sensitivity/` (copies of column 4 of
`lvmcore/sensitivity/sens_percentiles-{arm}.csv`, the only absolutely-scaled
table) so a decomposition does not depend on `$LVMCORE_DIR`, and
`sky_decomp.pixel_weights.absolute_sensitivity` is bit-identical to the loss's
own loader.  The weights are **row-normalised to mean 1** over the good pixels:
that keeps the relative weighting across wavelength — the entire point — while
leaving the overall scale where the unweighted mask had it, so
`moon_smooth_lambda`, `zodi_smooth_lambda`, the LSF `roughness_fraction` and
`line_weight`, all tuned against `ivar = 1`, keep their meaning.

Measured on the full 1 447-row `every10` telluric subset, refitted both ways
and scored on the ABSOLUTE single-fibre photon $\chi^2$ (`reduced_chi2`
becomes a weighted $\chi^2$ under the flag and its values stop being
comparable across the change):

| arm | full band | blue (<6000 Å) | rows improved |
|---|---|---|---|
| sci  | **−11.6%** | −1.9% | 1421/1445 |
| near | **−11.2%** | −3.1% | 1444/1445 |
| far  | **−14.0%** | −3.7% | 1438/1444 |

Fit status is unchanged.  The red carries ~3× the blue's weight (the sky is
brighter there but the sensitivity is ~7× smaller, so the variance is lower),
which led us to expect the blue continuum to be traded away for OH — it was
not, the blue improves on every arm.  The real cost is in the PARTITION, which
$\chi^2$ cannot see: the deployed diffuse-collapse gate (§1.11) goes 42 → 51
rows, i.e. **−0.62% yield** for an 11–14% $\chi^2$ gain, and the diffuse block
moves >0.1 dex on 5.5% of rows while OH, moon and zodi barely move.
`--fit-pixel-weight-clip FACTOR` bounds the weight dynamic range to
$[1/F, F]$ about the row mean (the raw weights span ~600× within a row); at
$F=3$ it halves the $\chi^2$ gain and is not needed at the measured collapse
rate, so it is off.

**The weights and the reversal retry (§1.12.1) interact, and the order matters.**
Confirmed on a 1000-row run with both enabled: $\chi^2$ −11.1% (sci) / −10.9%
(near) / −14.8% (far) full band, blue −2.5/−1.8/−4.4%, fit status identical to
baseline — but the weights RAISE the first-fit reversal count (sci 13 → 17) by
moving the moon/zodi partition, and the retry absorbs it (43/43 recovered, zero
rows left reversed in any arm).  Enabling the weights WITHOUT the retry would
cost yield rather than save it.

The remaining weight machinery is unchanged:

- **Science emission-line mask**: the windows in `SCIENCE_EMISSION_LINES` are
  zeroed per row (`IVAR = 0`), centred on the measured H$\alpha$ velocity.
- **Interline boost** (optional): $w_i \to w_i\cdot m^{\rm il}(\lambda_i)$
  with $m^{\rm il}<1$ on strong OH cores; disabled here.
- **Finite mask**: only pixels with $y_i,\sigma_i,w_i$ all finite and
  $w_i>0$ enter the QP.

## 1.9 Coefficient uncertainties

Every ML weight and diagnostic downstream needs to know how well the per-row
decomposition constrained each coefficient.  §1.9 defines the per-row
covariance we persist to disk and the way it degrades gracefully when a
coefficient sits at its non-negativity boundary.

At the QP optimum, split coefficients into active (interior, $c_k>0$) and
inactive (pinned at 0, $c_k=0$). On the active subset $\mathcal{A}$, the
regularised Fisher information is

$$
\mathbf{F}_{\mathcal{A}} \;=\; \mathbf{A}_{:,\mathcal{A}}^{\top}\operatorname{diag}(\mathbf{w})\mathbf{A}_{:,\mathcal{A}} \;+\; 2\lambda_{\rm m}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; 2\lambda_{\rm z}\mathbf{D}^{(2)\top}\!\mathbf{D}^{(2)}\big|_{\mathcal{A}} \;+\; \ldots
$$

The covariance is $\mathbf{F}_{\mathcal{A}}^{-1}$, inflated by
$\max(\chi^2_\nu,\,1)$ to absorb model misfit. Column-scale and data-scale
are undone before reporting; inactive coefficients receive `NaN`. See
`SkyDecomp._coef_err_active_set`. These uncertainties feed the ML loss
weights via `COEF_ERR` (§3.6.2).

**Joint covariance persistence.** The diagonal $\boldsymbol\sigma_{\rm coef}=\sqrt{\operatorname{diag}(\mathbf{F}_{\mathcal{A}}^{-1})}$ discards all off-diagonal information; for a near-collinear basis (adjacent Moon_bs
knots stay at $|\rho|\!\sim\!0.88$ even at the reduced $K\!=\!11$), that
marginal $\sigma$ substantially over-states the constraint on any
individual knot while under-stating the constraint on any joint quantity
(block amplitude, PCA scores, LSF-convolved flux over the moon block).
`SkyDecomp._coef_err_active_set` therefore also materialises, on the same
solve-group Hessians, the FULL active-set covariance restricted to the
moon and (when `split_zodi=True`) zodi B-spline blocks:

$$
\mathbf{\Sigma}_g \;=\; \big[\mathbf{F}_{\mathcal{A}}^{-1}\big]_{\mathcal{A}\cap g,\;\mathcal{A}\cap g}\qquad g\in\{\text{moon},\text{zodi}\}
$$

with the same column-scale and per-column $\chi^2$ inflation applied
consistently on the diagonal and off-diagonal (so
$\sqrt{\operatorname{diag}(\mathbf{\Sigma}_g)}$ reproduces the entries
of $\boldsymbol\sigma_{\rm coef}$ in that block); inactive rows/columns
are `NaN`.  These matrices are persisted per row to disk as the
`COEF_COV_MOON` and `COEF_COV_ZODI` ImageHDUs by `results_to_fits`
(shape $n_{\rm row}\!\times\!n_g\!\times\!n_g$; skipped for the
degenerate $n_g\!=\!1$ physical moon-zodi model).  Per-row storage
overhead at deployed dimensions is 15×15+5×5 = 250 float64 per row
$\approx 2\,\text{kB}$; a full corpus adds $\sim$100 MB across the
three arms.  The compact `*_meta_coef_*.fits` files carry both HDUs
through `extract_meta_and_coef_products`.  See §3.6.2 for how the
notebook consumes them.


## 1.10 Input data

Each row's inputs to the fit are:

- $\mathbf{y}$: 12401-pixel flux on 3600–9800 Å at 0.5 Å — per exposure, the
  median over the science fibres (after rejecting fibres whose Gaia DR3
  predicted G-band flux exceeds 1/100 of the fibre's flux + sky, hence
  `gaia1over100`) and the median over each sky telescope's fibres.
- $\mathbf{w}$: photon-noise weights from the absolute sensitivity (§1.7),
  with the median's effective fibre count $N_{\rm eff}=\tfrac{2}{\pi}n_{\rm fib}$.
- The DRP LSF width per pixel (seeds the fitted kernel, §1.6).
- Row metadata: MJD, pointing RA/Dec of all three telescopes, airmasses, PWV
  (for the telluric transmission), exposure duration and the SkyE/SkyW →
  near/far assignment.

The corpus consumed here is
`skysub/gaia-stars-mask-telluric-chi2-1.3.2/lvmsframe_median_stack_1.3.2_gaia1over100*`
with suffix `_palace_aijc_vnf_split_zodi_lsf_spline2d`. The per-coefficient
wavelengths are rebuilt from the variant's own basis on every run (§2.4) and
are not cached.

---

## 1.11 Row filtering

`data.apply_triplet_filters` gates the training corpus. On the deployed 1.3.2
corpus 13 688 of 18 668 rows survive; the per-gate fractions quoted below were
measured on earlier corpora and are indicative. The gates, in the order they
are reported:

- **Field exclusion** — 10° around the LMC and SMC (−22.8%): both combine
  bright stellar populations, dense H II regions and diffuse ionised gas at
  velocities close enough to airglow to blend with the science-arm fit.
- **Physical context sanity** — altitude $\ge 0$, airmass $\le 3$, van Rhijn
  factors finite and $\ge 1$, in all three arms (−0.1%).
- **Reduced $\chi^2$** — max across arms, capped at the 90th percentile
  (−10%). Any arm's decomposition failure disqualifies the observation.
- **Moon/zodi role reversal** (−2.5%, 434 rows) — see below.
- **Hard coefficient clips** on `feo` and `atom_k`, and an OH per-row-mean
  MAD filter (κ=4) that removes rows whose OH block ran away by factors of
  $10^6$–$10^{15}$.
- **Diffuse-block collapse** — rows where all three of HO2/FeO/O2Ac fall
  below $10^{-3}\times$ their corpus median, i.e. the QP fitted NO diffuse
  continuum at all and let the zodi spline carry it (the zodi/diffuse
  degeneracy taken to its limit). 188 of 14 447 science rows on
  `gaia-stars-mask`, and they are genuinely zero rather than faint: 1.279% of
  rows sit below $10^{-6}\times$ the median against 1.299% below
  $10^{-2}\times$, so the distribution is bimodal with nothing in between.
  Dropped when ANY arm is zeroed. Now also flagged per row (§1.12).
- **Science-continuum colour excess** — rows where the science fibre's
  red/blue continuum colour exceeds the sky arms' by $dC > 0.05$, i.e. the
  fibre carries continuum the sky model cannot represent and the moon spline
  absorbed it. Also flagged per row (§1.12).
- **κσ clipping** (κ=8, 3 iterations; OH-MAD κ=6) on the concatenated three-arm
  coefficient vector.

### 1.11.1 The moon/zodi reversal gate

A row is *reversed* when its fitted moon continuum is redder than its fitted
zodi continuum, $s_{\rm m} > s_{\rm z}$, where $s$ is the log-log slope of the
reconstructed component. The physics puts the moon near $-3.7$ and the zodi
near $-0.3$, so the ordering is unambiguous whenever both families carry flux.

Such a row is not merely noisy, it is **mislabelled**: its moon coefficients
describe zodiacal light. Training on it teaches the network a moon-geometry
mapping from zodi-shaped targets, and scoring a prediction against it measures
nothing. The gate therefore belongs with the $\chi^2$
decomposition-failure gate rather than with the target-outlier filters, and it
is applied identically to the every10 diagnostic samples (§3.8) — otherwise
those samples surface spurious outliers the model never trained on.

The test only applies where both families carry flux, $0.05 \le u/(u+v) \le
0.95$; a slope is meaningless when a component is switched off, and under the
decomposition priors (§1.4.3) dark-time rows sit at a moon share of ~0.02 by
construction. Rows failing that condition are kept and counted separately.
Reversal in **any** of the three arms disqualifies the row, matching the
$\chi^2$ convention.

Measured on an earlier 17 214-row corpus: 434 rows dropped (2.5%), which is 5.5% of
the 7 831 rows where the test applies. Per arm the rate is 2.2% (near), 1.3%
(far) and 4.7% (science), and 70% of flagged rows reverse in only one arm —
so the per-arm rate is the honest one and the residual reversals are
per-fit noise rather than a geometry-driven failure.


## 1.12 Reliability flags in the products

Every judgement in §1.11 is a corpus-build GATE: it decides which rows enter
training and throws the rest away.  Production sky subtraction cannot do that —
a real observation still owes a sky spectrum — so as of 2026-09-18 the same
judgements are also WRITTEN PER ROW, in a `reliability` bitmask column.
`sky_decomp/reliability.py` holds every definition once, so the fitter that
writes the flags and the analysis that reads them cannot drift apart.

The bits are split by severity, and the split is the useful part:

| severity | bits | how to read it |
|---|---|---|
| **error** (low 16 bits) | `fit_failed`, `reversed`, `diffuse_collapsed`, `sci_colour_excess` | The coefficients do not describe what their names say — a failed solve, swapped moon/zodi labels, a missing family, or a science fibre whose continuum is not sky. Do not train on or score these rows. |
| **warning** (bit 16 and up) | `reversal_untestable`, `reversal_retried`, `reversal_recovered`, `zodi_anchor_pinned`, `moon_share_pinned`, `diffuse_oh_cap_binding`, `diffuse_ratio_pinned`, `shape_bound_active` | A CONSTRAINT from §1.4.3/§1.4.4 shaped the fit, so the value is partly prior rather than data. Information, not a reason to drop the row. |

`has_error()` / `has_warning()` test the two masks.  `reliability = -1` means
**not evaluated** (a row served from the compact cache, say) and is deliberately
distinct from `0`: a test that did not run must never read as a clean row.
Bit values are persisted, so they are append-only.

**Why the warnings exist at all.**  The zodi anchor binds on 93% of moon-up
rows, which makes those zodi targets partly a reproduction of the Leinert
prediction rather than a measurement (see the changelog on anchor saturation) —
and until now nothing in the products said so.  A consumer could not tell a
freely-fitted zodi from one sitting on its ceiling.  Same for the moon block in
dark time, which pins at `amp_prior_floor = 0.02` and is ~1% of the continuum.

Each constraint bit is tested against the solver's OWN functional — the
`ratio_rows` block in `sky_decomp/fit.py` — by integrating the fitted component
planes over the same good pixels the solve used.  Pinned rows then read
$v/(\kappa_z Z) = 1.0000$ and ${\rm block}/{\rm cap} = 1.0000$ exactly, against
0.36–0.91 for interior rows.  **The good-pixel mask matters**: integrating the
full row instead is 0.7–0.8% off, which is precisely the 85 science-line pixels,
and that alone flips rows near a boundary.

Measured occupancy on 24 `every10` telluric science rows — two of these fire on
nearly every row and must not be mistaken for rare events:

| bit | rows | note |
|---|---|---|
| `zodi_anchor_pinned` | 18/24 | consistent with 93% of moon-up rows |
| `diffuse_oh_cap_binding` | 13/24 | every GATED row binds (gate is `moon_frac_po > 0.6`) |
| `moon_share_pinned` | 9/24 | dark time pins at 0.02, the "ghost" moon block |
| `diffuse_ratio_pinned` | 19/24 | the three species are individually unidentifiable (§1.4.4) |
| `shape_bound_active` | **24/24** | the boolean carries almost no information — read the `shape_bound_pairs` COUNT column instead (0–16 of 18 adjacent pairs) |

Columns written beside `reliability`: `reversal_separation` (signed, negative =
reversed), `reversal_moon_frac`, `reversal_retry_bound`, `sci_colour_excess`,
`moon_share`, `zodi_int`, `shape_bound_pairs`.  They land in the `META` HDU of
each `_decomp_{arm}` product and propagate into the `_meta_coef_` files, which
copy `META` wholesale.

### 1.12.1 Retry on reversal

A reversal (§1.11.1) is a shape-LABELLING artefact, not a brightness error: the
amplitudes of a reversed row are nearly right, only the moon/zodi
identification is swapped, because the data barely distinguish the two branches
($\rho({\rm moon},{\rm zodi}) = -0.948$).  Such rows are therefore recoverable
rather than merely droppable.  Refitting the 18 reversed science rows of the
telluric `every10` with a tighter adjacent-knot bound
(`moon_ratio_bound = zodi_ratio_bound`, deployed 0.70):

| bound | un-reversed | median $\chi^2$ ratio |
|---|---|---|
| 0.70 | 1/18 | 1.000 (control: reproduces the gate) |
| **0.85** | **18/18** | **1.028** |
| 0.95 | 18/18 | 1.084 |

All 18 recover at 0.85 for +2.8% median $\chi^2$ (worst single row +109%), and
the moon share moves in the fourth decimal.  `decompose_parallel` now does this
automatically (`SPLIT_ZODI_REVERSAL_RETRY = True`, bound
`SPLIT_ZODI_REVERSAL_RETRY_BOUND = 0.85`, disable with `--no-reversal-retry`),
retrying ONLY the rows that come out reversed so the deployed 0.70 still holds
for the 97.9% that are fine.  The retry is recorded either way:
`reversal_retried` plus `reversal_recovered` on success, and the ERROR bit
`reversed` if it stays reversed — so a retried row is always distinguishable
from a clean one.  A retry that fails to converge keeps the original fit, on the
grounds that a solved-but-reversed row beats an unsolved one.

### 1.12.2 At prediction time

`predict_sky_from_minimal_inputs` returns a `reliability` array in the same
vocabulary, with two deliberate restrictions:

* Only `diffuse_collapsed` is computable from the predicted COEFFICIENTS, using
  the ensemble's own `coef_upper_bound` as the per-coefficient scale.  It
  selects the same rows as the corpus gate (verified: identical 29 rows on the
  telluric `every10`, with thresholds differing 40×, because the population is
  bimodal).
* `reversed` needs the reconstructed moon and zodi CONTINUA, so it comes from
  `inference.reliability_from_components(components, wave)`, which the caller
  ORs in after reconstructing the spectrum it wanted anyway.

The constraint warnings never appear at prediction time: there is no QP, and
those bits describe how the row's TRAINING TARGETS were shaped — a property of
the corpus, not of the prediction.  `notebook_example_predict_sky.ipynb` §4.2
shows the flags being read and printed by name.

# Chapter 2 — Geometric normalisation & effective extinction

**Framing principle.** The ML transfer (Chapter 3) is trained and evaluated
in **intrinsic-emissivity space**, not in observed-flux space. Each row's
decomposition coefficients (Chapter 1) are pre-divided by the row's own
line-of-sight geometry factor $V(z;h)\cdot 10^{-0.4 k(X-1)}$ **before** they
enter the network, so the network sees the zenith-equivalent amplitude of the
underlying emitting layer. On the output side, predictions are multiplied
back by the science-arm geometry factor to restore the **observed** amplitude
at the science pointing:

$$
\boxed{\;\;\mathbf{c}^{\rm intrinsic}_{r,g} \;=\; \frac{\mathbf{c}^{\rm obs}_{r,g}}{V(z_r;\,h_g)\cdot 10^{-0.4\,k_g\,(X_r-1)}} \;\;\xrightarrow{\;\text{ML}\;}\;\; \hat{\mathbf{c}}^{\rm intrinsic}_{{\rm sci},g} \;\;\xrightarrow{\;\times V(z_{\rm sci};h_g)\cdot 10^{-0.4\,k_g\,(X_{\rm sci}-1)}\;}\;\; \hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g}.\;\;}
$$

The reason this framing works is that intrinsic emissivity is the quantity
that **actually transfers** from one line of sight to another: gravity-wave
fluctuations aside, the emitting layer at 87 km looks the same in whatever
direction you point during a 900-s exposure. Zenith angle, altitude and
extinction on the other hand differ between the two sky arms and the science
arm, and if the network had to learn those differences from context it would
be re-deriving well-known atmospheric physics on limited training data. By
dividing them out **in closed form before the network**, we let the ML head
focus on the remaining physics — gravity-wave and short-timescale variability
that the sky arms can genuinely constrain for the science arm.

The decomposition (Chapter 1) returns coefficients in **observed physical
units**: amplitudes at the actual pointing, at the actual zenith angle, at the
actual airmass. Chapter 2 is the closed-form transform that converts those to
the intrinsic emissivities the ML consumes, and back again on the output side.

## 2.1 The scaling law

The intrinsic-emissivity transform of §2.2 rests on the standard thin-shell
airglow scaling law: line-of-sight-integrated shell emission at zenith angle
$z$ enhances by a $\sec$-like factor $V(z;h)$ that saturates near the horizon
because the observer sits inside a curved shell rather than an infinite plane.

For an airglow layer at zenith angle $z$ and target airmass $X(z)$, the
observed amplitude relates to the intrinsic (zenith-equivalent) emissivity by

$$
A_{\rm obs} \;=\; A_{\rm int}\;\underbrace{V(z;h)}_{\text{slant path}}\;\underbrace{10^{-0.4\,k(\lambda)\,[X(z)-1]}}_{\text{extinction}},
$$

with the **van Rhijn** slant-path factor for a thin shell at height $h$ above
Earth radius $R_\oplus$

$$
V(z;h)\;=\;\left[1-\left(\frac{R_\oplus}{R_\oplus+h}\right)^{\!2}\sin^2 z\right]^{-1/2},\qquad V(0;h)=1.
$$

$V$ **saturates** toward the horizon (6.0 at $h=87$ km, 3.4 at $h=285$ km)
rather than diverging like $\sec z$, because the observer sits inside a curved
shell. Using plane-parallel airmass instead overstates the enhancement by 4%
at $z=60°$ for a mesospheric layer and 12% for an ionospheric one — a smooth
function of $z$, so it does not average away and would alias into whatever the
network learns as geometry.

## 2.2 Where it is applied

The training pipeline consumes coefficients from three simultaneous pointings
taken at different zenith angles.  If the network had to learn the geometry
trigonometry from the context vector we would be re-deriving well-known
physics on limited data; §2.2 spells out where the closed-form correction is
inserted so it stays consistent with the compressor and RobustScaler that
sit on either side of it.

`airglow_geometry_scale()` returns $V\!\cdot\!10^{-0.4k(X-1)}$ per row and per
coefficient, and is applied **in physical units, before the square-root /
asinh transforms and before robust scaling**. Neither of those operations
commutes with a division by the geometry factor: the scaler subtracts a
median, so $(\sqrt{c}-m)/V\ne\sqrt{c/V}-m$, and under the square root the
correct divisor would be $\sqrt{V}$ rather than $V$. Applying the correction
downstream of either transform produces an error of order $V$ itself.

The full training-time pipeline for each airglow group therefore is:

1. **Decompose** each row → observed coefficients $\mathbf{c}^{\rm obs}_{r,g}$.
2. **Divide out geometry** (this chapter):
   $\mathbf{c}^{\rm int}_{r,g} \;=\; \mathbf{c}^{\rm obs}_{r,g}\,/\,\big[V(z_r;h_g)\cdot 10^{-0.4k_g(X_r-1)}\big]$
   for both sky arms and the science arm, independently.
3. **Compress** (§3.3): elementwise transform + PCA truncation.
4. **Robust-scale** across the training set.
5. **ML forward** (§3.4): predict $\hat{\mathbf{s}}^{\rm int}_{\rm sci,g}$ (still in intrinsic-emissivity, robust-scaled, compressed space).

At inference the pipeline is inverted:

6. **Inverse robust scaling**, **inverse compressor**, then
7. **Multiply back by science-arm geometry**:
   $\hat{\mathbf{c}}^{\rm obs}_{{\rm sci},g} \;=\; \hat{\mathbf{c}}^{\rm int}_{{\rm sci},g}\cdot\big[V(z_{\rm sci};h_g)\cdot 10^{-0.4k_g(X_{\rm sci}-1)}\big]$,
   inside `predict_sci_coefficients_default()`.
8. **Reconstruct** the observed sci spectrum $\hat y_{\rm sci}(\lambda) = A\hat{\mathbf{c}}_{\rm sci}$.

**The ML target is therefore the intrinsic (zenith-equivalent) emissivity of
each layer, not the observed coefficient**. The **loss** is computed on
intrinsic-emissivity residuals, not on observed residuals. Rows at high
airmass therefore contribute to the loss with the same weight as rows at
zenith, which is the physically correct behaviour: the underlying physics we
are trying to interpolate does not know the target's zenith angle. The three
`vanrhijn_*` context columns are still fed to every encoder — they are
redundant for the airglow groups (whose geometry is pre-divided) but the
`moon` and `zodi` groups need them.

Because the reference point is $X=1$ rather than $X=0$, the recovered
"emissivity" is a **zenith-equivalent** amplitude, still carrying one airmass
of extinction. This is self-consistent and cancels in the sky-to-sci transfer,
but it is not a physical layer emissivity and should not be interpreted as
one.

`assert_context_is_physical()` guards every call. It checks physical ranges
and cross-checks any `vanrhijn_*` column against van Rhijn recomputed from
`alt`. Those two are computed independently upstream, so disagreement means
the context was transformed on the way in — the failure mode being guarded
against is evaluating geometry on scaler output, where an altitude of $\sim
0.5$ becomes a zenith angle of $\sim 89.5°$ and every row silently receives a
near-horizon factor of $\sim 6$.

## 2.3 Layer heights

Each coefficient group is tagged with an emitting-layer height $h$ that
determines the van Rhijn factor its columns receive.  The values below
follow the standard airglow-modelling literature.

Assigned per group via `GROUP_HEIGHT_FEATURE`:

| group          | van Rhijn feature | height       | physics                                         |
|:---------------|:------------------|-------------:|:------------------------------------------------|
| `mesospheric`  | `vanrhijn_87km`   | 87 km        | OH Meinel + O$_2$ atmospheric bands             |
| `atomic`       | `vanrhijn_95km`   | 95 km        | K, Na, mesopause metals / metastables           |
| `ionospheric`  | `vanrhijn_285km`  | 285 km       | O I 6300/6364 and F-region recombination lines  |
| `continuum`    | `vanrhijn_87km`   | 87 km        | HO$_2$ / FeO / O$_2$Ac mesopause chemiluminescence |
| `moon`         | (none)            | factor 1.0   | scattered moonlight — scattering geometry, not thin-shell emission |
| `zodi`         | (none)            | factor 1.0   | line-of-sight integrated interplanetary dust — not a thin shell     |

The `continuum` group name is retained from an earlier grouping in which
HO$_2$ / FeO / O$_2$Ac were treated as aerosol continuum; they are actually
airglow, but they are kept in a separate coefficient group from `mesospheric`
because they use a broadband basis rather than the OH line-group basis and
their compression pipeline stays sqrt-identity (§3.3). Moon and zodi
explicitly bypass van Rhijn: neither is thin-shell emission, and their
sky-to-sci transfer relies on other physics (Krisciunas–Schaefer geometry for
moon, Leinert lookup + airmass for zodi).

## 2.4 Per-coefficient effective wavelength

Extinction varies strongly across the LVM range, so a single wavelength per
group is not adequate: the `mesospheric` group alone spans O I 5577, Na D and
the OH / O$_2$ bands, and even within the OH Meinel system $k(\lambda)$ varies
by nearly a factor of two between the reddest and bluest bands.
`resolve_coef_wavelengths_a()` assigns a wavelength per coefficient, in this
order of precedence:

1. **basis centroid** — reconstructing with $\mathbf{c}=\mathbf{e}_j$ isolates
   basis component $j$, whose intensity-weighted centroid is
   $\lambda_{\rm eff}(j)=\sum\lambda|f_j|/\sum|f_j|$. This reads the actual
   basis, which is built from the same PMD population model, LSF and
   wavelength grid the decomposition uses at fit time, so the mesopause
   rotational Boltzmann factor for OH, the exact vibrational populations, and
   the blend structure across overlapping bands are all baked in by
   construction — it is the definitionally-correct weighting. Read off the
   decomposer's design matrix at run time (~1 s for all 388 coefficients) and
   deliberately not cached, so it cannot go stale against its basis.
2. **explicit wavelength token or species lookup** baked into `COEF_SCHEMA`
   for named species (K I 7699, N I 5199, Na D, O I 5577, O I 6300, O I 7774,
   O I 8446, O$_2$ b-band). Used only for the handful of coefficients whose
   basis support is empty on the current wavelength grid.
3. **group default** — `GROUP_EFFECTIVE_WAVELENGTH_A`, last resort.

Provenance is recorded per coefficient and printed for the airglow groups,
because a silently wrong wavelength becomes a silently wrong extinction
correction.

## 2.5 Effective extinction

The extinction coefficient $k(\lambda)$ enters as $10^{-0.4k(X-1)}$. Three
physical facts govern how it should be chosen, and the implementation follows
from them.

### 2.5.1 Only the ratio matters, so the absolute value barely does

The sky-to-sci transfer of an airglow amplitude is

$$
R\;=\;\frac{V(z_{\rm sci})}{V(z_{\rm sky})}\;10^{-0.4\,k\,(X_{\rm sci}-X_{\rm sky})},\qquad \frac{\partial\ln R}{\partial k}\;=\;-0.4\ln 10\;\Delta X\;\approx\;-0.921\,\Delta X.
$$

The absolute $k$ cancels; only $k$ times the airmass **difference** survives.
The three pointings are simultaneous and share one atmosphere, so a per-night
error $\delta k$ propagates as $0.921\,\delta k\,\Delta X$. With realistic
aerosol variability ($\delta k\sim 0.02$–$0.04$ mag/airmass) and $\Delta
X\sim 0.2$ that is a 0.2–0.7% transfer error, an order of magnitude below the
gravity-wave floor. **Per-observation extinction is therefore not modelled,
and does not need to be.**

### 2.5.2 A stellar curve is the wrong curve for an extended source

For a star, scattered photons are lost. Airglow is a quasi-uniform source
filling the sky, so photons scattered out of the beam are largely replaced by
photons scattered in from adjacent lines of sight, and the effective
attenuation is well below the single-scattering value. Scattering dominates
the total everywhere airglow matters — Rayleigh + aerosol is roughly 70–100%
of $k$ from 4000 Å to 1 µm at LCO's 2380 m — so a stellar curve
over-corrects, most severely in the blue. This is a **coherent bias**, not
random scatter.

### 2.5.3 $h$ and $k$ are nearly degenerate

Over the zenith range LVM observes, $\ln V(z;h)$ and $X-1$ are collinear to
$\rho>0.995$ for $z\le 60°$. A height error can masquerade as an extinction
error and vice versa. What the data constrain, and all the ML needs, is the
product $V\cdot 10^{-0.4 k(X-1)}$.

### 2.5.4 Implementation: fit $k_{\rm eff}$ from the airglow itself

Rather than model $k$, `fit_effective_extinction()` performs a **Bouguer fit
that uses airglow as its own source**. For coefficient $j$ and a simultaneous
sky pair,

$$
\ln\!\frac{A_{\rm near}}{A_{\rm far}}\;-\;\ln\!\frac{V_{\rm near}}{V_{\rm far}}\;=\;-0.4\ln 10\;k_{\rm eff}\,(X_{\rm near}-X_{\rm far})\;+\;\varepsilon,
$$

where $\varepsilon$ is the gravity-wave fluctuation between the two lines of
sight — zero mean in the log and uncorrelated with the airmass difference. The
intrinsic emissivity cancels because both arms are the same coefficient at the
same instant. The recovered $k_{\rm eff}$ **absorbs the multiple-scattering
correction, the airglow-versus-stellar difference, the site aerosol level and
any residual error in the assumed layer height**, none of which has to be
modelled explicitly.

Four details make the estimator trustworthy:

- **Layer height is held fixed.** Because of the degeneracy above, only $k_{\rm eff}$ is fitted. Fitting both would be ill-conditioned.
- **Selection is at column level only.** A per-row amplitude threshold is
  selection on the outcome: whichever arm sits at lower airmass has the
  smaller van Rhijn factor and fails the cut unless it carries a positive
  fluctuation, which correlates the retained residual with the sign of
  $\Delta X$. In testing, a 40th-percentile per-row cut inflated the
  recovered $k_{\rm eff}$ by a factor of 1.8. Coefficients are instead kept
  or dropped **whole**, via `min_positive_fraction`; `retained_frac` is
  reported so residual row-level loss is visible.
- **Errors are cluster-robust, clustered on rows.** All coefficients in a
  row share one gravity-wave fluctuation, so naive OLS errors are far too
  small.
- **Stability is reported.** Each bin carries split-half values, and the
  fitted intercept — which should be zero. A significantly nonzero intercept
  indicates a relative throughput offset between the two sky channels rather
  than anything atmospheric.

`resolve_coef_extinction_k()` then interpolates the fitted values in
wavelength over well-constrained bins, falls back to the generic LCO stellar
curve elsewhere, and **clips into $[0,k_{\rm generic}]$**: the multiple-
scattering argument makes the stellar curve an upper bound, so a fitted value
above it signals noise or an unmodelled gradient, not physics. The resulting
per-coefficient array is stored on the dataset as `coef_extinction_k`,
threaded into `airglow_geometry_scale`, and saved in the training artifacts so
that prediction uses exactly the values training used. Set
`USE_FITTED_EXTINCTION = False` to revert to the generic curve.

## 2.6 Moon and zodi are exempt

Neither the moon nor the zodi group is a thin-shell emitter, so
`airglow_geometry_scale()` returns factor 1.0 for their coefficient columns.
The physics they need is instead:

- **moon**: the Krisciunas–Schaefer atmospheric scattering geometry. The
  ML head consumes `moon_alt`, `moon_phase`, `moon_sep`, `airmass`
  directly and learns the full non-linear dependence.
- **zodi**: the Leinert V-band lookup $B_{500}(\lambda_\odot^{\rm rel},|\beta_{\rm ecl}|)$
  times airmass. The decomposition anchor also applies the ecliptic
  correction of §1.4.3 (c); the ML feature `zodi_log10_v` does not. The ML head consumes ecliptic geometry, `airmass` and
  (via the isolated zodi branch, §3.4.1) `vanrhijn_285km` as a smooth
  F-region-height proxy for the extended zodiacal geometry.

Extinction on moon and zodi is not fitted here either. Aerosol variability
enters the moon amplitude directly (Krisciunas–Schaefer $k_{\rm ext}$), but
the same per-night degeneracy with height-of-scatter argument applies: the
absolute value cancels between sky and sci, and only the airmass difference
survives, which is at the per-cent level.

---

# Chapter 3 — ML Reconstruction & Prediction

With each row's sky-arm spectra now compressed into a physics-normalised
coefficient vector, the problem reduces to a mapping between coefficient
vectors: use the two sky arms' coefficients (plus row metadata) to predict
the science-arm's at the same instant.  Chapter 3 defines that mapping —
a symmetric dual-encoder neural network with per-group heads — and every
choice that hangs off it: the grouping into physically-meaningful blocks
(§3.2), the per-group compressors (§3.3), the isolated branches for the
anisotropic groups (§3.4.1–§3.4.3), the arm-blend prior (§3.5), and the
mixed flux/score-space loss (§3.6).

## 3.1 Goal

Given a row's paired coefficients from the two sky arms (near, far) and the
row's geometric / temporal context ("ctx"), predict the science-arm
coefficients $\hat{\mathbf{c}}_{\rm sci}$. Reconstructing
$\hat{y}_{\rm sci}(\lambda) = \mathbf{A}\hat{\mathbf{c}}_{\rm sci}$ then gives
the sky spectrum at the science pointing so that the science exposure can
be pixel-level sky-subtracted.

## 3.2 Coefficient grouping

Coefficients are routed into six physically meaningful groups by
`COEF_SCHEMA`:

| group          | patterns             | $n_{\rm coef}$ | dominant physics driver                        |
|:---------------|:---------------------|---------------:|:-----------------------------------------------|
| `mesospheric`  | `OH_\d{3}`, `O2_b\d+`|          358   | thermospheric temperature, geomagnetic activity |
| `moon`         | `Moon_bs\d+`         |             15 | $g_{\rm moon}(\phi, h, r_{\rm sep})$           |
| `zodi`         | `Zodi_bs\d+`         |              5 | $B_{500}(\lambda_\odot^{\rm rel}, |\beta_{\rm ecl}|)$ × airmass |
| `continuum`    | HO2, FeO, O2Ac       |              3 | mesospheric chemistry (residual)               |
| `atomic`       | `ATOM_K`, `ATOM_Na`, `ATOM_Og` |    3 | mesopause K I 7699, Na I D, [O I] 5577        |
| `ionospheric`  | `ATOM_N`, `ATOM_Or`, `ATOM_Orc_OI*` | 4 | F-region N I 5199, [O I] 6300/6364, O I 7774/8446 |

$n_{\rm moon}=15$ from `n_spline_knots=11` and $n_{\rm zodi}=5$ from
`n_zodi_spline_knots=1`, for $P = 388$ coefficients per row (358 mesospheric).

`zodi` is promoted to its own group by this variant. The rationale is physical:
$g_{\rm moon}$ and $B_{500}$ depend on disjoint context sets (lunar geometry vs.
ecliptic geometry) and the previous baseline's ML head had to reconcile the two
on one output. Splitting the head lets each learn the projection appropriate to
its physical support.

## 3.3 Per-group compressor

Each group's raw coefficient block $\mathbf{c}_g\in\mathbb{R}^{n_g}$ is projected
into a compact "score" vector $\mathbf{s}_g\in\mathbb{R}^{n_g^{\rm score}}$ via
a fitted compressor consisting of three stages:

1. **Robust per-column scale** $\boldsymbol\rho_g$ = per-column median of
   positive training values on the near arm.
2. **Elementwise forward transform** $\mathbf{u}_g = f_{\rm kind}(\mathbf{c}_g/\boldsymbol\rho_g)$
   with `kind` chosen per group by `COMPRESSION_TRANSFORM_BY_GROUP`. The deployed
   defaults are `asinh` for `mesospheric` (coefficients span many decades and
   include OH lines close to zero) and `sqrt` for every other group (`moon`,
   `zodi`, `continuum`, `atomic`, `ionospheric`).
3. **Standardise + PCA truncation** on the pooled (near + far + sci) training
   scores:
   $\mathbf{s}_g = \big((\mathbf{u}_g - \boldsymbol\mu_g)/\boldsymbol\sigma_g\big)\,\mathbf{B}_g\big[:,\mathcal{K}_g\big]$,
   where $\mathbf{B}_g$ is the PCA basis (columns sorted by variance) and
   $\mathcal{K}_g$ retains components with **cross-arm correlation** above
   `COMPRESSION_XARM_THRESHOLD` on the held-out set. Groups with
   `COMPRESSION_USE_PCA_BY_GROUP[g]=False` skip PCA (identity basis).

Compressed target dimensions used in this variant:

- `moon`: 15 → 15 (PCA rotation; the retention threshold keeps all 15
  components, so the rotation is applied but nothing is dropped)
- `zodi`: 5 → 5 (no PCA)
- `mesospheric`: 358 → 358 (no PCA)
- `continuum`: 3 → 3 (no PCA)
- `atomic`: 3 → 3 (no PCA)
- `ionospheric`: 4 → 4 (no PCA)

Total compressed score dim = 388, matching the uncompressed count.

Inverse compressor:

$$
\hat{\mathbf{c}}_g \;=\; f_{\rm kind}^{-1}\!\big(\hat{\mathbf{s}}_g\,\mathbf{B}_g[:,\mathcal{K}_g]^\top\!\cdot\!\boldsymbol\sigma_g + \boldsymbol\mu_g\big)\odot\boldsymbol\rho_g,
$$

with $f_{\rm kind}^{-1}\in\{{\rm identity},\,x\mapsto x^2,\,\sinh,\,\exp\}$
clipped to the per-column training range to prevent $\sinh/\exp$ blow-up on
outlier predictions.

## 3.4 Architecture: symmetric dual-encoder with group heads

The network's task is to map two sky-arm coefficient vectors plus the row
context into the science-arm coefficient vector at the same instant.  Three
principles govern its architecture: shared encoders for the two sky arms
enforce arm-swap symmetry, per-group heads let physically distinct emitters
learn their own sky-to-sci transfer, and isolated branches route the
anisotropic groups (zodi, continuum) around the shared trunk so they see the
physical drivers the trunk squeezes out.

`DualEncoderGroupHeadMLPCompressed` is the deployed network. All widths and
hyperparameters live in `default_dual_group_config`; the values shown below
are defaults for this variant.

Notation: $\mathbf{s}^{\rm nr},\mathbf{s}^{\rm fr}\in\mathbb{R}^{n_s}$
compressed scores on the two sky arms (concatenated across groups,
$n_s = 388$ after RobustScaler-scaling); $\mathbf{x}^{\rm nr},\mathbf{x}^{\rm fr},\mathbf{x}^{\rm sc}\in\mathbb{R}^{n_x}$
row context features per arm (astropy geometry, VanRhijn heights, solar
activity, etc.).

**Step 1 — per-arm score encoder** (shared weights, applied to each sky arm):

$$
\mathbf{e}^{\rm nr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm nr};\,\mathbf{x}^{\rm nr}_{\rm model}]\big),\qquad
\mathbf{e}^{\rm fr} = \mathrm{MLP}_{\rm sky}\!\big([\mathbf{s}^{\rm fr};\,\mathbf{x}^{\rm fr}_{\rm model}]\big).
$$

Widths: `encoder_dims=(768, 384)` (default) with LayerNorm + GELU
between layers.

**Step 2 — context encoder** on the science-arm ctx only:

$$
\mathbf{e}^{\rm ctx} = \mathrm{MLP}_{\rm ctx}\!\big(\mathbf{x}^{\rm sc}_{\rm model}\big),\qquad \text{widths } (96,).
$$

**Step 3 — symmetric fusion.** Combine the two arm embeddings into
symmetric summaries so that swapping near ↔ far leaves the network
invariant:

$$
\mathbf{e}^{\rm mean} = \tfrac{1}{2}(\mathbf{e}^{\rm nr}+\mathbf{e}^{\rm fr}),\quad
\mathbf{e}^{\rm diff} = \mathbf{e}^{\rm nr}-\mathbf{e}^{\rm fr},\quad
\mathbf{e}^{\rm |diff|} = |\mathbf{e}^{\rm diff}|.
$$

Concatenate with the context embedding:
$\mathbf{z} = [\mathbf{e}^{\rm mean};\,\mathbf{e}^{\rm diff};\,\mathbf{e}^{\rm |diff|};\,\mathbf{e}^{\rm ctx}]$.
The signed `e_diff` term lets the trunk see the sign of any asymmetry
(near closer to moon than far, or vice-versa), while `|e_diff|` gives a
sign-agnostic magnitude that the trunk can gate on regardless of orientation.

**Step 4 — shared trunk MLP**:

$$
\mathbf{h} = \mathrm{MLP}_{\rm trunk}(\mathbf{z}),\qquad \text{widths } (320, 160).
$$

**Step 5 — per-group heads.** Each group $g$ has its own head that produces
a predicted signed **score vector** $\hat{\mathbf{s}}^{\rm head}_g\in\mathbb{R}^{n_g^{\rm score}}$:

$$
\hat{\mathbf{s}}^{\rm head}_g = \mathrm{MLP}_{\rm head,\,g}(\mathbf{h}),\qquad \text{widths } (\text{head\_dim}=192,\, n_g^{\rm score}).
$$

Heads emit **linear** outputs (no Softplus) in scaled-score space; the
non-negativity of physical coefficients is recovered later by the inverse
compressor's clipping.

### 3.4.1 Isolated zodi branch

Zodiacal light is anisotropic on the ecliptic plane and its transfer from
sky to sci depends on which side of the ecliptic each pointing sits on.  The
shared trunk squeezes those geometric distinctions into a 160-d bottleneck
that mostly carries moon information; the isolated zodi head bypasses that
squeeze and reads a physics-motivated ctx subset directly.

When `default_dual_group_config['zodi_ctx_restriction']` is a non-empty tuple
of ctx feature names, the zodi head **bypasses the shared trunk** and consumes
only:

- the **per-arm slice** of the restricted ctx features (near + far + sci
  concatenated, so the branch sees the local `zodi_log10_v` gradient and any
  per-arm moon geometry),
- the near and far zodi score blocks (5 dims each in the deployed reduced-basis
  variant; see §1.4).

Routed through a small two-layer MLP `zodi_branch: 67→64→32 (LayerNorm + GELU)`
followed by a linear head emitting $n_{\rm zodi}^{\rm score}=5$ signed scores,
one per `Zodi_bs` knot (`zodi_head_extra_dims = ()`).

The deployed restriction is 19 features:

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep', 'sun_alt', 'alt',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy',
 'zodi_po_log10', 'moon_frac_po')
```

giving $19 \times 3 + 2 \times 5 = 67$ input dims. Features cover three physical
roles:

1. **Zodi physics** &mdash; `ecl_beta_deg`, `ecl_lon_{sin,cos}`, `zodi_log10_v`,
   `sun_sep`, `airmass`, `vanrhijn_285km`. The Leinert lookup (§1.2.2) plus
   airmass (via the F-region-height van Rhijn as a smooth zodiacal-geometry
   proxy) fully determines the intrinsic zodiacal amplitude.
2. **Moon-driven contamination regime** &mdash; `moon_alt`, `moon_sep`,
   `moon_phase_{sin,cos}`, plus the four physics-prior moon-scatter proxies
   `moon_fli` $=(1-\cos\phi)/2$, `moon_up_smooth` $u=\sigma(h_{\rm moon}/5°)$,
   `moon_airmass_up` $=u\,X_{\rm moon}$ and `moon_signal_proxy`
   $={\rm FLI}\,u\,10^{-0.06X_{\rm moon}}/(1+\theta/45°)^2$.
   On moon-down rows the `Zodi_bs` coefficients carry whatever residual
   continuum shape the `split_zodi` decomposition could not place elsewhere,
   so the sky↔sci transfer depends on moon geometry even though the underlying
   zodiacal light does not. Adding the moon proxies lets the branch condition
   on "moon down" / "Q4 phase" / "bright moon close-in" regimes; combined with
   the per-regime Jensen lift (§3.6.6) the residual zodi bias on `moon_alt ≤ 0`
   rows fell from 7.7% to ~2% as the isolated branches were added.
3. **Anchor inputs** &mdash; `zodi_po_log10` and `moon_frac_po`, the physics-only
   $Z_{\rm pred}$ and $f_{\rm pred}$ that define the decomposition's zodi anchor
   and moon-share bracket (§1.4.3), so the head can reproduce a constrained
   amplitude (§3.9.1).  `sun_alt` and `alt` stay in the list as twilight and
   pointing-altitude terms.

Bypassing the trunk cuts moon-driven and geomagnetic-driven leakage into the
zodi head at the cost of the trunk's richer representation. Empirically this
trade-off wins for zodi calibration on high-$|\beta_{\rm ecl}|$ / bright-moon
rows without hurting overall mean_eRMSE.

### 3.4.2 Isolated continuum branch

Mirroring the zodi pattern, when `continuum_ctx_restriction` is a non-empty
tuple the continuum head **bypasses the shared trunk** through its own branch.
The trigger for this was a specific empirical finding: the
`residual_ctx_attribution` diagnostic (5-fold CV random-forest fit of per-row
per-group residual RMS on ctx) surfaced continuum with rf_R² = 0.70 driven
entirely by moon geometry — a non-linear coupling the shared 160-d trunk was
under-parameterised to represent.

The deployed default restriction is 9 features:

```
('moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'airmass',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin')
```

The last three are explicit interaction features (§3.4.5):
`moon_fli * moon_phase_cos` (2nd-order phase term) and
`moon_signal_proxy * ecl_lon_{cos,sin}` (moon-scatter × zodi anisotropy).

Branch: `continuum_branch: 128→64 (GELU)` (widened from `64→32` — the only
group with real headroom above the sky-arm noise floor
got the extra capacity), followed by a `(64,)`-hidden head emitting
$n_{\rm continuum}^{\rm score}=3$ scores. Input dim: $9 \times 3 + 2 \times 3
= 33$. Isolated-branch parameter cost: ~17k (13.0k branch + 4.4k head).

### 3.4.3 Additive moon-zodi coupling

Both moon (via the shared-trunk head) and zodi (via the isolated branch)
benefit from knowing the *joint* moon-scatter + zodi-geometry state — scattered
moonlight adds to the diffuse ecliptic background that the split-zodi
decomposition tries to route entirely into `Zodi_bs`. To let the two heads
*share* that joint representation without breaking the zodi branch's isolation,
an **additive coupling latent** is threaded into both head outputs.

When `moon_zodi_ctx_restriction` is a non-empty tuple (deployed default), the
model builds:

- a small `moon_zodi_coupling_branch: 100→64→32 (LayerNorm + GELU)` (~8.7k params),
- a **zero-initialised** projector
  $P_{\rm moon} = \mathrm{Linear}(32,\, n_{\rm moon}^{\rm score}=15)$ (495 params),
- a **zero-initialised** projector
  $P_{\rm zodi} = \mathrm{Linear}(32,\, n_{\rm zodi}^{\rm score}=5)$ (165 params).

The branch consumes the union of moon-scatter + zodi-geometry ctx: 20 features
$\times$ 3 arms plus 2 arms $\times (n_{\rm moon}+n_{\rm zodi}) = 40$ score
dims, for 100 input dims total.

The forward pass then becomes (for $g \in \{{\rm moon},\,{\rm zodi}\}$):

$$
\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; P_g\bigl(\mathrm{CouplingBranch}(x_{\rm mz})\bigr),
$$

where $\hat{\mathbf{s}}^{\rm head}_g$ is the moon shared-trunk head output
$\mathrm{head}_{\rm moon}(\mathbf{h})$ for $g={\rm moon}$ and the isolated zodi
branch output $\mathrm{ZodiHead}(\mathrm{ZodiBranch}(x_{\rm zodi}))$ for
$g={\rm zodi}$. Because the projectors are initialised to output exactly zero
(both weights and bias zero), **training starts byte-identical to the
uncoupled configuration**; the projectors learn what fraction of the
coupling latent to route into each head's residual.

The `moon_zodi_ctx_restriction` has 20 features: the zodi restriction (§3.4.1)
without `sun_alt` and `alt`, plus the 3 interaction features:

```
('airmass', 'vanrhijn_285km',
 'ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos',
 'zodi_log10_v', 'sun_sep',
 'moon_alt', 'moon_sep', 'moon_phase_sin', 'moon_phase_cos',
 'moon_fli', 'moon_up_smooth', 'moon_airmass_up', 'moon_signal_proxy',
 'moon_fli_x_phase_cos', 'moon_sig_x_lon_cos', 'moon_sig_x_lon_sin',
 'zodi_po_log10', 'moon_frac_po')
```

### 3.4.4 Data-flow diagram of the augmented model

```
    near-arm scores   far-arm scores   near-ctx (42)   far-ctx (42)   sci-ctx (42)
          |                 |               |               |             |
          +--- concat ------+               v               v             |
          |    scores/ctx                   |               |             |
          v                                 |               |             |
    +-----------+                       +----------------+                |
    | sky enc.  |                       |  sky encoder   |                |
    | (shared)  |------e_near-----+     |   (shared)     |------e_far     |
    +-----------+                 |     +----------------+          |     |
                                  v                                 v     v
                        e_mean = (e_near+e_far)/2  e_diff = e_near-e_far  |
                        |e_diff|                                          |
                        |                                                 |
                        v                                                 v
                        +---------- concat with e_ctx ---------------------+
                                          |
                                          v ctx encoder(sci) -> e_ctx
                                          |
                                    +-----------+
                                    |   trunk   |  (320 -> 160)
                                    +-----+-----+
                                          |
                                          v h (160-d)
                             +--------+---+---+--------+-----------------+
                             |        |       |        |                 |
                             v        v       v        v                 |
                         moon head  meso    iono     atomic              |
                                    head    head     head                |
                                    (shared trunk consumers)             |
                                                                         |
                        +------------------------------------------------+
                        |                                                |
                        v                                                v
                +---------------+                              +----------------+
                | zodi branch   |  (67-d in)                   | continuum br.  |
                | 64 -> 32      |                              | 128 -> 64      |
                | + zodi head   |                              | + cont. head   |
                | linear, n_z=5 |                              | (64,) + n_c=3  |
                +-------+-------+                              +--------+-------+
                        |                                              |
        (additive coupling latent, applied to moon + zodi)              |
        +----------------+                                              |
        |                                                               |
        v                                                               v
+-------------------+                                                +----+
| coupling branch   |                                                |    |
| 100 -> 64 -> 32   |                                                |    |
+-------+-----------+                                                |    |
        |                                                            |    |
        +--> P_moon = Linear(32, 15), zero-init  --> + moon head      |    |
        +--> P_zodi = Linear(32,  5), zero-init  --> + zodi head      |    |
                                                                     v    |
                                                              +---+ +---+ +---+ +---+ +---+
                                                     out[g] = |bl.|+|hd |+|D_g|  for each  g
                                                              +---+ +---+ +---+     of six
                                                    where D_g = coupling residual (moon,
                                                    zodi only) or 0 for the other four.

Blend for group g (see §3.5):
    hat{s}_g = alpha_g * s_g_near + (1 - alpha_g) * s_g_far + head_g + P_g(coupling)
    alpha_g is scalar OR per-row sigmoid(w.x_sci + b) for anisotropic
    groups (moon, zodi, continuum) via the ctx-alpha predictor.
```

A row's forward pass proceeds in five stages.  Each sky arm's compressed score
vector is concatenated with its 42-dimensional context and passed through the
**shared sky encoder** (widths $768\to 384$); tying the weights across the two
arms produces a single per-fibre embedding that is invariant to which physical
arm produced the input.  The two arm embeddings are then combined into three
exchange-symmetric summaries — the mean $\mathbf{e}^{\rm mean}$, the signed
difference $\mathbf{e}^{\rm diff}$, and its absolute value
$|\mathbf{e}^{\rm diff}|$ — so that the trunk sees both the direction and the
magnitude of any sky-to-sky asymmetry while remaining invariant under a
relabelling of the arms.  The science-arm context is embedded separately by a
compact **context encoder** (width 96), concatenated with the arm summaries,
and compressed by the **trunk MLP** ($320\to 160$) to a 160-dimensional latent
$\mathbf{h}$ that carries the row's shared geometric and temporal state.

Six per-group heads consume the pipeline outputs.  Four of the six — moon,
mesospheric, ionospheric, atomic — read $\mathbf{h}$ directly through a
two-layer head (width $192\to n_g^{\rm score}$).  The physically anisotropic
zodi and continuum groups instead route through **isolated branches**
(§3.4.1, §3.4.2), whose inputs are three-arm slices of a physics-motivated
context subset stacked with the two arms' native score blocks.  Bypassing the
trunk gives these two groups a direct view of the physical drivers — ecliptic
geometry for zodi, moon-scatter geometry for continuum — that the shared
representation squeezes out.

Moon and zodi are additively coupled by a small **coupling branch** (§3.4.3)
that runs once per batch on the union of moon-scatter and zodi-geometry
context; its 32-dimensional latent is projected into each of the two heads by
zero-initialised linear maps, so at $t{=}0$ the coupling contribution is
identically zero and training reproduces the uncoupled baseline byte for
byte.  During training the projectors learn what fraction of the joint
moon-zodi state to route into each head's residual without disturbing zodi's
isolated-branch representation; the four non-moon-non-zodi heads receive an
identically-zero coupling term by construction.  Each group's final score is
then an arm blend (§3.5) plus the head residual plus — for moon and zodi
alone — the coupling projection.

### 3.4.5 Context feature routing

The context vector is **42 features per arm** after the three augment stages
(ECLIPTIC-CTX-V1, PHYSICS-PRIORS-V1 and MOON-MODEL-CTX-V1). Every feature
enters the sky encoder (per-arm) and the ctx encoder (sci-arm), so the shared
trunk sees the full ctx. The specialised branches and the ctx-α predictor take
restricted slices.

> **A whitelist is a silence, not a warning.** The zodi and continuum heads sit
> on isolated branches whose context is the restriction list, so a feature
> absent from that list never reaches them and nothing says so. This is why
> `zodi_po_log10` had to be added to `zodi_ctx_restriction` explicitly (§3.9);
> without it the zodi head would have been asked to reproduce the anchor
> ceiling while blind to the prediction that defines it.

**Feature categories** (42 total):

| category | count | features |
|---|---:|---|
| Sci-pointing astrometry | 4 | `alt`, `az_sin`, `az_cos`, `airmass` |
| Moon geometry | 6 | `moon_alt`, `moon_sep`, `moon_phase_{sin,cos}`, `moon_az_{sin,cos}` |
| Sun geometry | 4 | `sun_sep`, `sun_alt`, `sun_az_{sin,cos}` |
| Arm geometry | 2 | `sci_sep`, `ew` (telescope identity: +1 SkyE, −1 SkyW, 0 sci) |
| Van Rhijn slant-path factors | 3 | `vanrhijn_87km`, `vanrhijn_95km`, `vanrhijn_285km` |
| Cyclic time features | 6 | `obstime_{day,lunation,year}_{sin,cos}` |
| Space-weather activity | 3 | `f107`, `f107_81d`, `kp` |
| Ecliptic geometry (ECLIPTIC-CTX-V1) | 3 | `ecl_beta_deg`, `ecl_lon_{sin,cos}` |
| Physics-prior moon-scatter proxies (PHYSICS-PRIORS-V1) | 5 | `zodi_log10_v`, `moon_fli`, `moon_up_smooth`, `moon_airmass_up`, `moon_signal_proxy` |
| explicit interaction features | 3 | `moon_fli_x_phase_cos`, `moon_sig_x_lon_{cos,sin}` |
| Frozen-model transfer + anchor inputs (MOON-MODEL-CTX-V1) | 3 | `moon_model_log_ratio`, `zodi_po_log10`, `moon_frac_po` |

**Feature-to-branch routing** (✓ = feature enters that branch as one of its
ctx dims; empty = does not). The **shared trunk** column stays implicit —
every feature feeds the trunk via the encoders.

| feature | isolated zodi (§3.4.1) | isolated continuum (§3.4.2) | moon-zodi coupling (§3.4.3) | ctx-α (§3.5) |
|---|:---:|:---:|:---:|:---:|
| `alt` | ✓ | | | |
| `az_sin`, `az_cos` | | | | |
| `airmass` | ✓ | ✓ | ✓ | ✓ |
| `moon_alt`, `moon_sep` | ✓ | ✓ | ✓ | |
| `moon_phase_sin`, `moon_phase_cos` | ✓ | ✓ | ✓ | |
| `moon_az_sin`, `moon_az_cos` | | | | |
| `sun_alt` | ✓ | | | |
| `sun_az_{sin,cos}` | | | | |
| `sun_sep` | ✓ | | ✓ | |
| `sci_sep`, `ew` | | | | |
| `vanrhijn_87km`, `vanrhijn_95km` | | | | |
| `vanrhijn_285km` | ✓ | | ✓ | |
| `obstime_{day,lunation,year}_{sin,cos}` | | | | |
| `f107`, `f107_81d`, `kp`, `ew` | | | | |
| `moon_model_log_ratio` | | | | |
| `zodi_po_log10`, `moon_frac_po` | ✓ | | ✓ | |
| `ecl_beta_deg` | ✓ | | ✓ | ✓ |
| `ecl_lon_sin`, `ecl_lon_cos` | ✓ | | ✓ | |
| `zodi_log10_v` | ✓ | | ✓ | |
| `moon_fli` | ✓ | ✓ | ✓ | |
| `moon_up_smooth` | ✓ | | ✓ | ✓ |
| `moon_airmass_up`, `moon_signal_proxy` | ✓ | | ✓ | |
| `moon_fli_x_phase_cos` | | ✓ | ✓ | |
| `moon_sig_x_lon_cos`, `moon_sig_x_lon_sin` | | ✓ | ✓ | |

**Head-to-branch routing** (which head consumes which upstream output):

| head | shared trunk | isolated zodi | isolated continuum | moon-zodi coupling residual |
|---|:---:|:---:|:---:|:---:|
| `moon` | ✓ | | | ✓ (additive) |
| `zodi` | | ✓ | | ✓ (additive) |
| `continuum` | | | ✓ | |
| `mesospheric` | ✓ | | | |
| `ionospheric` | ✓ | | | |
| `atomic` | ✓ | | | |

The specialised branches were added when the diagnostics pointed at
regime-specific structure the shared trunk was under-fitting; the additive
coupling was added to let moon share the zodi branch's ctx-restricted
representation without breaking zodi's isolation. See the changelog entries
for the empirical wins each branch bought.

## 3.5 Blend heads

Even before the network learns anything, a good first guess for the science
coefficients is a linear interpolation of the two sky arms.  The blend heads
give the network that starting point as an explicit prior and have each head
predict a residual on top; at zero head output the model returns the
$\alpha_g$-weighted arm blend, and training begins from the near-weighted
$\alpha_g = 0.85$ (`blend_init_alpha`).

Each group carries a **blend weight** $\alpha_g \in (0, 1)$ that mixes the
near/far arm scores with the head's residual prediction:

$$
\boxed{\;\hat{\mathbf{s}}_g \;=\; \alpha_g\,\mathbf{s}^{\rm nr}_g + (1-\alpha_g)\,\mathbf{s}^{\rm fr}_g \;+\; \hat{\mathbf{s}}^{\rm head}_g \;+\; \Delta_g^{\rm coupling}.\;}
$$

The additive coupling residual $\Delta_g^{\rm coupling}$ (§3.4.3) is non-zero
only for $g \in \{{\rm moon},\,{\rm zodi}\}$ when the coupling branch is
active; it is exactly zero at initialisation and remains identically zero for
the other four groups.

**Two parametrisations for $\alpha_g$:**

1. **Scalar α** (fallback): $\alpha_g$ stored directly (not through a
   sigmoid) as a single learnable scalar per group, clamped to
   $[\varepsilon,\, 1-\varepsilon]$ with $\varepsilon = 10^{-3}$ after each
   optimiser step. This is what the physically isotropic groups still use.

2. **Context-dependent α** (deployed for the
   anisotropic groups): when `alpha_ctx_features` is a non-empty tuple of
   ctx feature names, each group in the model's `_ALPHA_CTX_GROUPS` gets a
   **per-row per-group** linear predictor
   $$\alpha_g(x_{\rm sci}) \;=\; \sigma\bigl(\mathbf{w}_g \cdot \mathbf{x}_{\rm sci}^{\alpha} + b_g\bigr),$$
   where $\mathbf{x}_{\rm sci}^{\alpha}$ is the sci-arm slice of the requested
   features. The linear predictor's weights are initialised to zero and its
   bias to $\mathrm{logit}(\alpha_{\rm init})$ so that at $t=0$ every row
   gets $\alpha_g = \alpha_{\rm init}$ (deployed 0.85), preserving byte-identity
   with the scalar-α starting point.

**Deployed defaults:**

- `alpha_ctx_features = ('moon_up_smooth', 'ecl_beta_deg', 'airmass')` — three
  features that carry moon-brightness, zodi-anisotropy, and airmass geometry,
  so the anisotropic groups can learn distinct blend rules per regime.
- `_ALPHA_CTX_GROUPS = ('moon', 'zodi', 'continuum')` (hardcoded in
  `model.py`, not a config knob) — restricted to the
  physically anisotropic groups. The LOS-integrated thin-shell groups
  (`mesospheric`, `ionospheric`, `atomic`) keep a scalar α: their sky↔sci
  transfer is isotropic on the LVM scale, and extending ctx-α to all six
  groups degrades the ionospheric group substantially.

**α is not self-tuning at the default learning rate.** The loss surface in α
is nearly flat: the additive head absorbs the *systematic* part of a mis-set
blend, but not the per-row arm noise the blend passes straight through, so a
wrong α costs real accuracy while generating almost no gradient to correct
itself.  The ctx path is damped a further $\sim4\times$ by the sigmoid
($\sigma' = 0.25$ at $\alpha=0.5$, $0.128$ at $0.85$) and by splitting the
gradient across zero-initialised weights, and early stopping truncates the
climb.  `blend_init_alpha` therefore behaves as a hyperparameter rather than
an initialisation unless `alpha_lr_mult` (§3.7) is raised; at the deployed
`alpha_lr_mult = 30` α reaches its own optimum from either side.

Deployed optima at `blend_init_alpha = 0.85`: `mesospheric` $\approx 0.79$,
`ionospheric` $\approx 0.95$, `atomic` $\approx 0.93$, and the ctx-α groups
$\approx 0.86$–$0.91$ with per-row spans of $[0.59, 0.96]$ for `moon` and
$[0.51, 0.99]$ for `continuum`.  Every group prefers the near arm to the arm
mean, which `naive_baseline` shows directly: `B0_copy_near` beats
`B2_mean_geo` for all six groups.  `zodi` sits at $\alpha \approx 0.895$
rather than the $0.5$ an isotropic emitter would suggest, consistent with
`Zodi_bs` absorbing residual continuum that is not zodiacal light.

The ctx-α predictor reads only the **science-arm** ctx slice, so it cannot
express *which* arm is better placed (that is a `near_ctx − far_ctx`
question). It still learns useful per-row regime structure once it can
travel, but the arm-relative information remains unavailable to it.

## 3.6 Loss

The training loss is a **mixed-space** objective. The moon, zodi and continuum
groups are trained in **flux space** — the network's predicted compressed
scores are decompressed all the way to per-pixel spectra inside the loss and
compared to the target flux — while the other three groups (mesospheric,
ionospheric, atomic) stay in **scaled-score space** so that RobustScaler-
normalised residuals are comparable across compressed groups of very different
physical amplitudes. §3.6.1 defines the flux-space branch. §3.6.2 defines the
score-space branch. §3.6.3 covers the per-element COEF_ERR weighting.
§3.6.4 the diagnostic-only pixel WRMSE. §3.6.5 the row-weight boosts.
§3.6.6 the post-training per-coef Jensen-style lift.

### 3.6.1 Per-pixel flux loss on the continuum families

For each group $g \in$ `flux_mse_groups = ('moon', 'zodi', 'continuum')`
the training loss is computed **directly on the reconstructed per-pixel
spectra**, not on the compressed score residuals. The motivation is that the
diagonal COEF_ERR-weighted score loss (§3.6.2 + §3.6.3) puts per-row weight
$\propto 1/\sigma^2$, and the truth-conditioned $|z|$ diagnostic (§1.9) shows
$\sigma_{\rm scaled}$ is essentially flat across the moon-brightness axis:
bright-moon rows (with $|c_{\rm moon}| \sim 10$–20, where flux fit errors
dominate the deliverable sky subtraction) end up on the same per-row loss
scale as moon-down rows (with $|c_{\rm moon}| \sim 0.05$). A flux-space MSE
couples per-row weight to target amplitude directly: the same absolute
score-space error contributes $\sim (|c_{\rm bright}|/|c_{\rm dim}|)^2 \sim
10^5$ times more from a bright row than from a dim row, matching the ratio
at which they matter in flux.

**Inverse compressor in torch.** For each configured group $g$ the trainer
maintains a differentiable torch implementation of `inverse_group_compressor`
(§3.3):

1. Undo the RobustScaler on the group's compressed score
   $\hat{\mathbf{s}}_g^{\rm scaled}$:
   $\hat{\mathbf{s}}_g^{\rm raw} = \hat{\mathbf{s}}_g^{\rm scaled}\odot \mathrm{scale}_g + \mathrm{center}_g$.
2. Undo the group's PCA rotation:
   $\mathbf{z}_g = \hat{\mathbf{s}}_g^{\rm raw} \, B_g^\top$
   ($B_g$ = identity when `use_pca=False`, e.g. zodi; a small orthogonal
   matrix stored on device when `use_pca=True`, e.g. moon).
3. Destandardise to forward-transformed space:
   $\mathbf{y}_g = \mathbf{z}_g \odot \mathrm{sd\_vec}_g + \mathrm{mean\_vec}_g$.
4. Invert the per-element transform. All three groups use `kind='sqrt'` so
   $\mathbf{em}_g = \max(\mathbf{y}_g, 0)^2$
   (sign is intentionally lost by the forward $\sqrt{}$; both target and
   prediction go through the same clamp so gradients are consistent).
5. Restore per-row geometry:
   $\hat{\mathbf{c}}_g^{(r)} = \mathbf{em}_g \odot \mathbf{sc}_{\rm sci}^{(r)}$
   where $\mathbf{sc}_{\rm sci}^{(r)}$ is the per-row per-coefficient geometry
   factor from `airglow_geometry_scale(ctx_sci, ...)` (see §2).

The identical chain is applied to the target scaled-score
$\mathbf{s}_g^{{\rm true},{\rm scaled}}$ so that both paths are in the same
physical units.

**Flux basis matrix.** A row-independent basis matrix
$A_g \in \mathbb{R}^{n_g^{\rm coef} \times n_\lambda}$ is precomputed once,
before the ensemble seed loop, by `_precompute_flux_basis_and_geometry`, which
builds the corpus's own basis with `data.make_corpus_basis_decomposer`: the
variant's telluric decomposer at the representative (median-airmass) row, with
the real median LSF width and a nominal LSF surface.  Its `matrix_moon`,
`matrix_zodi` and `matrix_diffuse` rows — one per moon or zodi spline knot and
one per diffuse component (HO2, FeO, O2Ac, in that order) — are used on the
full native grid ($n_\lambda = 12\,401$): no resampling, binning or stride is
applied.

**Per-row loss.** With $A_g$ on device, the per-row per-group loss is

$$
\mathcal{L}_g^{\rm flux,(r)} \;=\; s_g \cdot \frac{1}{n_\lambda} \sum_{\lambda=1}^{n_\lambda} w_{r\lambda}\Big( \hat{\mathbf{c}}_g^{(r)} A_g - \mathbf{c}_g^{{\rm true},(r)} A_g \Big)^2_\lambda
$$

with $w_{r\lambda} = 1/(y_{r\lambda}\,s(\lambda))$ the photon-noise weight of
the row's total observed science flux (absolute sensitivity $s$, variance floor
5% of the row median, normalised to mean 1 per row; `flux_pixel_weighting`).

and the group is added into the total loss with the same balancing weight
$w_g = m_g / \sqrt{n_g^{\rm score}}$ (§3.6.2) as the score-space groups, so
`moon_group_weight` and `zodi_group_weight` retain the calibrated meaning they
had under the score-space loss.

The per-group **scale-match factor**

$$
s_g \;=\; \frac{\text{median}_{r \in \text{train}}\; L_g^{\rm diag,(r)}}{\text{median}_{r \in \text{train}}\; \overline{f_g^{{\rm true},(r)}(\lambda)^2}}
$$

is computed once at trainer entry: it aligns the median per-row flux-MSE with
the median per-row diagonal-Huber loss (§3.6.2), so `moon_group_weight` and
`zodi_group_weight` keep the balancing meaning they would have under the
score-space form.

**It replaces, not augments.** For a group in `flux_mse_groups` the
flux-space term *substitutes* for that group's coefficient-space
`smooth_l1` (an `if`/`else` in `compressed_loss`), so moon, zodi and continuum
are scored in flux units while the remaining three groups are scored in
coefficient units.
The term is also strictly per-group — each group's own coefficients through its
own basis — so no moon+zodi **sum** is constrained anywhere.

**Log-amplitude term.** The per-pixel term above is an *absolute* MSE, so it
is dominated by the brightest rows and is comparatively insensitive to a
fractional brightness error on a faint one.  A scale-free companion term is
therefore added for the groups named in `flux_amp_lambda`:

$$
\mathcal{L}_g^{\rm amp,(r)} \;=\; \lambda_g \left[ \log\!\left( \frac{\sum_\lambda \hat f_g^{(r)}(\lambda) + \epsilon_g}{\sum_\lambda f_g^{{\rm true},(r)}(\lambda) + \epsilon_g} \right) \right]^2 ,
$$

penalising the same fractional amplitude error equally at every brightness.
The floor $\epsilon_g$ is 5% of the group's median training amplitude
(`flux_amp_floor_frac`), so rows far below the typical brightness — dark-time
moon, which sits at the decomposition's 2% share floor — contribute
negligibly rather than dominating a log ratio.  It is **additive**, not a
replacement.

The deployed setting is $\lambda_{\rm moon}=5$, $\lambda_{\rm zodi}=0$,
$\lambda_{\rm continuum}=0$.  The zodi is excluded on physical grounds: its
amplitude is pinned by the decomposition's absolute Leinert anchor on the
majority of moon-up rows, so it is close to a deterministic function of
geometry there and a further amplitude penalty trades spectral shape for
amplitude accuracy the network already has.

### 3.6.2 Weighted Huber loss on the score-space groups

For $g \in$ {`mesospheric`, `ionospheric`, `atomic`}, let
$\hat{\mathbf{s}}_g^{(r)},\,\mathbf{s}_g^{{\rm true},(r)}\in\mathbb{R}^{n_g^{\rm score}}$
be the predicted and target scaled-scores for row $r$ and group $g$. The
per-row per-group loss is

$$
\mathcal{L}_g^{(r)} \;=\; \frac{w_g}{n_g^{\rm score}}\,\sum_{k=1}^{n_g^{\rm score}} w^{\rm pe}_{g,k,r}\,\mathrm{Huber}\big(\hat{s}_{g,k,r} - s^{\rm true}_{g,k,r}\big),
$$

with Huber the `smooth_l1` loss (quadratic below 1, linear above, in scaled
units); the total is the mean over the six groups of $w_g\langle\mathcal{L}_g\rangle_r$.

with $w_g$ the **per-group balancing weight**

$$
w_g \;=\; \frac{m_g}{\sqrt{n_g^{\rm score}}},
$$

where the multipliers $m_g$ are the current deployed defaults:

- $m_{\rm moon}=3.0,\; m_{\rm zodi}=2.0,\; m_{\rm continuum}=1.0,\; m_{\rm mesospheric}=1.0,\; m_{\rm ionospheric}=1.0,\; m_{\rm atomic}=1.0$.

These weights apply to the flux-space groups too (§3.6.1).

### 3.6.3 Heteroscedastic per-element weights

The decomposition-side coefficient uncertainties $\boldsymbol\sigma_{\rm coef,sci}$
(§1.9) are propagated through the compressor with a first-order Jacobian:

$$
\sigma_{s_{g,k,r}} \;=\; \big\lVert \nabla_{c_g} s_{g,k} \big\rVert \cdot \sigma_{{\rm coef},g,r},
$$

divided by the RobustScaler column scale so the weights live in the same space
as $\hat{\mathbf{s}}_g$. The per-element weight is

$$
w^{\rm pe}_{g,k,r} \;=\; \frac{1}{\sigma_{s_{g,k,r}}^2},\qquad \sigma_{s_{g,k,r}} \ge \sigma^{\rm floor}_{g,k},
$$

with $\sigma^{\rm floor}_{g,k}$ = `coef_err_sigma_floor_rel[g]` × per-column
median finite sigma. The floors are 5% for moon, zodi, continuum, and
ionospheric; 20% for mesospheric and atomic, because their $p_{99}/p_{50}$
sigma ratios span $10^{3}$–$10^{6}$. Weights are
normalised so $\mathbb{E}_{\rm train}[w^{\rm pe}]=1$ per column; missing/
boundary sigmas fall through to the floor.

**Joint covariance diagnostic path.** The `coef-cov-loader` notebook cell
opens each of the three decomp FITS products, reads the `COEF_COV_MOON` and
`COEF_COV_ZODI` HDUs when present, slices them by `filtered_triplet['row_index']`,
and attaches `coef_cov_{moon,zodi}_{near,far,sci}` to `filtered_triplet`.
Two diagnostic cells consume these:

- `truth_conditioned_sigma_calibration` — computes the Mahalanobis residual
  $|z|_{\rm joint,g} = \sqrt{\mathbf{r}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{r}_g / n_{\rm active}}$
  per test row and reports its median vs the calibrated target
  $\sim\sqrt{\text{med}(\chi^2_n)/n}\approx 0.7$.
- `moon_sigma_investigation` — reports the joint per-block SNR
  $\sqrt{\mathbf{c}_g^{\top}\mathbf{\Sigma}_g^{-1}\mathbf{c}_g / n_{\rm active}}$
  binned by moon regime, alongside the independence-assuming
  $\lVert\mathbf{c}_g\rVert / \sqrt{\sum_j\sigma_j^2}$ for direct comparison.

These are diagnostic-only: the deployed loss uses only the diagonal
per-element $w^{\rm pe}$ weight above.

### 3.6.4 Physics-space pixel WRMSE (diagnostic only)

Additionally the notebook tracks a **physics-space** metric per row:

$$
{\rm WRMSE}^{(r)} \;=\; \sqrt{\frac{1}{N_\lambda}\sum_\lambda \big(\hat{y}_r(\lambda)-y_r(\lambda)\big)^2 / \sigma_{\rm tot}^2(\lambda)}
$$

with $\sigma_{\rm tot}$ from `FLUX_SIGMA_TOTAL`. This is used for diagnostic
reporting (worst-row visualisations, extrapolation quality `eRMSE`, per-
lunation drift) but is **not** part of the training gradient. The training
loss is §3.6.1 (moon, zodi, continuum) plus §3.6.2 on the remaining three
groups.

### 3.6.5 Row weights

Row weights are **uniform**: $w_{\rm row}=1$ for every row.  Per-row
reweighting by observing regime was evaluated and not adopted; the changelog
records the schemes tested.  The mechanism is not a config knob — the row
weight vector is constructed as `np.ones(...)` in `trainer.py`.

### 3.6.6 Post-training per-coef Jensen-style lift calibration

After the ensemble is assembled, a **single-pass empirical bias correction**
is computed on the train+val subset and stored in `jensen_corrections`,
applied at inference in `expand_scores_to_coefs`. Three flavours:

- **`moon` (per-coef vector)**: for each of the 15 `Moon_bs` knots the ratio
  $\ell_k = \bar{c}^{\rm true}_k / \bar{c}^{\rm pred,\,naive}_k$ is computed
  from calibration-row means over the **rule-free** rows only — rows whose moon
  amplitude the moon-down rule (§3.9.2) replaces at inference are excluded,
  because their pre-rule prediction is not what is delivered (falls back to
  all rows if fewer than 200 remain), ignoring near-zero knots
  ($|\bar{c}^{\rm pred}| < 0.05|\bar{c}^{\rm true}|$) which default to
  $\ell_k=1$. Clipped to $[0.7, 1.4]$ to protect against a broken knot
  cascading. The moon per-coef lift nulls the spectral tilt bias that a
  single scalar cannot correct.
- **`zodi` (per-coef vector, regime-conditioned)**: three moon-alt regime
  buckets are lifted independently (`moon_up` for `moon_alt > 10°`,
  `moon_horizon` for `-10° \le moon_alt \le 10°`, `moon_down` for
  `moon_alt < -10°`) with a 5°-wide smooth boundary. Each bucket uses the
  same per-coef ratio $\ell_k$ formula on its subset; if a bucket has fewer
  than 30 calibration rows it falls back to the global per-coef lift.
  Clipped to $[0.7, 1.4]$. At inference the row's `moon_alt` selects the
  bucket (smoothed at the boundaries).
- **`continuum`, `mesospheric`, `ionospheric`, `atomic` (scalar)**: fitted on
  the **flux-weighted amplitude** $A_g = \mathbf{c}_g\cdot\mathbf{v}_g$ with
  $\mathbf{v}_g$ the native-grid template integrals,
  $\ell = \overline{A_g^{\rm true}}/\overline{A_g^{\rm pred,\,naive}}$, clipped
  to $[0.5, 2.0]$, and broadcast to every coefficient of the group. The
  unweighted mean coefficient is a different functional and can have the
  opposite sign (the 357 OH template integrals span $\sim10^{9}$ and are
  anti-correlated with coefficient size). Skipped if the amplitude is
  degenerate or its relative magnitude is below 5%.

At inference each predicted coef vector for group $g$ is multiplied by
$\boldsymbol\ell_g$ (per-coef for moon, regime-conditioned per-coef for zodi,
broadcast scalar for the others) inside `expand_scores_to_coefs`. This closes
the residual coherent bias that the network's own outputs leave in place
without adding trainable parameters.

## 3.7 Optimisation and splits

Training runs are deliberately conservative: a fixed AdamW schedule, early
stopping on a night-held-out validation set, and a 10-seed ensemble whose
mean is the deployed prediction.  The interesting design work is upstream in
§3.3–§3.6; §3.7 records the training-loop settings for reproducibility.

- **Optimiser**: AdamW, `lr=1e-3`, `weight_decay=2e-4`, `grad_clip=1.0`,
  `n_epochs=400`, `patience=100` (early stop on val loss; judged in flux
  space, where longer training wins even though coefficient metrics disagree). The blend-α direct-
  parametrised params (§3.5) sit in a separate optimiser group with
  `weight_decay=0` and are clamped to $[\varepsilon, 1-\varepsilon]$ with
  $\varepsilon=10^{-3}$ after each step. The ctx-α predictors (§3.5) share
  that group.
- **`alpha_lr_mult=30.0`**: the blend-α group trains at
  `alpha_lr_mult × lr`. At the default 1.0 α stays pinned to its init (§3.5);
  at 30 both the scalar and ctx-α paths reach their own optima, and the ctx
  predictors develop real per-row structure instead of sitting flat.
- **Batching**: `batch_size=512`. All training data (~O(10 MB) after RobustScaler
  standardisation of scores and ctx, fit on train rows only) is staged once
  onto the training device and iterated with `torch.randperm` — no per-batch
  CPU→device transfers.
- **Split**: rows are grouped by `night_id = floor(mjd - 0.5)` and stratified
  by lunar phase into train / val / test partitions via
  `split_indices_by_moon_phase` (defined in `mlp_predictor.ml_utils`). Whole
  nights are held out because neighbouring exposures within a night are
  correlated. The
  same random seed splits the corpus identically across ensemble members so
  they all see the same held-out test rows.
- **Ensembling**: 10 seeds `(42, 43, ..., 51)` train independently. At
  inference the ensemble mean of the per-seed physical-space predictions is
  returned; the per-row std is also reported as an epistemic-uncertainty
  flag by the `predictive_uncertainty_ensemble` diagnostic.

## 3.8 Diagnostics

The `mlp_predictor.diagnostics.Diagnostics` class runs the diagnostic cells
against a captured globals dict, so the cell bodies (stored verbatim in
`skysub/mlp_predictor/diagnostics_cells/*.py`) stay readable and re-runnable
from the notebook. The active suite covers:

- **Coefficient-space quality**: `coef_hist_prepost`,
  `amplitude_error_vs_ctx` (per-family integrated-amplitude error against the
  context that physically drives that family),
  `naive_baseline` (vs `copy_near` / `near_geo` / `mean_geo`),
  `rmse_dual_diagnostic`, `rmse_worst_stability`.
- **Full-spectrum reconstruction**: `full_spectrum_single_row`,
  `full_spectrum_batch_rmse` (reconstructs the full held-out val + test split
  from the corpus, in parallel; per-row per-component residuals and the
  prediction / decomposition $\chi^2$ ratio, with stable per-row colours across
  the spectrum panels),
  `worst_recon`, `pipeline_state_check`, `physical_space_cap`.
- **Regime & context slicing**: `per_lunation_drift`, `per_context_slice`,
  `sky_arm_zodi_bias` (verdict keys off the **ML** bias, gated on both a
  slice-relative and an all-test-relative tolerance; a large arm bias with a
  near-zero ML bias is reported as the ML *working*, not as a tuning target).
- **Sigma calibration & ensemble spread**: `resid_over_sigma_per_group`,
  `resid_vs_sigma_per_decile`, `truth_conditioned_sigma_calibration`
  (Mahalanobis $|z|_{\rm joint}$ on the persisted COEF_COV blocks),
  `moon_sigma_investigation` (joint vs marginal SNR by moon regime),
  `predictive_uncertainty_ensemble`, `ensemble_spread_calibration`
  (reliability curve for ensemble std vs actual error).
- **Missing-feature and noise-floor tests**: `residual_ctx_attribution`
  (linear + RF fit of per-row residual RMS on ctx, 5-fold CV R²; also appends
  diagnostic-only `helio_lon_*` probe columns the model never sees, so a
  candidate feature can be tested before spending a training run),
  `sky_arm_disagreement_floor` (ML error vs sky-arm intrinsic disagreement Δ),
  `wavelength_residual_atlas` (aggregated pred − true λ, reusing the batch
  cell's reconstructions, absolute + fractional per band; plus a per-coefficient-group
  attribution of the trimmed-mean residual with each group's `flux_share_%`
  and own-units `self_bias_%`, a chi2 gate that drops failed decompositions
  from the sample, and a tail-concentration report).

  **Reading `sky_arm_disagreement_floor` correctly.** The cell's printed
  interpretation says err/Δ ≈ 1 is the floor. That is too optimistic. With
  Δ = ½·RMS(near − far) and both arms carrying independent noise σ,
  Δ ≈ 0.707σ, while a *perfect* predictor still scores err = σ against a sci
  truth carrying its own independent σ — i.e. **err/Δ ≈ 1.41 is the floor**
  for a group whose truth is as noisy as the arms. Reading mesospheric's 1.40
  as headroom is a misreading. Values well below 1 (ionospheric ≈ 0.33, zodi ≈ 0.67)
  mean the head is using ctx the arms do not carry.
- **Optional deeper cells**: `swrmse_coef_map`, `wrmse_vs_ctx_correlation`,
  `wrmse_vs_ctx_scatter`, `per_seed_vs_ensemble`. These are commented out by
  default in the notebook (they add ~1 min each) but are supported by the
  `Diagnostics` class the same way.

Each call cell in the notebook opens with two comment lines — `# Targets:`
(what the plot / table shows) and `# Look for:` (what interpretation flags a
regression) — so the diagnostic body can be inspected and rerun in isolation.

## 3.9 Constraint-derived amplitudes

Half the corpus has one of its two continuum amplitudes set by a decomposition
constraint rather than by the data (§1.4.3). Where that is true the target is
a known deterministic function of geometry, so it is **derived at prediction
time rather than learned**. Both rules only ever rescale a coefficient block,
so the spline *shape* stays whatever the network predicted; both are no-ops on
artifacts that do not carry them, so older ensembles are bit-identical; and
both are idempotent and linear in the coefficients, so the ensemble mean of
rule-satisfying members satisfies the rules too. The zodi rule runs **first**,
because the moon rule reads the predicted zodi amplitude and should read the
corrected one.

### 3.9.1 Bright moon: the zodi ceiling

Where the Leinert anchor binds, $\int\!{\rm zodi} = \kappa_z c\,Z_{\rm pred}$
exactly. $Z_{\rm pred}$ includes the ecliptic correction of §1.4.3 (c) when
the corpus was built with it. The rule reads it from the cache's
`zodi_total_po`, which is why the cache must match the decomposition's
`ZODICORR`. `apply_zodi_ceiling_rule` does two things with
$S\,Z_{\rm pred}$, with $S$ fitted per run on gated training rows:

* **clamp**, every valid row. $S\,Z_{\rm pred}$ is a hard upper bound —
  over 14 457 corpus rows the fitted zodi exceeds it on 0.01% and never by
  more than 0.0024 dex — so clipping to it can only move a prediction toward
  the truth.
* **snap**, on `moon_frac_po > 0.6`, guarded by `snap_frac = 0.9` so it fires
  only where the network already predicts ≥ 90% of the ceiling; a gated row
  the network puts well below it keeps its own answer, which is how genuinely
  interior rows survive.

The gate is on the *predicted moon fraction*, not altitude, and the guard is
what lets the loss stay untouched: `zodi_ceiling_amp_free` defaults to
**False** because the bright-moon zodi amplitude is a deterministic function
the network partly learns — and now receives `zodi_po_log10` for — and its
learned value is exactly what the guard uses to separate pinned rows from
interior ones. Blinding it would destroy the signal the guard runs on. That is
the deliberate asymmetry with §3.9.2.

$S$ is **basis-dependent**: it is fitted with the trainer's native-grid flux
bases, and the 1.3.2 ensemble carries $S = 3.2266$ (5728 fit rows) against the
earlier full-grid 3.2244. Sanity-check it for stability run to run, not against
a number.

### 3.9.2 Dark time: the moon share floor

Where the share bracket has collapsed onto $\epsilon$ the moon is a ghost
(§1.4.3), so:

* the flux loss is made **amplitude-blind** on those rows —
  `moon_down_amp_free` rescales the prediction to the true integral before the
  pixel term and drops the log-amplitude term, leaving the shape to train. The
  rescale factor is left attached to the autograd graph on purpose, so the
  rescaled prediction has *exactly* the true integral and the amplitude
  gradient is identically zero rather than merely small;
* the amplitude is restored at prediction time from the **near arm's measured
  moon/zodi ratio**, $\int\!{\rm moon}({\rm sci}) =
  {\rm clip}\big((\int\!{\rm moon}/\!\int\!{\rm zodi})_{\rm near},0,R\big)
  \cdot \int\!{\rm zodi}({\rm sci})$, falling back to the fitted constant $R$
  when the near arm is unusable.

**The gate is `moon_frac_po` ≤ $\epsilon/\kappa_f$, not `moon_alt` ≤ 0.**
Scattered moonlight with the moon just below the horizon is real and the model
carries it as $\exp(-8.2182\tanh(\text{depth}/5°))$ — only $-0.70$ dex at
$-1°$, $-2.72$ at $-5°$, saturating at $-3.57$ near $-15°$. Testing the
condition that actually defines the regime beats every altitude cut on
coverage *and* exactness at once:

| gate | cover | pinned | \|d\|>0.05 |
|---|---|---|---|
| `moon_alt` ≤ 0 | 49.7% | 92.6% | 7.4% |
| `moon_alt` ≤ −4 | 47.0% | 96.1% | 3.9% |
| `moon_alt` ≤ −12 | 41.4% | 95.7% | 4.2% |
| **`moon_frac_po` ≤ 0.00667** | **47.8%** | **96.2%** | **3.7%** |

No fixed altitude can match it because the depth term *multiplies* phase and
separation: a thin crescent 2° down contributes nothing while a full moon 5°
down still does. Per bin the fitted moon is pinned on 100% of rows at −8…−6°
but only 54.2% at −4…−2°, 21.2% at −2…−1° and 1.1% at −1…0°, so the last few
degrees were pure contamination at ~1 dex each. `moon_alt` survives only as
the fallback when a triplet has no `moon_frac_po`.

The ratio transfer exists because the share is **bimodal**, not constant:
96.25% of gated rows sit on the ceiling and 3.25% want no moon at all, with
0.49% in between. It is neutral on the deployed (filtered) split and clearly
better on unfiltered rows — see §3.9.3 for why that distinction matters, and
the changelog for the honest null result.

### 3.9.3 The degeneracy flag, and why filtered metrics mislead here

The zero-moon rows are **failed fits**, not a physical state: median reduced
$\chi^2$ 2.09 against 0.139 on healthy dark rows, with the continuum split
moon 0.0001% / zodi 100.0% / diffuse 0.0% against 1.04 / 43.1 / 55.8. The QP
has put everything into `Zodi_bs` and zeroed both other families — the
zodi/diffuse degeneracy at its limit, the same pathology `diffuse_zeroed_frac`
(§1.11) exists for, and 67.2% of them trip that gate too.

Nothing downstream can recover such a row: the ML is predicting from
degenerate inputs. So `trainer.degenerate_continuum_flag` marks them, using
**only the two sky arms** — both with $\int\!{\rm moon}/\!\int\!{\rm zodi} <
0.5R$ — which gives 87.8% precision at 94.8% recall on 1.87% of rows, cannot
misfire with the moon up (0 of 7543 such rows; their 1st-percentile ratio is
6× the threshold), and needs no science-side truth.

> **The training filters do not run in production.** Every science exposure
> needs a sky subtraction, so rows §1.11 would drop are still delivered. This
> cuts both ways: a fix aimed at them will look like a no-op on the notebook's
> filtered metrics — the ratio transfer above moved the test set by nothing
> because the filters already remove ~93% of the rows it targets — and a
> pathology that looks rare in training may not be rare in delivery. Measure a
> fix on the population whose metric you are trying to move, and flag what you
> cannot fix.

## Changelog and measured record (split-zodi variant)

The chapters above describe the deployed method. This cell holds the evidence
behind it: the A/B tests, the settings that were tried and rejected, and the
reasoning that is historical rather than part of the current design.

> **Numbers are not comparable across corpus boundaries.** The decomposition
> was regenerated twice: on 2026-09-04 with the moon/zodi identifiability
> constraints (§1.4.3), and on 2026-09-05 with the recentred Leinert anchor.
> Both moved the split *and* the moon+zodi sum. In particular the near–far
> moon disagreement fell 12× at the first boundary ($p_{50}$ 0.0554 → 0.0045),
> so any `err/Δ` from the sky-arm floor diagnostic has a different denominator
> either side. Use `naive_baseline` per-group gains for cross-corpus
> comparison, and note that even those shift when a baseline improves: the
> moon's gain fell +24.0% → +15.7% across the second boundary while its
> *absolute* error also fell, because `B0_copy_near` improved faster.

> **Seed-block noise floor.** On `new-oh-3` an identical `default` config gave
> `mean_eRMSE` 7.9759 and 7.8525 on two 10-seed blocks — a 0.12 swing,
> comparable to most effects measured here. Single-block deltas below that are
> hypotheses, not findings. Several results in this record were overturned by
> replication on a second block.

---

#### Caches: one copyable, one per-variant

**The moon/zodi model cache copies over.** It depends only on META geometry,
the LSF and the wavelength grid, and those are byte-identical between the two
corpora — WAVE, all three LSF planes and the META geometry columns hash the
same, 1447 every10 / 14469 full-corpus rows both sides. Copied in and verified
to load and validate at both scales, which saves the ~26 min rebuild.

**The wavelength basis must be built per variant** (and, until 2026-09-18,
its cache must not be copied). `OH_i` is a different line group in each flavour: built both ways,
OH centroids differ by a median **45.1 Å** (max 837 Å) and ATOM by 0.004 Å
(max 15.6 Å), while Moon / Zodi / diffuse are **exactly 0**. Those arrays set
the per-coefficient extinction and van Rhijn geometry, so the wrong ones
corrupt training inputs rather than just a plot.

`coef_wavelengths_from_basis` now accepts `decomposer=`, and
`build_coef_wavelengths` builds the variant's own basis for it — for
telluric from the median-`sci_airmass` row, since the transmission enters a
centroid only as a smooth weight (median 0.9997) and so matters far less than
the line grouping it exists to capture. It also now reads centroids straight
off `design_names` / `design_matrix` rather than doing 388 $e_j$
reconstructions: **bit-identical** (max $|\Delta\lambda|$ and $|\Delta k|$
both exactly 0.0 over 33 probed coefficients spanning every family) and **<1 s
instead of ~13 min**, with the old loop kept as a fallback.

**The cache was then removed entirely (2026-09-18).** Timed end to end on
`gaia-stars-mask-telluric-chi2`: cold (build + extinction fit) **1.6 s**, warm
(cache hit + extinction fit) **0.4 s**, identical wavelengths. Paying 1.2 s a
run is not worth keeping a stale-cache failure mode that had already fired
twice — the coefficient NAMES are identical across variants, so a name-keyed
cache cannot tell two bases apart. `wavelength_cache_matches_corpus`,
`populate_wavelength_cache` (now `build_coef_wavelengths`), the `cache_path`
plumbing through `coef_wavelengths_from_basis` and
`DataConfig.wavelength_cache_path` are all gone; any leftover
`coef_wavelengths_basis_v4.npz` in a corpus directory is unused and can be
deleted.

> **The pre-existing bug this closes.** `gaia-stars-mask-cont2`'s committed
> `coef_wavelengths_basis_v4.npz` was **stale in OH**: median **306 Å** from
> what the current split-zodi basis produces, max 4813 Å (OH_356 cached
> 4768 Å against 8618 Å computed), while Moon/Zodi/diffuse/ATOM agreed
> exactly. The deployed cont2 runs therefore used OH wavelengths off by
> hundreds of Ångstrom, and because the file carried no suffix stamp it was
> treated as legacy and reused. Whether that moves the ML numbers is
> untested; it can no longer recur.

### Improved OH line strengths and zodi model become the defaults (2026-09-24)

Both are tested on 1000 rows locally and not yet run on the full corpus. From
this date `decompose_parallel` defaults to
`--fit-model palacecorr-aijc-vnf-split-zodi-lsf-spline2d` and to
`SPLIT_ZODI_ZODI_CORRECTION = "lvm-ecl-2026-09"`. The method is in §1.2.3 and
§1.4.3 (c).

**Test set.** 1000 rows from the 1.3.2 corpus, 50 in each of 20 cells:
moon down or up, × five $|\beta|$ bins, × $|\Delta\lambda| < 120°$ or
$\ge 120°$. They span 568 nights, and the three arms were fitted per row.

* **Local runs match the cluster.** A local rerun with the old defaults
  matches the cluster corpus to $3\times10^{-11}$ median (p99
  $8\times10^{-6}$) in the coefficients, and its reduced $\chi^2$ ratio is
  1.00000.
* **Each change was tested alone** against that baseline. Median per-row
  $\chi^2$ ratios are new/old, under one fixed photon weighting (`ab_score`).

**Improved OH coefficients** (`palacecorr`; Katkov, `4f3cb16`):

| sci | r | z | full |
|---|---|---|---|
| OH-dominated pixels | **×0.030** | **×0.28** | **×0.10** |
| whole band | ×0.51 | ×0.65 | ×0.69 |
| OH-free pixels | ×0.996 | ×0.987 | ×0.997 |

* Every row improves, moon-up and dark alike. sky1 and sky2 behave the same
  way: OH pixels ×0.031 and ×0.033 in r, full band ×0.66 and ×0.56. Blue is
  ×0.9994.
* The systematic residual at the OH lines, stacked over rows, falls from 1.1%
  to 0.14% of the line peak in r.
* The fit is still far from the noise: absolute $\chi^2$ against one fibre's
  noise goes 1073 → 630.
* **The coefficients barely move** (OH: median 0, $p_{90}$ 0.8%); the OH
  flux moves instead, +0.11 dex in r.
* The continuum partition, flags and reversals are unchanged. OH transfer
  consistency (the scatter of sci/near OH flux) stays at 0.017 dex.

**Zodi model** (`lvm-ecl-2026-09`):

* The dark-time trend of fitted/anchor zodi with $|\beta|$ disappears:
  +0.03 → +0.25 dex from ecliptic to pole becomes a flat +0.04 to +0.07.
* Dark rows on the anchor bound: 12% → 1% (sci), 8% → 0% (sky1). The free
  dark rows' scatter falls 0.114 → 0.108 (sci), 0.112 → 0.098 (sky1) and
  0.126 → 0.081 (sky2).
* On moon-up rows the zodi rises +0.05 to +0.08 dex and the moon falls about
  0.01 dex; the total is unchanged to $4\times10^{-5}$. Rows on the ceiling
  go 88% → 84%.
* Median $\chi^2$ is unchanged in every band, and 61–67% of rows improve in
  blue.
* Reversal retries on moon-up rows fall 4.0% → 2.4%, with no fit failures.

**The zodi fit had to be redone once**: a first version was fitted against
broken context geometry. The ML context columns `sun_sep` and `moon_sep` had
been ICRS separations, which put the Sun and Moon at the solar-system
barycentre. `moon_sep` was really $180° -$ elongation, and `sun_sep` was
close to noise ($\rho = +0.13$ with the true elongation). The first
correction fit took its longitude from `sun_sep` and got the longitude
dependence wrong; the deployed fit uses true geometry.

* The context bug is fixed as `CTX_GEOMETRY_VERSION = 2`: topocentric
  separations, a geocentric Sun for the ecliptic longitude, and inference's
  moon-phase sign fixed.
* `load_ensemble` warns on a version mismatch.
* The META that `medians_computation` writes was always correct.
* A v2 retrain was neutral: $\Delta\log\chi^2 = -0.40\%$, CI
  $[-1.27, +0.30]$.
* The decomposition was never affected, because it computes its own geometry.

**What has to change downstream:**

* **Retrain after the cluster run.** The moon-up moon and zodi targets move,
  and the cache must be rebuilt: `moon_model_cache` takes the correction from
  `ZODICORR` and refuses a mismatch.
* **ML-side support for `palacecorr`.** `data.DECOMP_VARIANTS` has a
  `telluric-palacecorr` entry that carries its OH file (`palace_oh_suffix`).
  * `telluric_row_kwargs` puts the file in the per-row basis bundle, and
    `make_reconstruction_decomposer` takes it from there. Every
    reconstruction, the wavelength basis and the training flux loss therefore
    use the OH templates the row was fitted with.
  * `make_telluric_row_lookup` now requires `decomp_suffix`.
  * An unregistered suffix from the telluric family now raises. Before, it
    silently fell back to the non-telluric class with the canonical OH file.
* **Provenance.** Every product's primary header now carries `ZODICORR`, and
  `palacecorr` runs also carry `OHFILE` and `OHRIDGE`. Before this date the
  full (non-compact) writer did not stamp `ZODICORR`, and the cache reads a
  missing key as `"none"`.

---

### Telluric variant, full run (2026-09-17): better spectra, and two metrics retired

First complete notebook run on `gaia-stars-mask-telluric` (10 045 rows after
filtering, n_test = 1011; cont2 was 9 943 / 972).

**Flux space improves across the board.** The decomposition floor drops 32% and
the reconstruction follows it down:

| | cont2 | telluric |
|---|---|---|
| reconstruction $\chi^2$ | 3.889 | **2.768** |
| self-fit floor | 3.572 | **2.437** |
| ratio to floor | **1.09x** | 1.14x |
| blue $\chi^2$ / floor / ratio | 1.202 / 0.955 / 1.26x | 1.169 / 0.989 / **1.18x** |
| sci pRMSE | 4.925e-15 | **4.068e-15** |

**Continuum amplitudes improve a lot; moon/zodi/OH slightly worsen.**
continuum MAD 0.01434 $\to$ **0.01086** ($-24\%$), HO2 0.02150 $\to$ 0.01504,
FeO 0.01791 $\to$ 0.01408, O2Ac 0.01672 $\to$ 0.01295. Against that: moon
0.01163 $\to$ 0.01230, zodi 0.00022 $\to$ 0.00068 (both inside or near the
repeat-run yardstick), OH total 0.00648 $\to$ 0.00678.

Naive-baseline gains rise for the families that transfer well — moon +35.1%
$\to$ **+42.2%**, zodi +40.5% $\to$ **+46.6%** — and fall for continuum
(+9.4% $\to$ +3.7%), which is consistent with the continuum targets having
become easier for the naive baseline too.

> **Two metrics are now retired for cross-variant reading.** `mean_eRMSE`
> (8.516 $\to$ 5.319), the mesospheric coefficient sRMSE and GROUP-EQUAL are
> NOT comparable between variants: the OH coefficients are a different basis on
> a different scale. The mesospheric coefficient gain reads **−75.4%** and
> GROUP-EQUAL **−55.3%**, while the flux companion reads −0.2% — the
> degeneracy is far worse here than the −8.3% seen on cont2, because grouping
> OH by upper level makes the coefficients much more degenerate among
> themselves. Measured offline on one corpus with both bases, the OH flux gain
> is +5.4% (split-zodi basis) / +4.3% (telluric basis), so the basis is worth
> ~1 pp and the rest of the spread against −0.2% is the row set. OH transfer
> is near parity with copying the near arm either way.

**Two confounds in this run, both now fixed for the next one.**

1. *The wavelength cache was stale for every earlier run.* cont2's committed
   `coef_wavelengths_basis_v4.npz` had OH centroids a median 306 Å (max
   4813 Å) from what its own basis produces. The telluric run is the first
   with a correct cache, so it is **not a clean A/B** — the per-coefficient
   extinction and van Rhijn geometry changed too. Isolating it needs a cont2
   rerun with a rebuilt cache (the stale file has been deleted).
2. *The amplitude weights and the flux companion used the split-zodi basis.*
   `amplitude_error_vs_ctx` and `naive_baseline` built
   `SkyDecompLSFSurfaceIterative` unconditionally, so on this run the OH
   amplitude and the flux companion were the wrong linear functional (moon /
   zodi / diffuse are byte-identical between variants, so only OH is
   affected). Both now build the variant's basis from a representative row,
   supplied as the `TELLURIC_BASIS_KW` extra; the transmission moves $v_g$ by
   a median 0.7% (max 21%) across the airmass range, and the amplitude metrics
   are ratios using the same $v$ on both sides, so a representative row is
   adequate.

> **Correction to earlier entries.** The OH wavelength sub-band rows
> (3600–5500 / 5500–7500 / 7500–9800 Å) in every previous headline were an
> artefact of that stale cache. Rebuilt correctly, **all 357 OH centroids land
> in 7500–9800 Å in BOTH variants** — each OH coefficient is a multi-band
> group whose 90% flux span is a median ~800 Å across a median 2 clusters, so
> its flux-weighted centroid is not a line position. The claim that the
> post-lift-fix OH bias was “confined to the blue” is withdrawn, and the
> sub-band split carries no information as currently defined.

---

### Telluric decomposition variant (2026-09-17): switchable, and it moves the floor

`palace-aijc-vnf-split-zodi-lsf-spline2d` in `gaia-stars-mask-telluric/`
(suffix `_palace_aijc_vnf_split_zodi_lsf_spline2d`): PALACE VNF lines with OH
grouped by (v_upper, N_upper, F_upper), a continuous M-spline 2-D LSF, and
**telluric absorption in the fit**. Selected with `DECOMP_VARIANT` in the
config cell; `'split_zodi'` keeps the production path.

**It fits the data substantially better.** Decomposition self-fit $\chi^2$
(absolute, one 900 s fibre, `BESTFIT_LSF` against `FLUX_SCI` — no
reconstruction involved, so no re-assembly error), every10 sci:

| | full band $p_{50}$ | $p_{90}$ | blue $p_{50}$ |
|---|---|---|---|
| cont2 (split-zodi) | 4.004 | 23.52 | 1.174 |
| **telluric** | **2.823** | **20.46** | 1.185 |

Full band **−30%**, better on **85.7%** of rows, while the blue is unchanged
(+1.0%, better on only 44.5%). That is exactly the signature of fixing OH: the
gain sits in the red where OH lives and where telluric absorption bites, and
the blue continuum is untouched. It therefore moves the **floor** the ML is
measured against — the reconstruction/self-fit ratio is the metric to read
after retraining, not the absolute $\chi^2$.

**Drop-in on the ML side:** the same 388 coefficient names in the same order
and the same extension layout, so targets, group indices and integrated
amplitudes are directly comparable.

**Not drop-in on the reconstruction side**, in two ways that both bite:

1. The LSF is stored as `continuous_mspline_density`, not
   `native_grid_channel_median`. `SkyDecompLSFSurfaceIterative` cannot read it
   at all — `evaluate_lsf_surface` raises *state does not contain a discrete
   11-tap LSF*.
2. Every family matrix is divided by that **row's** DRP transmission, so the
   design matrix is per-row and cannot be hoisted. Rebuilding costs ~0.17
   s/row/arm (~4 min for a 500-row sample across three arms).

Per-row inputs mirror `decompose_parallel._telluric_decomposer_for_row`:
`pwv_mm` from `META.pwv_med` falling back to **15.0 mm** when missing or
non-positive (−999.9 on ~9% of every10 rows); the DRP transmission always uses
**`sci_airmass`**, because the DRP divided *every* fibre by the transmission
along the science line of sight, while `source_airmass` is that arm's own and
is selected by the sky_near/far **label**, not the column name, since the
near/far assignment flips per exposure.

Plumbing: `data.DECOMP_VARIANTS`, `telluric_row_kwargs`,
`make_telluric_row_lookup`, `make_reconstruction_decomposer` and
`reconstruct_with_lsf(..., telluric=)`. The config cell sets
`cfg.data.decomp_suffix` from the variant and the diagnostics extras carry
`TELLURIC_ROW_FOR`, which the four reconstructing cells
(`full_spectrum_batch_rmse`, `full_spectrum_single_row`, `worst_recon`,
`wavelength_residual_atlas`) pick up. **Decide the variant from the suffix,
never from the META columns** — both corpora are built from the same input
stack, so `pwv_med` and the per-arm airmasses are present either way and
cannot discriminate.

Integrated amplitudes keep the **telluric-free** template integrals $v_g$ in
both flavours: the coefficient multiplies the same physical template and the
transmission is a per-row correction applied to it, so that is both the
physical amplitude and the only choice that leaves the two corpora comparable.

*Re-assembly fidelity*, reconstruct-from-stored-coefficients against the stored
`BESTFIT_LSF`: telluric **0.3%**, split-zodi **3–4%**. The new path is the more
faithful of the two; the split-zodi 3% is pre-existing and cancels in the
diagnostics because both sides go through the same re-assembly.

---

### `gaia-stars-mask-cont2` re-run (2026-09-12): reproduction, and the settled config

`W = 0.15` is the default again and cont2 is the corpus it produces. The
notebook was re-run against it — **restored** the saved ensemble rather than
retraining (`FORCE_RETRAIN = False`, same seeds 42–51, same split
train 7928 / val 1043 / test 972), so this is a reproduction check of the
diagnostics, not a new training run.

*Decomposition side reproduces exactly.* The self-fit $\chi^2$ floors are
identical to the run of 2026-09-11 — 3.572 full band, 0.9549 blue — which
confirms the same corpus and the same 500-row sample.

*Everything the diffuse constraints touch reproduces to <0.2%:* continuum
amplitude MAD 0.01434 $\to$ 0.01436, HO2 0.02150 $\to$ 0.02151, FeO 0.01791
$\to$ 0.01789, O2Ac 0.01672 $\to$ 0.01674, OH total 0.00648 $\to$ 0.00648,
and the three OH sub-bands identical to five digits.

*A 1–3% wobble remains on the moon/mesospheric side*, and it is NOT seed
noise (nothing was retrained) and NOT float noise: moon MAD 0.01163 $\to$
0.01192, zodi MAD 0.00022 $\to$ 0.00048, mesospheric ML 45.0 $\to$ 45.56
(gain −8.3% $\to$ −9.7%), reconstruction $\chi^2$ 3.889 $\to$ 3.891, blue
1.202 $\to$ 1.238. Excluded: prediction is deterministic on a given device
(bit-identical on repeat) and MPS-vs-CPU differs by only $5\times10^{-8}$
relative, three orders of magnitude too small; and the known
save-drops-the-rules bug is excluded — `moon_down_amp_rule` (R = 0.024042)
and `zodi_ceiling_rule` (S = 0.645034) are both present and populated in the
saved ensemble, as are all six `jensen_corrections`. **What remains is the
in-memory-vs-restored boundary** (2026-09-11 trained and used the in-session
artifacts; this run used the deserialised ones). Worth one `FORCE_RETRAIN =
True` run on cont2 to close it — note the restored path also cannot report
per-seed metrics, so `mean_eRMSE` and the seed std are unavailable here.

*Useful by-product — a repeat-run yardstick* for deciding which config
differences are real: continuum / FeO / OH MADs reproduce to $\le$0.2%,
reconstruction $\chi^2$ to 0.05%, moon MAD to 2.5%, blue $\chi^2$ to 3%, the
mesospheric gain to 1.4 pp, and zodi MAD not at all (0.00022 vs 0.00048 — it
is anchor-pinned near zero on most rows and its MAD is fragile).

**Best config so far: this one (`cont2`, $W = 0.15$).**

| | cont (no cap) | **cont2 ($W$=0.15)** | cont3 ($W$=0.30) | repeat noise |
|---|---|---|---|---|
| FeO/OH moon Q4/Q1 (the leak) | 3.44x | **1.33x** | 2.16x | none (decomp-side) |
| FeO amplitude MAD | 0.02355 | **0.01790** | 0.01948 | 0.1% |
| continuum amplitude MAD | 0.01868 | **0.01435** | 0.01438 | 0.1% |
| zodi amplitude MAD | 0.00163 | 0.00035 | 0.00013 | fragile |
| OH total amplitude MAD | 0.00659 | 0.00648 | **0.00636** | 0% |
| recon / self-fit $\chi^2$ | 1.09x | 1.09x | **1.08x** | 0.05% |

cont2 wins the two that are both decisive and reproducible — the leak by a
factor 1.6 over cont3 and 2.6 over no cap, and FeO MAD by 8.8% over cont3
against 0.1% repeat noise. It ties cont3 on continuum MAD (0.2% apart, inside
noise) and loses only marginally on OH MAD (1.9%) and the $\chi^2$ ratio
(1%, inside noise). Against no cap it is better everywhere except the
degenerate mesospheric coefficient metric.

---

### `gaia-stars-mask-cont3` (2026-09-12): $W=0.30$, and the OH “regression” was an artefact

> **Retraction.** The $W = 0.15 \to 0.30$ change was made to buy back an OH
> regression that **does not exist in flux space**. It bought back 1.6 pp of a
> metric artefact and gave away half the leak suppression the constraint exists
> for. **Reverted: $W = 0.15$ is the default again as of 2026-09-12.**
>
> `gaia-stars-mask-cont2` is the corpus those settings produce and is
> directly reusable — verified that cont2 and cont3 are **100.00%**
> bit-identical on every ungated row in all three arms (852/852, 851/851,
> 844/844) and 0% identical on gated rows, and that `sky_decomp/` was
> untouched between the two commits. $W$ was the only change.

The mesospheric **coefficient** sRMSE scores all 357 OH sticks equally, but the
OH block is internally degenerate — neighbouring sticks trade amplitude with
almost no change to the convolved spectrum. Projecting exactly the same
coefficient errors through the convolved stick basis,
$(\mathbf{c}_{\rm pred}-\mathbf{c}_{\rm true})_{\rm OH}\,M$:

| corpus | mesospheric **coef** gain | OH **flux** gain | OH flux resid (ML) |
|---|---|---|---|
| cont (no cap) | **+1.2%** | +5.6% | 0.18585 |
| cont2 ($W=0.15$) | **−3.8%** | +5.1% | 0.16195 |
| cont3 ($W=0.30$) | **−3.6%** | +5.2% | 0.16174 |

> **The flux column above is on a LOOSER corpus than the notebook's.** It was
> computed offline with the cheap gates only (reversal and colour skipped), so
> its test rows include the harder ones the deployed corpus drops, which
> inflates the naive baseline's error and hence the ML's gain. Run in-cell on
> the deployed cont2 corpus the same comparison gives **ML 0.16184, naive
> 0.1628, gain +0.6%** against a coefficient-space −9.7%. The **sign flip is
> the robust result** — it reproduces on both corpora and all three configs —
> but the ML's OH advantage in flux is marginal, not ~5%. Use the in-cell
> `naive_baseline` companion, not the offline table, for absolute levels.

The coefficient gain swings 4.8 pp and twice reads *LOSES*; the flux gain is
flat at +5.5 / +4.9 / +5.0% and always positive. On cont3 the ML's OH error is
3.6% **larger** in coefficient norm and 5.0% **smaller** in the spectrum. Since
mesospheric is 358 of 388 coefficients, GROUP-EQUAL inherits the artefact
(+3.0% $\to$ −5.1% $\to$ −3.9%). Every flux-space and amplitude metric was
flat or improving throughout: reconstruction $\chi^2$ 3.984 / 3.889 / 3.865,
ratio to the self-fit floor 1.09 / 1.09 / **1.08**, blue ratio 1.27 / 1.26 /
**1.20**, OH total amplitude MAD 0.00659 / 0.00648 / **0.00636**.

*What $W=0.30$ actually did.* The bound landed exactly where specified — gated
$p_{99}$ at $-0.3448$ against the designed $-0.3489$ (cont2: $-0.4947$ against
$-0.4989$) — and dark rows stayed 99.7% bit-identical to uncapped. But the
leak came back: FeO/OH by moon-amplitude quartile Q4/Q1 **1.33x $\to$ 2.16x**
against 3.44x uncapped, i.e. the excess removed fell from ~87% to ~52%, and
$\rho(\log A_{\rm FeO}/A_{\rm OH}, A_{\rm moon})$ went $+0.210 \to +0.396$
against $+0.484$. On gated rows the block moved −14.0% instead of −32.3%,
FeO −27.9% instead of −47.7%.

*What is genuinely better at 0.30:* seed-to-seed `mean_eRMSE` std 0.614 $\to$
0.354 (against 0.191 uncapped), and the blue $\chi^2$ ratio 1.26 $\to$ 1.20.
*What is worse:* FeO amplitude MAD 0.01791 $\to$ 0.01948, and the leak above.
Since the constraint exists for the leak and the ML cost it was loosened for
is not real, $W=0.15$ is the better point. The seed-stability difference is the
one argument the other way — though `mean_eRMSE` is itself the absolute
coefficient-space metric and inherits the same degeneracy.

*Diagnostic fixed so this cannot recur:* `naive_baseline` now prints a
**flux-space mesospheric companion** beside the coefficient row and says so
loudly when the two disagree in sign; `naive_baseline_result['mesospheric_flux']`
carries it.

---

### `gaia-stars-mask-cont2` (2026-09-11): the moon-gated diffuse/OH cap on the full corpus

Constraint (e) of §1.4.4 applied corpus-wide. Clean A/B against
`gaia-stars-mask-cont`: identical row gates, only the cap differs, filtered
corpora 9 943 vs 9 951 rows, so the row-set confound of
[eRMSE is not row-set invariant] is negligible here. Full tables and the
mechanism are in §1.4.4; the summary is that it **buys the continuum and costs
OH**.

| | cont | cont2 |
|---|---|---|
| continuum amplitude MAD | 0.01868 | **0.01434** (−23%) |
| FeO / HO2 MAD | 0.02355 / 0.02733 | 0.01791 / 0.02150 |
| zodi MAD | 0.00163 | **0.00022** |
| moon MAD | 0.01205 | 0.01163 |
| OH total MAD | 0.00659 | 0.00648 |
| recon / self-fit $\chi^2$ | 3.984 / 3.657 = 1.09x | 3.889 / 3.572 = 1.09x |
| FeO/OH moon Q4/Q1 | 3.43x | **1.33x** |
| mesospheric gain | +1.3% | **−8.3%** (loses) |
| GROUP-EQUAL gain | +3.0% | **−5.1%** (loses) |
| `mean_eRMSE` / seed std | 7.956 / 0.191 | 8.516 / **0.614** |

Decomposition `reduced_chi2` median ratio 1.0000. The cap is **correct as
specified** — measured from the stored `COMP_*` planes it binds at
$\log_{10} = -0.4924$ against the designed $-0.4989$ — and $c$ is confirmed
as the dark-time median to 0.013 dex. See the basis warning in §1.4.4: a
cross-family ratio built from `matrix_oh` is 5.01x wrong and briefly produced
a false report of a units bug here.

**$W$ raised 0.15 $\to$ 0.30, now the default.** $W = 0.15$ was half the
measured one-sided dark-time scatter of 0.305 dex, so the bound cut into real
dark-time variability instead of only clipping excursions past it — a risk
`decompose_parallel.py` had flagged in advance (0.15 is ~0.7x the 0.216 dex
robust sigma) and which this A/B confirmed. At 0.30 the bound admits the full
dark-time $1\sigma$, should roughly halve the 64–69% of gated rows that bind,
and still clips the bright-moon tail with room to spare (gated $p_{99}$ sits
~1.6 dex above the dark-time median). $c$ is unchanged and confirmed.

Note which diagnostics *failed* to catch the 0.15 problem: the watch-items
named for it were all clean — blue $\chi^2$ on gated-but-not-binding rows
$-0.03\%$, dark rows $-0.00\%$, collapse rate 2.28% $\to$ 2.35%, decomposition
`reduced_chi2` ratio 1.0000. The damage showed up only in ML transferability,
which no decomposition-side diagnostic sees. Watch the mesospheric and
GROUP-EQUAL naive-baseline gains on the next arm.

---

### Retrained 2026-09-11: amplitude-weighted lift + looser row gates

Two code changes measured offline and not yet validated by a retrain. Both are
in the same arm, so a single A/B against the deployed numbers below decides them
together; if it loses, the revert points are named at the end of each item.

**OUTCOME (run of 2026-09-11 14:06).** Both changes are live and behaved as
designed. The two are confounded in one arm, so the gate question is not settled;
see the caveat at the end.

*The lift fix worked, cleanly.* The calibration table now reports `flux amplitude`
as the fitted functional, and the mesospheric lift came out **1.0072** (scaling OH
up) where it had been **0.98566** (scaling it down). The OH integrated-amplitude
bias went from **-0.00793 dex to -0.00003**, and its MAD improved as well,
0.00806 -> **0.00659** (-18%). The residual OH bias is now confined to the blue
(3600-5500 A median -0.00705, MAD 0.02368, against +0.00031 redward of 7500 A),
i.e. a shape effect rather than a scale error -- consistent with the finding that
a single per-row rescale removes only 22% of the OH coefficient residual.

*The chi2 overlay gives the headline its missing denominator.* The batch-RMSE cell
now draws the DECOMPOSITION'S OWN self-fit chi2 behind the reconstruction chi2, on
the same rows, pixels and sigma -- the only difference being fitted rather than
predicted coefficients. Measured: reconstruction **3.984** against a self-fit floor
of **3.657**, a ratio of **1.09x** full band; blue **1.255** against **0.9899**,
ratio **1.27x**. The sky-to-science transfer costs 9% over the decomposition's own
residual, so on these rows the ML is no longer the error source -- the
decomposition is. `headline_summary` reads these instead of the hard-coded
3.96 / 1.20 floors, which were corpus-specific and would have silently
misreported any other corpus.

*Added 2026-09-11:* the chi2 figure gained a second row pairing the two numbers
**per row** rather than as two distributions — decomposition self-fit on $x$,
prediction on $y$, log-log, with 1x / 2x / 10x reference lines and the row
identity on hover. That separates three things a marginal histogram cannot: a
point on the diagonal is at the decomposition's own floor and is as good as it
can be *however large both numbers are*; a point above the diagonal is a
transfer failure and the only kind this predictor can fix; a point far to the
right is a decomposition failure, where the target itself is untrustworthy and
a large $y$ is not the predictor's fault. Rows past 2x are drawn in red and
counted. Log-log deliberately, unlike the fenced linear histograms above: those
hide the tail, and this panel exists for it.

*Yield, on the comparable cheap-gate basis:* 10 102 -> **10 545, +443 rows
(+4.39%)**. The full-chain count for this run is 9 951, which is NOT the 10 102
baseline's gate set (see the deployed-configuration note below).

*`mean_eRMSE` 7.642 -> 7.956, and that is mostly a harder exam, not a worse
model.* `mean_eRMSE` is an ABSOLUTE per-coefficient RMSE, so admitting brighter
rows raises it mechanically. Reproduced on the cheap-gate corpus with this
ensemble: the newly admitted test rows score **12.25** against **9.27** for the
rows that also pass the old gates (aggregate 9.55), and their OH is **1.58x**
brighter. Per group the newly admitted rows are worse by 8.9x (moon), 6.5x
(continuum), 2.1x (ionospheric), 1.3x (mesospheric). At 4.1% of the test set that
moves the aggregate by about +3%, against the +4.1% observed -- the right
mechanism and the right order of magnitude.

> **The gate change is NOT settled, and the two changes are confounded.** The
> previous `mlp_ensemble_split_zodi_cont.pt` was overwritten by this retrain, so
> the model-to-model A/B on a common row set is no longer recoverable. To settle
> it, retrain at `kappa=6.0` / `oh_kappa=4.0` with the lift fix in place and
> compare the two on the rows both corpora share. Until then: the lift fix is a
> measured sign error with a mechanism and a held-out validation, and should be
> kept regardless; the gate change is a judgment about which data belong in the
> corpus, and the aggregate metric cannot answer it because the metric is not
> invariant to the row set.

*One regression to watch:* zodi integrated-amplitude MAD 0.00050 -> **0.00163**
(3.3x). Both are tiny and the zodi is anchor-dominated on most rows
(§1.4.3), but it moved in the wrong direction and neither change predicts it.

**1. The empirical mean-bias lift now corrects the flux-weighted amplitude.**
`trainer.py` fitted the scalar Jensen lift as unweighted `mean(true_coef) /
mean(pred_coef)`. What the delivered sky flux depends on is the linear
functional `A_g = c_g . w_g`, with `w_g` the per-coefficient template integral,
and for the OH block the two disagree in **sign**: across the 357 sticks `w`
spans 1.4e9x and rho(w, mean true coef) = **-0.488**, so the unweighted mean is
dominated by exactly the sticks carrying the least flux. The deployed lift is
**x0.98566** (scaling OH down) where the flux needs **x1.0209** (up), which took
the OH band-integrated bias from -0.38% before calibration to **-1.81% after**.
The calibration step was making the bias about 5x worse.

The functional is the whole story, and the groups themselves show it: the two
that already used per-coefficient lifts came out unbiased in amplitude, both
scalar-lift groups did not, ordered by how far their template integrals spread.

| group | lift kind | amplitude bias (dex) | `w` spread |
|---|---|---|---|
| moon | per-coef (15) | -0.00225 | 109x |
| zodi | per-coef x 3 regimes | -0.00000 | 3.4x |
| continuum | scalar (3) | +0.00169 | 11.7x |
| mesospheric | scalar (358) | **-0.00793** | 1.4e9x |

Fixed by threading `calib_amplitude_weights` (native-grid template integrals,
keyed to `coef_names` by name from `design_names` / `design_matrix`) out of
`_precompute_flux_basis_and_geometry` and into the calibration. Native grid
deliberately: the stride-5 flux-loss grid aliases each 2.35 A OH stick by a
different amount, which is precisely the weighting being fitted. Verified lift
change: mesospheric 1.00988 -> 1.02085, continuum +0.71%, ionospheric +0.39%,
atomic +0.00% (its three integrals are equal to four digits -- a null). Expected
gain, held out by half-split: band-integrated |flux error| p50 **-21%**, p90
**-12 to -17%**, at a cost of +2.2% on OH coefficient RMSE. Read the flux number
as the deliverable, not the coefficient one. Revert by ignoring
`calib_amplitude_weights` in the scalar branch of the calibration block.

This also matters more than its size suggests: OH carries **72-78%** of the
band-integrated flux error tail (oracle: substituting the true OH amplitude cuts
p90 by 72% and p99 by 65%, while moon, zodi and continuum each buy under 3%), and
a -1.8% offset is a *bias*, so unlike the scatter it does not average down over
many fibres.

**2. Row gates loosened: `kappa` 6 -> 8, `oh_kappa` 4 -> 6.** Both coefficient-
outlier gates were cutting valid extreme data. Discriminator was the
decomposition's own fit -- absolute single-fibre photon chi2 from `BESTFIT_LSF`
vs `FLUX_SCI` -- plus the median **fractional** residual, since photon chi2 grows
with brightness even at fixed fractional accuracy. Of the every10 rows passing
every other gate:

| population | n | chi2 | ratio | frac resid | ratio | airmass |
|---|---|---|---|---|---|---|
| kept | 1141 | 3.65 | 1.00x | 0.0445 | 1.00x | 1.28 |
| cut: kappa-sigma only | 37 | 4.61 | 1.26x | 0.0452 | **1.02x** | 1.36 |
| cut: OH-MAD only | 34 | 8.64 | 2.37x | 0.0646 | 1.45x | 1.63 |
| cut: the *other* gates | 230 | 96.1 | **26x** | -- | -- | -- |

The other gates find real failures (p90 3610). Neither of these does. The
kappa-sigma rows are fitted indistinguishably well and 59% sit at worst |z| < 8
against a kappa=6 threshold; the offenders are `ATOM_Orc_OI0845` / `OI0777` /
`ATOM_Og` / `ATOM_K`, bright atomic airglow lines. That gate is also
**mis-specified**: it uses a non-robust mean/std, and median std/MAD over its 30
columns is 13.46 against the Gaussian 1.483. The heavy tails are in the moon
knots (std/MAD 29-104, skew about 2 -- bimodal, near 0 moon-down and large
moon-up), which inflates their sigma and *disarms* the gate there, leaving the
near-Gaussian atomic columns (std/MAD 2.14) carrying the tightest effective
threshold. It had become an accidental atomic-line gate -- the same failure
`data.py` already documents for the OH block, still present in moon. Raising
kappa is a mitigation; the fix is a robust per-column scale.

The OH-MAD rows are high-airmass pointings (airmass 1.63 at p86 of kept, alt
37.8 deg at p14, van Rhijn 1.60 at p86) -- where the OH column is physically
largest -- sitting only **1.15x past** a threshold built to catch runaways at
1e6-1e15x the median (p90 1.48x, max 2.27x). Their fits are genuinely 1.45x
worse fractionally, plausibly from LSF and telluric treatment at high airmass:
real data the current decomposition fits less well, not bad data. Extremity is
coherent across all three arms of the same exposure on 84% of cut rows against
62% of kept, and a QP failure would hit one arm, not three.

Recovers about 54 of the 73 cut every10 rows, concentrated in atomic-line
brightness and in OH at high airmass. **Not yet shown to help the model**: the
OH-MAD rows bring 1.45x noisier targets, and the LMC/SMC cut is standing
precedent that a correct explanation can still lose a controlled A/B (it did,
by 15%). Revert to `kappa=6.0` / `oh_kappa=4.0` in the filter cell and in the
`apply_triplet_filters` defaults.

---

### Deployed configuration (2026-09-11, `gaia-stars-mask-cont`)

Corpus re-decomposed with the hybrid diffuse templates (§1.2.4) and the two
diffuse-block constraints (§1.4.4). 14 447 rows → 10 102 after filtering, i.e.
**yield unchanged** — but note that 10 102 counts the CHEAP GATES ONLY: the
reversal and colour gates stream 65 GB and were skipped when comparing, so it is
NOT comparable to a full-chain count. The full-chain count at these settings was
never measured. Reproduced exactly at `kappa=6.0` / `oh_kappa=4.0` on 2026-09-11,
cheap gates only. Quote 10 102 only against another cheap-gate count.
(10 107 before). Decomposition `reduced_chi2` median ratio
new/old 1.0030 (sci), 1.0015, 1.0012 — the new basis and constraints cost
0.1–0.3% of the fit's own full-band $\chi^2$.

10-seed ensemble, night-held-out split (n_test = 920), ensemble saved as
`mlp_ensemble_split_zodi_cont.pt`:

| | value |
|---|---|
| ensemble `mean_eRMSE` | 7.642 |
| seed-to-seed std / ensemble stderr | 0.154 / 0.049 |
| `naive_baseline` gain vs best naive | moon **+39.1%**, zodi **+50.7%**, continuum +7.8%, mesospheric +4.6%, ionospheric +10.4%, atomic +14.3% |
| group-equal aggregate | +6.1% (ML 7.4803 vs `B1_near_geo` 7.9651) |

Amplitude MAD, $\log_{10}({\rm pred}/{\rm true})$: moon **0.01211** (moon-up
0.00850, **moon-down 0.01745**), zodi **0.00050** (moon-up 0.00001,
**moon-down 0.01703**), HO2+FeO+O2Ac **0.01872**, OH total 0.00806.
Per-species diffuse is meaningful for the first time: HO2 0.02502,
FeO 0.02205, O2Ac 0.01713.

**What improved, and what did not.** Against the 2026-09-09 record below —
different corpus *and* different split, so read gains not absolutes — the
continuum families improved substantially: moon MAD −24%, zodi −81%, diffuse
−29%, and **moon-down halved on all three** (moon 0.03648 → 0.01745, zodi
0.03458 → 0.01703). That is the effect predicted when the diffuse prior was
proposed: constraining the species helps the zodi spline in dark time, and the
moon-down moon follows because the $R$ rule couples it to the zodi.

OH went the other way — MAD 0.00648 → 0.00806 (+24%), mesospheric gain
+8.0% → +4.6% — and with it the aggregate (`mean_eRMSE` 7.380 → 7.642, +1.7σ
on the seed std; group-equal +5.6%). The aggregate is dominated by 357 OH
coefficients at scale ~38, and the 0.27 dex the diffuse gave up went to the
moon (+0.096 dex) and to OH (per-row mean 34.2 → 38.8). OH improvements are
expected from the LSF and telluric work, so this is being left alone for now.

> **How much of the continuum win is tautological.** The ratio bracket removed
> ~10 dex of diffuse target variance, so the targets are simply easier.
> `best_val` halved (0.0091–0.0102 → 0.0045–0.0055) — a target-variance
> effect, not skill, so do **not** compare val losses across these corpora.
> And the continuum *gain over naive* barely moved (+8.5% → +7.8%) while its
> MAD fell 29%, i.e. copying the near arm improved by about as much. Read the
> diffuse MAD drop as "the problem got easier for everyone"; the zodi and
> moon-down improvements are the real physical win, since those are not
> directly constrained.

**The ML is now at the decomposition's flux-space floor.** Single-fibre
absolute reduced $\chi^2$ on 500 every10 sci rows: prediction 3.916 full band
and 1.18 blue, against the decomposition's own fit at 3.955 and 1.204 on the
same rows. There is no remaining headroom in flux space against these targets
— the prediction reproduces the observed spectrum as well as the fit it was
trained on, and its p90 is far *better* (7.2 vs 28.1) because it does not
reproduce the decomposition's own bad-row tails. Further ML work on this
corpus cannot improve the subtraction; only the decomposition can.

**Rejected along the way.** (i) Adopting canonical PALACE for all three
diffuse shapes — costs ×1.43 in blue $\chi^2$ and biases the residual to
$-0.26\sigma$, because canonical O2Ac's in-band portion is a tail that runs
2× high. (ii) Centring the species-ratio bracket on PALACE's own reference
shares — 25.7% of the blue $\chi^2$ against 0.67% for the corpus median.
(iii) An F10.7 amplitude prior — unmeasurable in this corpus: `f107_81d`
spans only ×1.48 around the cycle-25 maximum and even OH shows no trend
($\rho=+0.01$), so the test is underpowered, not negative. (iv) Predicting
`zodi + diffuse` instead of the split — the sum's MAD (0.0113) is *worse* than
the anchored zodi alone (0.0065), and the ML's zodi and diffuse errors are
positively correlated (+0.22, +0.34 in dark time), so it is getting totals
wrong rather than mis-splitting. (v) An FeO-only version of constraint (e) —
see §1.4.4.

**Diagnostic note.** `worst_recon` ranks by absolute `sci_pRMSE`, which is
effectively a brightness ranking: the ten worst rows have median brightness
5.5× the corpus median, and the worst (src 1198, 7.4× bright, moon share
0.888) has a fractional moon error of only −16%. Rank by a noise-normalised
error if you want the genuinely badly-fitted rows.

---

### Deployed configuration (2026-09-09, `gaia-stars-mask`)

Corpus: 14 447 rows → 9756 after filtering (§1.11). Decomposed with the four
identifiability constraints, the recentred anchor, and the science
emission-line mask. Context is 42 features per arm (§3.4.5); both
constraint-derived amplitude rules are on (§3.9).

10-seed ensemble, night-held-out split (n_test = 893):

| | value |
|---|---|
| ensemble `mean_eRMSE` | 7.380 |
| seed-to-seed std / ensemble stderr | 0.115 / 0.036 |
| `naive_baseline` gain vs best naive | moon +23.7%, zodi **+40.0%**, continuum +8.5%, mesospheric +8.0%, ionospheric +8.2%, atomic +12.5% |
| group-equal aggregate | +8.4% (ML 7.0856 vs `B1_near_geo` 7.7366) |

Amplitude MAD, $\log_{10}({\rm pred}/{\rm true})$: moon 0.01584 (moon-up
0.00977, moon-down 0.03613), zodi 0.00265 (**moon-up 0.00003**, moon-down
0.03458), HO2+FeO+O2Ac 0.02644, OH total 0.00648.

*Previous deployed record, for the cross-corpus caveat above:* 2026-09-05 on
`new-oh-3`, 17 214 → 10 459 rows, n_test 1137, `mean_eRMSE` 7.976 (std 0.251),
gains moon +15.7% / zodi +41.2% / continuum +22.0% / mesospheric +6.2% /
ionospheric +5.8% / atomic +22.1%, group-equal +7.5%.

---

### Loss: what earns its place

**Per-group flux term** (§3.6.1), 10 seeds, gain over `B0_copy_near`:

| `flux_mse_groups` | moon | zodi | continuum | mesospheric |
|---|---|---|---|---|
| `('moon','zodi')` | +24.0% | **+41.3%** | +19.8% | +6.7% |
| `('moon',)` | +22.6% | +31.6% | +19.1% | +7.4% |
| `('zodi',)` | +3.0% | +27.6% | +6.8% | +4.9% |
| `()` | +5.5% | +23.2% | +12.7% | +2.3% |

The moon needs *its own* flux term — a 20-point cliff that tracks only whether
moon is in the list. The effect is **not separable**: zodi reaches +41.3% only
when both are present, and is better served by the moon's term (+31.6%) than
by its own (+27.6%). Continuum, which had no flux term in this sweep, tracks
the moon arm exactly — the moon head sits on the shared trunk, so scoring it
in flux space shapes the representation every group reads. Dropping the term
entirely triples the seed-to-seed spread (0.28 → 0.81).

**Continuum added to the flux term** (2026-09-05) — the largest single ML gain
found. The diffuse basis existed all along but was never wired into
`_precompute_flux_basis_and_geometry`, so naming `continuum` in
`flux_mse_groups` silently did nothing. Without a flux term the group inherited
the naive baseline's scale error almost exactly:

| | with | without |
|---|---|---|
| gain over `B0_copy_near` | +32.9% | +2.8% |
| signed flux bias | +0.30% | **−10.63%** |

Identical on both seed blocks. Its per-coefficient calibration had looked
healthy throughout (+1.54% lift) because the three diffuse basis functions
have very different flux integrals, so the mean coefficient is not the
flux-weighted bias.

**Log-amplitude term** (§3.6.1). A global λ over moon *and* zodi improves the
moon (relative integrated-amplitude error −20%) and monotonically damages the
zodi: gain +38.3% → +35.7% / +29.0% / +21.8% at λ = 1 / 5 / 20, with the tail
+6.7% / +17.7% / +25.3%. Cause: the zodi amplitude is pinned by the absolute
anchor on most moon-up rows, so a further penalty trades colour for amplitude
it already has. Hence per-group λ. Moon-only λ = 5 is near-optimal (λ 5 / 20 /
50 give moon +24.2% / +18.2% / +9.7%), and on `new-oh-3` its only replicated
effect is its designed one: removing it makes moon relative amplitude 13–26%
worse, while eRMSE, moon gain and mesospheric all flip sign between blocks.
λ_zodi = 1 was tested and rejected — its only consistent effect is a zodi loss
of 1.4–2.5 pp.

**Group weights.** Raising $m_{\rm mesospheric}$ to 1.75 degraded all six
groups, mesospheric included, and doubled the seed std: that group is
noise-limited, not capacity-starved. Lowering $m_{\rm moon}$ to 2.0 was
neutral on moon err/Δ but doubled moon's NIR self-bias — because moon trains
in flux space, its weight sets the flux-space bias directly.

**Row weights.** Every per-row reweighting scheme tried was rejected on
measurement: a high-airmass boost and a moon-down-ecliptic boost never helped,
and a bright-moon-close boost (FLI ≥ 0.90, separation ≤ 30°, moon up, ×1.5)
leaked into the blue band of the residual atlas (+23% RMS|frac|) for a mid-band
gain the flux-space loss now supplies directly.

---

### Architecture: what was tried and rejected

| change | verdict | evidence |
|---|---|---|
| drop the moon-zodi coupling | **rejected** | moon +24.0% → +19.5%, zodi flat. The coupling is the moon-context fix it claims to be — its projectors load 3.7× more into moon than zodi, and the damage is moon-only. |
| single branch for moon+zodi, retiring both the shared-trunk moon head and the isolated zodi branch | **rejected** | bigger moon improvement (~7% moon-up sky-arm) but reintroduced moon→zodi contamination and regressed atlas mid-band RMS by 35%. |
| ctx-α extended to all six groups | **rejected** | ionospheric sRMSE +23%. The thin-shell groups are isotropic on the LVM scale. |
| `zodi_head_extra_dims` (32,) → () | **adopted** | every group within ±1 pp; 7 461 parameters bought nothing measurable. |
| `moon_group_weight` 3.0 → 1.5 | neutral | moon −0.3 pp. |

**The moon error tail: five rejected interventions, and why they were all the
same experiment.** The tail is a per-row multiplicative brightness error —
83–89% of its MSE is removed by a single rescale, and the least-squares and
integrated-flux definitions of that amplitude agree to ρ = 0.991–1.000, with
tail shape error only 2–5%. Attempts to correct it:

| intervention | result |
|---|---|
| arm context for the blend α | moon flux tail +3.9 to +5.1% worse; more capacity, monotonically worse |
| moon geometry in α, sci-only | marginal, inside noise |
| physical-model transfer ratio moon(sci)/moon(near) | MAD 0.019 dex vs **0.010** for assuming the ratio is 1 |
| multiplicative gain on the blend | moon relative amplitude 2.0–4.3% **worse** across three variants |

All five are the same mapping — context → amplitude correction — in different
parametrisations. Regressing the *signed* amplitude error
$\log_{10}(\mathrm{true}/\mathrm{pred})$ on the 37 context features, fitting
on 8 276 training rows and scoring on held-out test, gives **R² = −0.007**
(RandomForest) and −0.009 (Ridge) for the moon. There is no context signal to
correct it with, which is why every parametrisation failed identically.

Two claims made along the way were wrong and are recorded so they are not
re-derived. The architecture was never the constraint: `out[g] = blend + head`
ends in an unbounded `Linear`, so any amplitude was always reachable. And α was
never short of range — its convex leverage $|n-f|/\max(n,f)$ is 11.4% (moon)
and 44.6% (zodi) on the tail against amplitude errors of 7.4% and 34.6%, i.e.
1.3–1.5× the authority needed.

The error's *magnitude* is predictable even though its sign is not (RF R² ≈
0.45 on moon residual RMS; ensemble spread correlates ρ = +0.92 with error), so
these rows can be flagged but not corrected.

**Blend α convergence.** Measured end-of-training α from otherwise identical
runs differing only in init, before `alpha_lr_mult` was raised:

| group | init 0.5 | init 0.7 | init 0.85 |
|:---|---:|---:|---:|
| `mesospheric` | 0.685 | 0.774 | 0.789 |
| `ionospheric` | 0.634 | 0.800 | 0.947 |
| `atomic`      | 0.735 | 0.838 | 0.927 |
| `moon` (ctx-α)      | 0.533 | 0.729 | 0.859 |
| `zodi` (ctx-α)      | 0.522 | 0.715 | 0.857 |
| `continuum` (ctx-α) | 0.553 | 0.734 | 0.862 |

Every group lands within ~0.03 of wherever it starts. `alpha_lr_mult = 30`
(§3.7) is what makes α self-tuning; it produced most of the gains in that
revision.

---

### Decomposition: the identifiability constraints

Measured on 200 lunation-stratified sky spectra, against the unconstrained
split:

| | before | after |
|---|---|---|
| moon/zodi colours in the WRONG order (moon-up, n=168) | 164 | **2** |
| fitted moon log-log slope (physics: −3.7) | +0.21 | **−4.10** |
| fitted zodi log-log slope (physics: −0.3) | −4.04 | **−1.64** |
| moon share with the moon >5° BELOW the horizon | 0.45 | **0.02** |
| ρ(fitted zodi, Leinert B500) | 0.09 | **0.90** |
| ρ(fitted zodi, lunar illumination) — should be ~0.1 | 0.94 | **0.12** |
| median rms ratio | 1.000 | 1.010 |

The rms cost is concentrated entirely at bright moon (×1.49 median for
FLI > 0.8, ≈1% of the continuum, blue-weighted); part of it is the old fit
overfitting through the spline hole described in §1.4.3(a), so the comparison
flatters the unconstrained version.

**Not adopted.** A colour envelope carrying the aerosol scattering channel and
a multiple-scattering enhancement measured neutral: 15 moon knots at β = 0.7
can already manufacture the separation-dependent colour, so the envelope was
never the binding constraint. A blue-window relaxation of the moon bound buys
~1.5%, because the moon spline never reaches its blue bound — the *amplitude*
anchor is what binds there. Both remain in `fit.py`, off by default.

**Anchor recentring** (2026-09-05). The anchor brackets $\int\!{\rm zodi}$
around a prediction whose absolute normalisation had never been calibrated,
and it was saturated: 67.7% of all rows and 93.1% of moon-up rows sat exactly
on the ceiling. The gap is multiplicative, not a pedestal (slope +0.08 of
$\log_{10}(Z_{\rm fit}/Z_{\rm pred})$ against $\log_{10} Z_{\rm pred}$), and a
censored-Gaussian MLE on the leakage-free dark-time rows puts it at 1.61×
(moon-up demands 8.3×, which is leakage, not calibration). Recentring by 1.6
with κ_z held at 2 improved every guardrail — reversals per arm 2.19/1.31/4.72%
→ 1.11/1.11/2.70%, corpus 10 249 → 10 459 rows — and unpinned the zodi colour:
median slope −1.746 → −0.874 against a physics target of −0.3, with only 36%
still on the ratio bound.

It did **not** reduce bright-moon pinning (85.5% → 86.5%), because the demand
there is 8.3× and the ceiling only moved to 3.2×. Dark-time rows moved from 0%
to 20% pinned at the *floor*.

**Reading zodi metrics.** Where the anchor binds, the target is a deterministic
function of geometry and the network reproduces it easily; where it does not,
the error is 4.3× larger:

| anchor state | share of corpus | median relative flux error |
|---|---|---|
| ceiling | 53.0% | **2.2%** |
| floor | 7.1% | 7.3% |
| interior | 39.8% | **9.5%** |

So the headline zodi gain is part learned geometry formula, part genuine
prediction, and zodi metrics should be split by anchor state. Note the perverse
incentive: tightening κ_z *raises* apparent zodi skill while *lowering* the
corpus's information content, so the zodi gain must not be used to tune the
bracket. The zodi error tail is 1.41× enriched in interior rows and contains no
floor-pinned rows.


---

### Constraint-derived amplitudes (2026-09-09)

Four notebook runs on the same corpus and the same 893-row split, so each
change is separately attributable. Amplitude MAD,
$\log_{10}({\rm pred}/{\rm true})$:

| | pre-both | moon-down only | + zodi ceiling | + `moon_frac_po` gate | + ratio transfer |
|---|---|---|---|---|---|
| moon all | 0.01742 | 0.01608 | 0.01636 | 0.01587 | 0.01584 |
| moon, moon-up | 0.00984 | 0.00947 | 0.00945 | 0.00977 | 0.00977 |
| moon, moon-down | 0.04554 | **0.03701** | 0.03874 | **0.03648** | 0.03613 |
| moon **std** | — | 0.1612 | 0.1604 | **0.1082** | 0.1113 |
| zodi all | 0.01695 | 0.01589 | **0.00380** | 0.00265 | 0.00265 |
| zodi, moon-up | 0.01025 | 0.01051 | **0.00004** | 0.00003 | 0.00003 |
| zodi, moon-down | 0.03438 | 0.03402 | 0.03523 | 0.03458 | 0.03458 |
| HO2+FeO+O2Ac | 0.02587 | 0.02524 | 0.02656 | 0.02644 | 0.02644 |
| OH total | 0.00655 | 0.00660 | 0.00658 | 0.00648 | 0.00648 |

group-equal sRMSE 7.1154 → 7.1076 → 7.0856 → 7.0856; ML over best naive
+8.0% → +8.1% → +8.4%.

**The zodi ceiling is the largest single ML gain in this record**: moon-up zodi
amplitude MAD 0.01051 → 0.00004, a factor 260 on 535 test rows, lifting the
zodi gain +26.2% → +39.3%. `zodi all` std also improves (0.0552 → 0.0530), so
the tail did not pay for the bulk. Prerequisite was moon/zodi model cache
**v2**, which stores the physics-only prediction alongside the learned one;
v1 had only the learned variant, which is 1.5–1.7× larger and hid the effect
entirely (§1.4.3).

**The gate matters more than the rule.** Moving the moon-down gate from
`moon_alt ≤ 0` to `moon_frac_po ≤ ε/κ_f` cut the rule's own contamination from
**6.1% to 0.6%** of gated rows and the moon amplitude **std from 0.1604 to
0.1082 (−33%)** while the MAD barely moved — the signature of a tail fix.
Variance budget: $0.1604^2\times893 = 22.98$ units against 10.45 after, i.e.
~13 test rows carrying ~1 dex each, consistent with the ~20 near-horizon rows
a 893-row split should hold. The moon_down moon sRMSE penalty against
`B0_copy_near` fell −54.4% → −17.9% at the same time.

**Null result, kept for the lesson.** The near-arm ratio transfer (§3.9.2) was
predicted to turn 240 badly-wrong rows into 39, measured on the full 14 447-row
corpus. On the deployed split it did nothing: the fit's own run-time report
read `the fraction wrong by >0.5% of moon+zodi falls 0.55% -> 0.55%`,
`mean_eRMSE` was bit-identical (inference-only change), and the test moon std
went 0.1082 → 0.1113. The training filters already remove ~93% of the
population it targets. **Measure a fix on the population whose metric you are
trying to move.** It was kept because it is strictly the better estimator and
does help unfiltered rows, which is what production sees (§3.9.3) — but no
metric win is claimed for it. The lesson that generalises is the second one:
making a rule self-report its own effect at fit time is what caught this
inside the run rather than three runs later.

**Rejected without a run: re-testing `moon_scatter_envelope`.** The hypothesis
was that a shape-limited moon spline forces the leakage the anchor caps. It is
refuted by the per-pair bound occupancy — moon-up saturation is localised at
the band *ends* (pair 0 68.9%, pairs 11–13 62–73%) with the middle interior,
and only 0.1% of rows have the whole blue half pinned — which is consistent
with the 2026-09-03 verdict that the envelope is neutral. The spline is not
globally slope-limited.

**Uncertainty side effect.** Ensemble spread no longer carries zodi *amplitude*
information on snapped rows, since every member snaps to the same ceiling. The
per-coefficient spread survives (the snap rescales the block and leaves the
shape), and on moon-up rows the zodi p50 σ/err ratio is 0.47 — spread now
over-estimates the error, the safe direction, and far better calibrated than
moon (3.6) or continuum (1.9). Do not source a zodi amplitude σ from ensemble
spread on those rows.

### Open: the zodi/diffuse degeneracy

Two independent findings now point at the same place, and it is the next thing
to work on:

- The κ_z release test (§1.4.3): freeing the anchor moves 44% of the diffuse
  block and 22% of the moon into the zodi for a 0.6% rms gain, with the total
  continuum conserved to 0.5%. The three continuum families are substantially
  interchangeable at fixed χ².
- The failed-fit population (§3.9.3): on 1.7% of rows the QP puts 100% of the
  continuum into `Zodi_bs` and zeroes the moon *and* the diffuse block, with
  χ² 15× worse. `degenerate_continuum_flag` marks them; nothing fixes them.

---

### Removed code paths

Each was verified unexercised before removal, and each removal was verified
behaviour-preserving on a 10-seed regression (all metrics identical to machine
precision).

- **Multiplicative blend gain**, **`alpha_arm_ctx`**, **`alpha_hidden_dims`**,
  and the flux-MSE/amplitude-term decoupling (2026-09-05) — all measured, all
  rejected, none exercised by the deployed configuration. Their findings are
  kept in `mlp_predictor.ablations.RETIRED` and `_NOTES`; `apply()` refuses the
  retired configs with the reason.
- The bright-moon-close row-weight boost and its three knobs — carried at 1.0
  (off); row weights are now uniform by construction (§3.6.5).
- 14 config keys the trainer never read. Two were active hazards:
  `moon_zodi_mode` looked like the coupling's off-switch and was not, and
  `alpha_ctx_groups` looked like it restricted context-dependent α but the
  model hardcodes `_ALPHA_CTX_GROUPS`. An A/B driven through either would have
  measured nothing. `ablations.apply()` now refuses any override the trainer
  does not consume.
- Module docstrings that were removal logs rather than descriptions.

### Pipeline robustness

- **Zero-valued LSF pixels** (2026-09-05). A gaia-selected input carried 9
  fibres (of 14 469) with a single `LSF_SCI` pixel of exactly 0.0 at a
  spectrograph arm join — 8 at 5800 Å, 1 at 7570 Å. The physical model rightly
  rejects a non-positive FWHM, which killed the worker chunk.
  `decompose_parallel._sanitised_lsf_row` now interpolates such pixels for the
  amplitude prior, where the caller wants only scalar band integrals (switching
  the repair from linear to nearest-neighbour moves the moon fraction by 7e-12).
  A row with no usable LSF at all still fails explicitly.


In [ ]:
# --- Package imports ---
%load_ext autoreload
%autoreload 2

import os
os.environ.setdefault("LVMCORE_DIR", "/Users/droryn/prog/lvm/lvmcore")

import numpy as np
import pandas as pd
import plotly.io as pio

# Notebook cwd is skysub/, so mlp_predictor/ and sky_decomp/ are top-level packages.
from mlp_predictor import config, data, wavelengths, compressor, trainer, diagnostics
from mlp_predictor.config import DataConfig, PipelineConfig

# Journal-style plotly axes (as in the pre-refactor notebook).
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "inside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

FACTOR = 1e14

# --- Corpus + artifact locations: the one place to switch corpora -----------
# Every input path -- meta, per-arm coefficients, per-arm decomposition, the
# basis grid and the coefficient wavelengths -- is derived from DECOMP_DATA_ROOT and
# DECOMP_STEM by mlp_predictor.config.DataConfig, and the trained ensemble is
# written beside the corpus it was trained on, so a single edit here moves the
# whole pipeline.
#
#   root          stem                                     rows   selection
#   'new-oh-3'    lvmsframe_median_stack_1.2.1_p40_p70    17 260  faint-fibre percentile
#   'gaia-stars'  lvmsframe_median_stack_1.2.1_gaia1over100 14 469  GAIA stellar SB < 1% of sky
#
# BOTH must be set when switching: the stem differs between corpora, and a root
# change alone silently looks for the wrong filenames under the new root.
#
# The coefficient wavelengths are no longer cached: they are read off the
# decomposer design matrix in ~1 s per run, from THIS root's own basis, so they
# cannot go stale.  Compressors and the ensemble are still never transferable
# between corpora -- the moon/zodi split and its sum move whenever the
# decomposition changes.
# --- DECOMPOSITION VARIANT -------------------------------------------------
# Two decomposition flavours of the same input stack, switchable here:
#
#   'split_zodi'  the production LSF-surface-iterative split-zodi fit
#                 (gaia-stars-mask-cont2, W = 0.15 diffuse/OH cap)
#   'telluric'    palace-aijc-vnf-split-zodi-lsf-spline2d: PALACE VNF lines
#                 with OH grouped by (v_upper, N_upper, F_upper), a continuous
#                 M-spline 2-D LSF, and TELLURIC ABSORPTION included in the fit
#
# The coefficient vector is the SAME 388 names in the same order, so the ML side
# is drop-in and every coefficient-space number is directly comparable between
# the two. What differs is the RECONSTRUCTION path, in two ways that both bite:
#
#   * the LSF is stored as a continuous M-spline density rather than a discrete
#     11-tap surface, so the iterative class cannot read it at all; and
#   * every family matrix is divided by that ROW's DRP transmission, so the
#     design matrix is per-row and cannot be hoisted (~0.17 s/row/arm to
#     rebuild, about 4 min for a 500-row sample across three arms).
#
# `mlp_predictor.data.DECOMP_VARIANTS` holds the suffixes;
# `make_telluric_row_lookup` supplies the per-row transmission, and the
# diagnostics pick it up from the `TELLURIC_ROW_FOR` extra installed below.
#
# Integrated amplitudes A_g = c_g . v_g keep the TELLURIC-FREE template
# integrals in both flavours: the coefficient multiplies the same physical
# template either way and the transmission is a per-row correction applied to
# it, so the plain integrals are both the physical amplitude and the only
# choice that leaves the two corpora comparable.
# 2026-09-18: the non-telluric variant was removed from this branch; the
# telluric fit is the only supported decomposition (see data.DECOMP_VARIANTS).
# 2026-09-24: 'telluric-palacecorr' is the same telluric fit with the SkyFar
# ridge-corrected OH line strengths (decompose_parallel's default from that
# date).  Same 388 coefficients; the variant entry selects the matching OH line
# file for every reconstruction, which is what makes its flux numbers valid.
DECOMP_VARIANT = 'telluric'

_VARIANT_ROOT = {
    'telluric':            'gaia-stars-mask-telluric-chi2-1.3.2',
    'telluric-palacecorr': 'gaia-stars-mask-telluric-palacecorr-1.3.2',
}
if DECOMP_VARIANT not in _VARIANT_ROOT:
    raise ValueError(f"DECOMP_VARIANT must be one of {sorted(_VARIANT_ROOT)}; "
                     f"got {DECOMP_VARIANT!r}")
DECOMP_DATA_ROOT = _VARIANT_ROOT[DECOMP_VARIANT]
DECOMP_STEM = 'lvmsframe_median_stack_1.3.2_gaia1over100'
# Same filename under both roots -- the root already separates them, so the two
# variants cannot overwrite each other's ensemble.
ENSEMBLE_FILENAME = 'mlp_ensemble_split_zodi_cont-1.3.2.pt'   # 'new-oh' legacy name: mlp_ensemble_split_zodi_new_oh.pt

# --- Physical-model moon transfer ratio as a context feature ---------------
# Adds ONE ctx column, log10(moon_model_sci / moon_model_near), gated to 0 on
# moon-down rows, from the cache built by mlp_predictor.moon_model_cache.
# A/B on 2 x 10 seeds, identical rows/split/seeds: moon amplitude MAD on
# MOON-UP test rows 0.01492 -> 0.00771 (-48.4%), p90 -23.6%, zodi MAD -16.8%,
# continuum +5.0%, mean_eRMSE 7.858 -> 7.774.  The moon-up MAD of 0.0077 beats
# the physical model's own 0.0119, i.e. the net does better than either the
# model or its old inputs alone.
# The cache is per corpus and BUILDS ITSELF on first use -- expect a one-off
# ~26 min on 8 cores the first time a new corpus is
# trained on, and ~3 min again when the first every10 diagnostic cell runs
# (the corpus and its every10 subsample need separate caches, since row N of
# one is not row N of the other).  Build them ahead of time with
#   python -m mlp_predictor.moon_model_cache <prefix> --n-workers 8
# A cache that exists but does NOT match the corpus is never silently rebuilt;
# it raises and names the offending row, because a mismatch is as likely to be
# the wrong prefix as a stale file.
# Set False to A/B.  NOTE this changes ctx_names, so it invalidates any saved
# ensemble -- cell 10's guard will refuse to load one.
USE_MOON_MODEL_FEATURE = True

cfg = PipelineConfig()
cfg.data.decomp_data_root = DECOMP_DATA_ROOT
cfg.data.decomp_stem = DECOMP_STEM
# The suffix is NOT a DataConfig default any more -- it identifies the variant.
cfg.data.decomp_suffix = data.DECOMP_VARIANTS[DECOMP_VARIANT]['suffix']
USE_TELLURIC_RECON = bool(data.DECOMP_VARIANTS[DECOMP_VARIANT]['telluric'])
SAVED_ENSEMBLE_PATH = f"{DECOMP_DATA_ROOT}/{ENSEMBLE_FILENAME}"

print(f"Corpus:   {cfg.data.decomp_data_root}/{cfg.data.decomp_stem} "
      f"(suffix={cfg.data.decomp_suffix!r})")
print(f"Ensemble: {SAVED_ENSEMBLE_PATH}")

In [ ]:
# --- Load triplet + apply filters + augment context ---
triplet = data.build_triplet_coef_dataset(
    input_fits_path=cfg.data.input_fits_meta,
    sky_near_decomp_fits_path=cfg.data.coef_fits("sky1"),
    sky_far_decomp_fits_path=cfg.data.coef_fits("sky2"),
    sci_decomp_fits_path=cfg.data.coef_fits("sci"),
    context_columns=list(cfg.data.context_columns),
    return_chi2=True,
)
print("Loaded triplet shapes:",
      {k: v.shape for k, v in triplet.items()
       if hasattr(v, "shape") and k in ("coef_near","coef_far","coef_sci","ctx_sci")})

# Native wavelength grid, for the moon/zodi role-reversal check below.
from astropy.io import fits as _fits_w
with _fits_w.open(cfg.data.input_fits_for_basis) as _hw:
    _wave_native = np.asarray(_hw["WAVE"].data, dtype=float)
    _wave_native = _wave_native if _wave_native.ndim == 1 else _wave_native[0]

filtered_triplet = data.apply_triplet_filters(
    triplet,
    thin_every_n=1, chi2_qmax=90.0, chi2_min=0.0, chi2_max=10.0,
    hard_coef_bounds={"feo": (0.0, 36.01), "atom_k": (0.0, 10.01)},
    # 2026-09-11: kappa 6 -> 8, oh_kappa 4 -> 6.  Both gates were cutting valid
    # extreme data, not broken decompositions.  Discriminator was the
    # DECOMPOSITION-S OWN fit (absolute single-fibre photon chi2 from
    # BESTFIT_LSF vs FLUX_SCI) plus the median FRACTIONAL residual -- chi2
    # alone is confounded, since at fixed fractional accuracy photon chi2
    # grows with brightness.  On every10, of the rows passing every OTHER
    # gate: the other gates find real failures (chi2 96.1 vs 3.65 kept, 26x,
    # p90 3610) while these two do not.
    #
    #   cut by kappa-sigma (n=37): chi2 1.26x kept, fractional residual
    #     1.02x -- indistinguishable.  59% sit at worst |z| < 8 against a
    #     kappa=6 threshold, and the offenders are ATOM_Orc_OI0845 / OI0777 /
    #     ATOM_Og / ATOM_K, i.e. bright atomic airglow lines.  The gate is
    #     MIS-SPECIFIED: it uses a non-robust mean/std and median std/MAD over
    #     its 30 columns is 13.46 against the Gaussian 1.483.  The heavy tails
    #     are in the MOON knots (std/MAD 29-104, skew ~2 -- bimodal, ~0
    #     moon-down and large moon-up), which inflates their sigma and
    #     DISARMS the gate there; the atomic columns are the most Gaussian
    #     (std/MAD 2.14) and so carry the tightest effective threshold.  It
    #     had become an accidental atomic-line gate.  Same failure the code
    #     already documents for the OH block, still present in moon.
    #
    #   cut by the OH-MAD gate (n=34): airmass 1.63 (p86 of kept), alt 37.8
    #     deg (p14), van Rhijn 1.60 (p86) -- exactly where the OH column is
    #     physically largest.  The gate scalar sits 3.56x the kept median
    #     against a threshold at 3.10x, i.e. 1.15x PAST a threshold built to
    #     catch runaways at 1e6-1e15x.  Their fits are 1.45x worse
    #     fractionally, plausibly because LSF and telluric treatment degrade
    #     at high airmass: real data the CURRENT decomposition fits less
    #     well, not bad data.
    #
    # Extremity is coherent across all three arms of the same exposure on 84%
    # of cut rows vs 62% of kept -- a QP failure would hit one arm, not three.
    #
    # Recovers ~54 of the 73 cut every10 rows, concentrated in atomic-line
    # brightness and in OH at high airmass -- and OH carries 72-78% of the
    # band-integrated flux error tail.  NOT yet shown to improve the model:
    # the OH-MAD rows also bring 1.45x noisier targets, and the LMC/SMC cut
    # is precedent that a real explanation can still lose a controlled A/B.
    # Revert to kappa=6.0 / oh_kappa=4.0 if this arm loses.
    kappa=8.0, kappa_iter=3, oh_kappa=6.0, oh_kappa_iter=3,
    exclude_field_regions=[data.LMC_EXCLUSION, data.SMC_EXCLUSION],
    # Drop rows where the moon and zodi splines swapped roles in ANY arm: the
    # moon coefficients of such a row describe zodiacal light, so they teach
    # the network the wrong geometry mapping rather than merely adding noise.
    # Tested only where both families carry flux (moon share in [0.05, 0.95]),
    # so dark-time rows -- pinned near a 0.02 moon share by the decomposition
    # priors -- are exempt by construction.
    reversal_decomp_fits={
        "near": cfg.data.decomp_fits("sky1"),
        "far":  cfg.data.decomp_fits("sky2"),
        "sci":  cfg.data.decomp_fits("sci"),
    },
    reversal_wave=_wave_native,
    reversal_min_component_frac=0.05,
    reversal_min_separation=0.0,
    # Drop rows whose SCIENCE fibre carries continuum the sky basis cannot
    # represent.  The Moon_bs spline is the only flexible continuum in the
    # basis -- the zodi total is pinned by the Leinert anchor and the three
    # diffuse species have fixed shapes -- so field continuum lands in the
    # moon coefficients and that row's moon target stops describing scattered
    # moonlight.  Measured on this corpus: rho = +0.405 between the colour
    # excess and the sci-minus-near moon spline colour distortion, against
    # +0.034 for the far-minus-near sky-arm control, and rho = -0.331 against
    # the signed ML moon amplitude error on moon-up rows.  Costs 7.1% of rows.
    # NOTE this needs the full stack, not input_fits_for_basis (every10) or
    # input_fits_meta (no flux at all).
    colour_excess_input_fits=cfg.data.input_fits_flux,
    colour_excess_max=data.SCI_COLOUR_EXCESS_MAX,
    # Drop rows where the QP collapsed the whole diffuse block (HO2+FeO+O2Ac)
    # to ~0 in any arm.  188 sci rows on this corpus (1.30%), genuinely zero
    # rather than faint, and overwhelmingly dark time (10% moon-up against 50%
    # corpus-wide) -- the zodi/diffuse degeneracy taken to its limit, with the
    # Zodi_bs spline carrying the whole continuum.  A target of ~2e-6 against a
    # near-arm 322 measures nothing; those rows alone put the continuum p95
    # log-error at 8 dex.  Existing gates already remove ~66% of them (their
    # median sci chi2 is 1.98 against 0.185), so the marginal cost is small.
    diffuse_zeroed_frac=data.DIFFUSE_ZEROED_FRAC,
)

data._augment_triplet_with_ecliptic(filtered_triplet, force=True, meta_fits_path=cfg.data.input_fits_meta)
data._augment_triplet_with_physics_priors(filtered_triplet, force=True)
# MOON-MODEL-CTX-V1: the frozen physical model's moon transfer ratio.  Last of
# the augments so the feature is always the final ctx column, which keeps the
# every10 diagnostic cells' layout identical to the training one.
if USE_MOON_MODEL_FEATURE:
    data._augment_triplet_with_moon_model(
        filtered_triplet, cfg.data.decomp_prefix, force=True)
print(f"Augmented ctx: n_ctx={len(filtered_triplet['ctx_names'])}")


In [ ]:
# Preview pre- vs post-filter coefficient histograms (no ML state needed).
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

_tmp = diagnostics.Diagnostics(diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    extras={"triplet": triplet},
)).coef_hist_prepost()


In [ ]:
# --- Per-row joint covariance blocks (COEF_COV_MOON / COEF_COV_ZODI) ---
import numpy as _np
from astropy.io import fits as _fits

def _load_cov_for_row_index(decomp_path, row_index):
    row_index = _np.asarray(row_index, dtype=_np.int64)
    with _fits.open(decomp_path) as _h:
        _moon = (_np.asarray(_h["COEF_COV_MOON"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_MOON" in _h else None)
        _zodi = (_np.asarray(_h["COEF_COV_ZODI"].data[row_index], dtype=_np.float64)
                 if "COEF_COV_ZODI" in _h else None)
    return _moon, _zodi

_row_idx_cov = _np.asarray(filtered_triplet["row_index"], dtype=_np.int64)
for _arm, _path in {
    "near": cfg.data.decomp_fits("sky1"),
    "far":  cfg.data.decomp_fits("sky2"),
    "sci":  cfg.data.decomp_fits("sci"),
}.items():
    _m, _z = _load_cov_for_row_index(_path, _row_idx_cov)
    filtered_triplet[f"coef_cov_moon_{_arm}"] = _m
    filtered_triplet[f"coef_cov_zodi_{_arm}"] = _z
print("Per-row joint covariance loaded (or None where COEF_COV_* HDU absent).")


In [ ]:
# --- Populate coefficient wavelengths + effective extinction ---
# `decomp_suffix` selects WHICH BASIS the wavelengths are built from.  Every
# variant uses the same 388 coefficient NAMES, so the name list alone cannot
# tell two bases apart -- but the telluric fit groups OH by (v_upper, N_upper,
# F_upper) and its per-stick centroids differ from the split-zodi ones by a
# median 46.9 A (p90 288 A, max 859 A, 310 of 357 sticks past 2 A).  Those
# wavelengths set the per-coefficient extinction and van Rhijn geometry, so
# building from the wrong basis is not cosmetic.
#
# There is no longer a wavelength cache on disk: the centroids and k_eff are
# read off the decomposer's design matrix in ~1 s, so the cache saved ~1 s per
# run while keeping alive the failure mode above (an .npz copied between
# corpora passed the name check and carried the wrong OH centroids).  Any
# leftover coef_wavelengths_basis_v4.npz in a corpus directory is now unused
# and can be deleted.
ext = wavelengths.resolve_wavelengths_and_extinction(
    filtered_triplet,
    input_fits_for_basis=cfg.data.input_fits_for_basis,
    use_fitted_extinction=True,
    verbose=True,
    decomp_suffix=cfg.data.decomp_suffix,
)
group_indices = ext.group_indices
N_MOON_KNOTS, SPLIT_ZODI, N_ZODI_KNOTS = wavelengths.infer_spline_knots(
    filtered_triplet["coef_names"])


## Symmetric Dual-Encoder Group-Head Model

Key design choices in this implementation:
1. Shared near/far encoder weights for symmetry and sample efficiency.
2. Fusion vector uses:
   - $e_{mean} = 0.5(e_{near} + e_{far})$
   - $e_{diff} = e_{near} - e_{far}$
   - $|e_{diff}|$
3. Context branch encodes science-context only, now including folded time features and van Rhijn shell factors.
4. Group-specific heads map fused representation into layer-aware coefficient groups rather than a single flat atomic bucket.

**Note.** `DualEncoderGroupHeadMLP` below is the parent architecture. What
the notebook actually trains is `DualEncoderGroupHeadMLPCompressed` (defined
two cells down, together with the compressor fit), which reuses this
architecture verbatim on a per-group compressed coefficient space and emits
signed linear head outputs. See §5.5 in the methods cell above for the
compression pipeline and the empirical argument for switching to it.


In [ ]:
# --- Fit per-group compressors on a moon-phase-stratified split ---
from mlp_predictor.ml_utils import moon_phase_deg_from_ctx, split_indices_by_moon_phase

_moon_phase = moon_phase_deg_from_ctx(filtered_triplet)
_split_tr, _split_va, _split_te = split_indices_by_moon_phase(
    filtered_triplet["obstime_mjd"], _moon_phase, seed=42)

group_compressors, compress_geom_kwargs = compressor.fit_all_group_compressors(
    filtered_triplet, group_indices,
    train_idx=_split_tr, held_idx=_split_va,
    xarm_threshold=compressor.COMPRESSION_XARM_THRESHOLD,
    verbose=True,
)
filtered_triplet["compress_train_idx"] = _split_tr
filtered_triplet["compress_val_idx"] = _split_va
filtered_triplet["compress_test_idx"] = _split_te


In [ ]:
# --- Training config (edit here to override deployed defaults) ---
# Full deployed config expanded in place so every knob is visible; every value
# is initialised to the mlp_predictor.trainer default and can be changed below.

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 2026-08-27c: Phase F -- additive moon-zodi coupling.  Adds a small
# coupling branch (~9k params) that produces a shared latent from the
# moon-zodi ctx union, projected additively into the moon head output
# and the zodi head output via zero-init linear projectors.  Both
# existing paths (moon on shared trunk, zodi on isolated branch) are
# preserved -- this is the "preserve zodi isolation, give moon the
# shared context" fix for Phase E's atlas-mid regression (2026-08-27b).
# At init the projector outputs are zero, so training starts byte-
# identical to Phase A''; the projectors learn to route the coupling.
# To A/B against Phase A'' (no coupling), set
# "moon_zodi_coupling_enabled": False -- or ablations.apply(train_cfg,
# "A1_no_coupling").  The old instruction ("set the four moon_zodi_* entries
# to None") stopped working when the 2026-08-27 trim deleted the no-coupling
# path: it raised on an empty restriction instead of disabling anything.
# Phase E (moon_zodi_mode="shared_branch") is gone from model.py entirely.
# 2026-08-26f: Phase A' landed on top of Phase D.  Adds three explicit
# interaction features to the ctx augment pipeline (computed in
# data._augment_triplet_with_physics_priors):
#   moon_fli_x_phase_cos = moon_fli * moon_phase_cos          # 2nd-order phase
#   moon_sig_x_lon_cos   = moon_signal_proxy * ecl_lon_cos    # moon x zodi geom
#   moon_sig_x_lon_sin   = moon_signal_proxy * ecl_lon_sin    # moon x zodi geom
# All three fed into the isolated continuum branch (see continuum_ctx_restriction
# below).  Attacks the residual_ctx_attribution RF top importances on continuum
# (moon_fli 0.21, moon_phase_cos 0.15, ecl_lon_cos 0.09, moon_signal_proxy 0.05).
# A/B vs 26e (Phase D, same 10 seeds, n_test=1231):
#   mean_eRMSE  21.12 -> 20.98 (-0.7%)  |  mean_eWRMSE 14.98 -> 14.71 (-1.8%)
#   seed std    0.801 -> 0.509 (-36%, most of Phase-D variance recovered)
#   sky-arm continuum p50 err/Delta: all 1.19 -> 1.16, moon_down 1.10 -> 1.08,
#     close_zodi 1.27 -> 1.32 (only regression; still << pre-Phase-D 1.40).
#   wavelength_residual_atlas blue RMS|frac|: 0.535% -> 0.209% (-61%, huge).
#     mid +11% (still 0.63%), NIR +1.5%.  Mean bias grew to -0.4% mid / -0.6% NIR.
#   naive_baseline continuum ML now beats B1_near_geo in every regime (first
#     time): all +2.0%, moon_up +0.2%, moon_down +7.1%, close_zodi +2.4%.
# To A/B against pre-Phase-A' (Phase D only), drop the three trailing entries
# from `continuum_ctx_restriction` below.  The augment always emits them; they
# are simply ignored by the branch when not listed.
# ----------------------------------------------------------------------
# 2026-08-26e: Phase D landed on top of Phase A+B+C.  Widens the isolated
# continuum branch from (64, 32) -> (128, 64) and adds a 64-d hidden layer
# inside the continuum head (+12k targeted params, +2.3% full-model params).
# A/B vs 26d (Phase A+B+C, same 10 seeds, n_test=1231):
#   sky-arm floor, continuum p50 err/Delta:
#     all        1.247 -> 1.186  (-5%)
#     moon_down  1.212 -> 1.095  (-10%, target hit)
#     close_zodi 1.401 -> 1.274  (-9%,  target hit)
#   naive_baseline continuum sRMSE all: 0.01554 -> 0.01461 (-6%, recovers
#     the Phase-B accounting regression; ML now also beats B1_near_geo on
#     continuum moon_down for the first time).
#   wavelength_residual_atlas: blue -12%, NIR -5%, mid +5% (still zero-bias).
#   Aggregate mean_eRMSE 20.99 -> 21.12 (within seed noise); seed std
#     0.41 -> 0.80 (+96%, variance-inflation from the extra 12k params;
#     ensemble stderr still 1.2% of mean).
# To A/B against the pre-Phase-D config (Phase A+B+C only), override:
#     "continuum_branch_dims":     (64, 32)  # current: (128, 64)
#     "continuum_head_extra_dims": ()        # current: (64,)
# To revert further to the pre-Phase-A/B/C baseline, additionally override:
#     "moon_group_weight":         4.0       # current: 2.0
#     "continuum_group_weight":    1.5       # current: 1.0
#     "zodi_head_extra_dims":      ()        # current: (32,)
#     "continuum_ctx_restriction": None      # current: 6-tuple moon-geom
#     "alpha_ctx_features":        None      # current: 3-tuple
#     "alpha_ctx_groups":          None      # current: (moon, zodi, continuum)
# See 2026-08-26d + 2026-08-26e changelog entries for the full A/B tables.
# ----------------------------------------------------------------------
train_cfg = {
    "name": "dual_group_mlp_compressed",

    # Optimisation schedule.
    # 2026-09-18: 50 -> 300 epochs, patience 12 -> 100.  Early stopping does not
    # protect quality here (the loop restores `best_state`), so patience only
    # saved compute -- and at 12 it fired inside the plateau noise, stopping two
    # of four seeds at epoch 17 and 24 with models from epoch 5 and 12.  The
    # stall before the eventual best epoch is 28-131 epochs.  Measured in FLUX
    # space, 300 epochs beats 50 by -8.9% fractional RMS on the reconstructed
    # sky (-13.8% blue, -12.0% integrated), better on 4/4 paired seeds; the
    # coefficient metrics say the opposite because 92% of mean_eRMSE is the
    # mesospheric block.  See trainer.default_dual_group_config for the
    # patience simulation table.
    "n_epochs": 400,
    "batch_size": 512,
    "lr": 1.0e-3,
    "weight_decay": 2.0e-4,
    "patience": 100,

    # Architecture (widths).
    "encoder_dims": (768, 384),
    "ctx_dims": (96,),
    "trunk_dims": (320, 160),
    "head_dim": 192,
    "zodi_head_extra_dims": (),   # A4 (2026-09-04): was (32,).  Ablation showed the 7461-param
                                #  extra layer buys nothing measurable -- every group
                                #  within +/-1pp, aggregate -0.144 (inside the 0.28 seed
                                #  std).  The zodi head sits at err/delta = 0.354, far
                                #  inside its irreducible band, so extra capacity there
                                #  only chases noise.

    # Per-group loss weights m_g (see §3.6.2).
    "moon_group_weight": 3.0,
    "zodi_group_weight": 2.0,
    "continuum_group_weight": 1.0,
    "mesospheric_group_weight": 1.0,
    "ionospheric_group_weight": 1.0,

    # Loss shaping.
    # ADOPTED 2026-09-04: continuum joins moon and zodi.  Until this date the
    # diffuse basis was never wired into _precompute_flux_basis_and_geometry,
    # so naming "continuum" here silently did nothing -- and with no
    # flux-space term the group simply inherited the naive baseline's scale
    # error: signed flux bias -4.21% against B0_copy_near's -4.14%, while its
    # per-coefficient calibration looked fine (the three diffuse basis
    # functions have very different flux integrals, so the mean coefficient
    # is not the flux-weighted bias).
    # Replicated on two 10-seed blocks: continuum gain +8.6pp / +11.4pp,
    # bias -4.21% -> -0.43% and -4.03% -> -0.79%.  Every other group is
    # neutral or better in both blocks and mean_eRMSE improves ~0.04.
    "flux_mse_groups": ("moon", "zodi", "continuum"),
    # Additive log-amplitude term in flux space (§3.6.1).  0.0 = off, the
    # deployed default.  The per-pixel flux term above is ABSOLUTE and so is
    # dominated by the brightest rows; this one is scale-free and targets the
    # measured tail, 83-89% of whose MSE is pure brightness.  lambda must be
    # O(1-10) to matter -- log**2 is ~0.01 for a 0.1 dex miss while the
    # per-pixel term is rescaled to the coefficient loss's magnitude.
    # ADOPTED 2026-09-04: moon only.  Replicated on two independent 10-seed
    # blocks -- moon relative integrated-amplitude error -20.3% / -18.6%,
    # moon core flux RMSE -2.3% / -1.4%, continuum +1.4pp / +1.6pp.
    # Costs mesospheric ~-0.7pp and mean_eRMSE +0.05..+0.09: that aggregate
    # is dominated by the 358 mesospheric coefficients (scale ~34), not by
    # moon and zodi (scale ~0.5), so it moves against the groups this helps.
    # ZODI IS DELIBERATELY 0.  A global lambda drives the zodi gain from
    # +38.3% to +21.8% (lambda 20): the zodi amplitude is already pinned by
    # the decomposition's absolute Leinert anchor -- re-measured 2026-09-09 on
    # gaia-stars-mask with the exact per-row design, 87.5% of moon-up rows sit
    # exactly on the kappa_z ceiling -- so penalising it harder only trades
    # colour for amplitude the network already had.
    # That ceiling is CORRECT and should not be widened: releasing it on 294
    # lunation-stratified spectra moves the freed zodi up 0.193 dex while the
    # moon falls 0.107 and the diffuse block falls 0.435, with the total
    # continuum conserved to 0.5% and rms improving only 0.6%.  The excess
    # tracks the MOON, not the ecliptic -- partial rho(excess, FLI | B500) =
    # +0.733 against partial rho(excess, B500 | FLI) = -0.322, which has the
    # wrong sign for zodiacal light.  So on bright-moon rows the zodi target
    # is a constraint, exactly as the moon target is on moon-down rows, and
    # the matching fix is a ceiling RULE (needs the physics-only Z_pred in the
    # moon/zodi cache, which v1 does not carry -- it stores the
    # learned-parameter variant).
    # lambda 5 is near-optimal; moon-only 20 and 50 give +18.2% and +9.7%
    # moon gain against +24.2% at 5.  See ablations.describe("S2a_flux_amp_1").
    "flux_amp_lambda": {"moon": 5.0, "zodi": 0.0},
    "flux_amp_floor_frac": 0.05,   # eps = 5% of the group median amplitude

    # Per-pixel inverse-variance weighting of the flux-space term, 2026-09-08.
    # Until now that term used a plain mean over wavelength, i.e. it weighted
    # 3600 A -- where the total throughput is 4.3x worse than at 5000 A -- the
    # same as the middle of the b channel, even though a fixed flux error there
    # is far fewer photons.  The model is the standard-star sensitivity curves
    # in $LVMCORE_DIR/sensitivity: counts = flux/sens, so sigma_flux =
    # sqrt(flux * sens) and w = 1/(flux*sens), with `flux` the TOTAL observed
    # science flux because the whole photon budget sets the noise.  Weights are
    # normalised to mean 1 per ROW, so this changes the weighting ACROSS
    # wavelength without re-weighting rows against each other.
    # See mlp_predictor.noise for the per-arm normalisation, which the
    # published curves do not carry and which is only good to ~1.5x.
    # Set False to A/B against the unweighted term.
    "flux_pixel_weighting": True,
    "flux_pixel_weight_floor_frac": 0.05,   # variance floor, x the row median

    # Moon-down moon amplitude: DERIVED, not learned (2026-09-09).
    # Where the moon-share bracket has collapsed onto amp_prior_floor the
    # decomposition does not measure a moon amplitude at all.  The physics-only
    # model predicts a moon fraction of ~3e-5 in deep dark time, so the
    # bracket sits at the floor (0.02) and the QP sits on that ceiling:
    # measured on gaia-stars-mask with the exact per-row design, 87.5% of
    # moon-down sci rows are pinned, i.e.
    # A_moon = R * A_zodi with R fitted per run.  That rule reproduces the
    # true moon-down amplitude to MAD 0.00051 dex over 7189 rows, against the
    # network's 0.0455 -- the target simply carries no information, and this
    # is 49.7% of the corpus.
    # The shape is not learnable either (93.9% of adjacent Moon_bs knot pairs
    # sit on a beta ratio bound, a corner of the feasible polytope; the SCI
    # and NEAR fits of the SAME exposure disagree at L1 0.311 against a
    # shuffled null of 0.612) but it is only 1.04% of the fitted continuum
    # there, so it is left in the loss rather than dropped.
    #   moon_down_amp_free -> the flux loss is made amplitude-blind on these
    #     rows (prediction rescaled to the true integral, log-amp term
    #     dropped), so only the shape trains.
    #   moon_down_amp_rule -> predict_sci_coefficients_default restores the
    #     amplitude analytically from the PREDICTED zodi.
    # Set both False to A/B against the pre-2026-09-09 behaviour.
    # GATE, revised 2026-09-09 after the first run.  `moon_alt <= 0` is the
    # wrong criterion: scattered moonlight with the moon just below the
    # horizon is real, and the model carries it as
    # horizon_scale = exp(-8.2182 * tanh(depth / 5 deg)) -- only -0.70 dex at
    # -1 deg, -2.72 at -5, saturating at -3.57 near -15.  The rule is valid
    # exactly where the moon-share bracket has collapsed onto amp_prior_floor,
    # i.e. where amp_prior_tol * moon_frac_po <= amp_prior_floor, so the gate
    # tests THAT: moon_frac_po <= 0.02/3.  It beats every altitude cut on both
    # coverage and exactness simultaneously:
    #     moon_alt <= 0            49.7% cover  92.6% exact  7.4% |d|>0.05
    #     moon_alt <= -4           47.0%        96.1%        3.9%
    #     moon_alt <= -12          41.4%        95.7%        4.2%
    #     moon_frac_po <= 0.00667  47.8%        96.2%        3.7%
    # No fixed altitude can match it because the depth term multiplies phase
    # and separation: a thin crescent 2 deg down contributes nothing while a
    # full moon 5 deg down still does.  The gated rows reach up to -0.69 deg
    # while the rows it drops run from -0.0 to -5.32 (median -1.43), and those
    # dropped rows were pinned only 0.4% of the time.
    # moon_down_alt_deg is now only the FALLBACK, used when a triplet has no
    # moon_frac_po (i.e. no v2 model cache).
    # The moon share on gated rows is BIMODAL, not constant: 96.25% sit on the
    # 0.02 ceiling and 3.25% want no moon at all (t = A_moon/(R A_zodi) < 1e-4),
    # with 0.49% in between.  A flat R over-predicts that second mode by the
    # whole component, 2.40% of the sci continuum.  The NEAR ARM knows which
    # mode a row is in -- 96.0% of zero-moon sci rows have the near arm also at
    # t < 0.5, against 0.72% of the pinned ones -- so transfer the measured
    # per-row ratio instead of assuming R:
    #     A_moon(sci) = clip( (A_moon/A_zodi)|near, 0, R ) * A_zodi(sci)
    # No threshold, no mode assignment.  Moon flux error as a fraction of the
    # sci continuum over 6914 gated rows:
    #     flat R * A_zodi      median 0.00086%  p99 2.4038%  >0.5% on 3.47%
    #     near ratio * A_zodi  median 0.00060%  p99 0.0107%  >0.5% on 0.56%
    #     arm-mean ratio       median 0.00059%  p99 0.3959%  >0.5% on 0.81%
    #     copy near verbatim   median 0.10913%  p99 0.9695%  >0.5% on 9.07%
    # i.e. 240 badly-wrong rows become 39, and it beats flat R on the PINNED
    # rows too because the measured ratio absorbs per-row variation.  Near
    # alone beats the arm mean (the far arm's own mode can differ), and
    # copying the near AMPLITUDE is much worse -- it is the RATIO that
    # transfers.  Honest cost: on 22.0% of rows the per-row ratio is noisier
    # than the robust median and slightly worse, median 0.0006% of continuum
    # but p99 1.04% -- those are the ~40 rows where the near arm is zero and
    # the science row is not.  Net absolute error over gated rows falls 87.8%.
    # Set False to A/B against the flat R.
    "moon_down_amp_free": True,
    "moon_down_amp_rule": True,
    "moon_down_ratio_transfer": True,
    "moon_down_frac_max": 0.02 / 3.0,
    "moon_down_alt_deg": 0.0,

    # Bright-moon zodi amplitude: DERIVED, not learned (2026-09-09).  The
    # mirror of the moon-down rule above, on the other half of the corpus.
    # 87.5% of moon-up sci rows sit exactly on the decomposition's Leinert
    # ceiling, so the target there is kappa_z * calibration * Z_pred.  That
    # ceiling is CORRECT: releasing kappa_z on 294 lunation-stratified spectra
    # only re-partitions a conserved continuum (zodi +0.193 dex, moon -0.107,
    # diffuse -0.435, total +0.005) for 0.6% of rms, and the excess tracks the
    # MOON not the ecliptic -- partial rho(excess, FLI | B500) = +0.733 against
    # partial rho(excess, B500 | FLI) = -0.322, the wrong sign for zodiacal
    # light.  So it is moon-into-zodi leakage the bracket exists to stop.
    # Two uses, measured on the FULL corpus (14 457 valid rows):
    #   CLAMP everywhere -- S * Z_pred is a hard upper bound (only 0.01% of
    #     rows exceed it at all, max excess 0.0024 dex), so clipping to it
    #     can only move a prediction toward the truth.
    #   SNAP on gated rows -- fraction of gated rows within 0.01 dex of the
    #     ceiling, with p95/p99 of |log10(S Z / A_zodi)| over them:
    #       moon_alt > 0        50.3% cover  83.9% exact  p95 0.310
    #       moon_frac_po > 0.5  43.8%        94.2%        p95 0.025
    #       moon_frac_po > 0.6  41.5%        97.2%        p95 0.0005
    #       moon_frac_po > 0.7  38.1%        98.6%        p95 0.0002
    #     The every10 subsample is optimistic by ~5 percentiles here and would
    #     have picked 0.5; these are the corpus numbers.
    # The snap is guarded by snap_frac: it fires only where the network
    # already predicts >= 90% of the ceiling, so gated rows that are genuinely
    # interior keep their own answer.  Scored over EVERY valid row with the
    # network simulated at its measured 0.0156 dex scatter:
    #       clamp only            MAD 0.00661  p99 0.0384  max 0.0750
    #       gate 0.6, snap 0.90   MAD 0.00190  p99 0.0375  max 0.0791
    #       no gate,  snap 0.80   MAD 0.00097  p99 0.0813  max 0.1548
    # i.e. 3.5x better MAD and a slightly better p99 for 0.05 dex on the
    # single worst row; dropping the gate trades MAD for a 2x worse p99.
    # zodi_ceiling_amp_free is deliberately False: unlike the moon-down case
    # the bright-moon zodi amplitude is NOT noise, and the network's learned
    # value is exactly what the guard uses to spot interior rows.
    # Needs moon/zodi model cache v2 (v1 stored only the learned-parameter
    # prediction, ~1.5x larger, which hid this effect entirely).
    "zodi_ceiling_rule": True,
    "zodi_ceiling_amp_free": False,
    "zodi_ceiling_gate_frac": 0.6,
    "zodi_ceiling_snap_frac": 0.9,

    "blend_init_alpha": 0.85,
    "alpha_lr_mult": 30.0,          # enough leverage to get the ctx-dependent alpha to learn efficiently
    
    # COEF_ERR weighting (§3.6.3).
    "coef_err_sigma_floor_rel": dict(trainer.DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),

    # 10-seed ensemble.
    "ensemble_seeds": (42, 43, 44, 45, 46, 47, 48, 49, 50, 51),
    # Train members in parallel worker processes (2026-09-18).  Members are
    # independent given their seed, and the parallel path is verified to give
    # byte-identical members in the requested order.  Worth ~1.5x on a real
    # 300-epoch run: one member already uses about half the GPU, so 2 workers
    # do not halve the time (4 would give ~1.9x).  Set to 1 to train
    # sequentially.  Start method is 'spawn' -- fork dies on a live MPS
    # context; a SCRIPT caller then needs an `if __name__ == "__main__":`
    # guard, a notebook does not.
    "ensemble_workers": 4,
    
    # Isolated zodi branch context restriction (§3.4.1).
    "zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep", "sun_alt", "alt",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        # 2026-09-09, and REQUIRED for the zodi ceiling rule to do anything:
        # the zodi head sits on an ISOLATED branch whose context is this
        # whitelist, so a feature missing here never reaches it.  On the 43%
        # of rows where the anchor binds, the target IS
        # S * 10**zodi_po_log10, so this is the target itself rather than a
        # correlate -- it supersedes `zodi_log10_v`, which is a Leinert
        # interpolation with none of the per-row LSF or airmass geometry.
        # `moon_frac_po` is the gate the rule uses, so the head sees the same
        # quantity that decides whether its answer will be snapped.
        "zodi_po_log10", "moon_frac_po",
    ),
    # Phase B (2026-08-26): moon-regime-conditioned isolated head for continuum.
    # Phase A' (2026-08-26): explicit interaction features added below (from
    # residual_ctx_attribution rankings; computed in data._augment_triplet_with_physics_priors).
    "continuum_ctx_restriction": (
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "airmass",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
    ),
    # Phase D (2026-08-26): widened continuum branch (only group with real
    # headroom above sky-arm floor; residual_ctx_attribution RF R^2 = 0.68).
    # Baseline (64, 32) with () extra layers -> 4k params on the branch.
    # Deployed (128, 64) with (64,) extra layer -> 16k params, +12k targeted.
    "continuum_branch_dims": (128, 64),
    "continuum_head_extra_dims": (64,),
    # Context-dependent per-group blend alpha (§3.5).  alpha is the pure
    # AMPLITUDE knob -- it interpolates the two arms' scores and cannot
    # change their colour.  Two things to know before editing this:
    #  * "moon_up_smooth" is IDENTICAL across near/far/sci (it describes the
    #    moon, not the pointing), so it carries no arm information at all.
    #    Median |near-far| as a fraction of each feature's own spread:
    #    moon_sep 0.88, ecl_beta_deg 1.01, airmass 0.65, moon_up_smooth 0.00.
    #  * the predictor reads sci_ctx ONLY.  Feeding it the arm contexts, and
    #    giving it depth, were both measured and both made the moon tail
    #    worse; alpha also already commands 1.3-1.5x the amplitude range the
    #    tail needs, so reach was never the limit.  See ablations.RETIRED.
    "alpha_ctx_features": (
        "moon_up_smooth", "ecl_beta_deg", "airmass",
    ),
    # Restrict ctx-alpha to the anisotropic groups; others keep scalar alpha.
    # Phase F (2026-08-27c): additive moon-zodi coupling.  Keeps both existing
    # paths intact (moon on shared trunk, zodi on isolated branch, continuum on
    # its own branch).  Adds a small coupling branch (64->32) computing a shared
    # latent from the moon-scatter + zodi-geometry ctx union; the latent is
    # projected additively into the moon head output (Linear(32, n_moon)) and
    # the zodi head output (Linear(32, n_zodi)) via zero-initialised linear
    # projectors, so training starts byte-identical to Phase A''.  This is the
    # "preserve zodi isolation while giving moon the shared context" alternative
    # to Phase E (`moon_zodi_mode="shared_branch"`), which was rejected on
    # 2026-08-27b for the atlas-mid regression it caused.  +9k params.
    # "moon_zodi_ctx_restriction": None, # Phase A baseline
    "moon_zodi_ctx_restriction": (
        "airmass", "vanrhijn_285km",
        "ecl_beta_deg", "ecl_lon_sin", "ecl_lon_cos",
        "zodi_log10_v", "sun_sep",
        "moon_alt", "moon_sep",
        "moon_phase_sin", "moon_phase_cos",
        "moon_fli", "moon_up_smooth",
        "moon_airmass_up", "moon_signal_proxy",
        "moon_fli_x_phase_cos",
        "moon_sig_x_lon_cos", "moon_sig_x_lon_sin",
        # 2026-09-09: the physics-only pair, added here too because this
        # branch's whole job is the moon/zodi partition and `moon_frac_po` is
        # exactly the quantity the decomposition's share bracket is stated in.
        # Not speculative: it upgrades `zodi_log10_v` and `moon_signal_proxy`,
        # both already listed, with the same geometry done properly.
        "zodi_po_log10", "moon_frac_po",
    ),
    "moon_zodi_coupling_dims": (64, 32),
    # Off-switch for the coupling above, added 2026-09-04 (the no-coupling
    # path had been deleted, so the old "set moon_zodi_* to None" A/B raised).
    # Ablation A1 kept it ON: disabling costs the moon 4.5pp of its gain over
    # B0_copy_near (+24.0% -> +19.5%) while zodi is flat, so the coupling is
    # the moon-context fix its docstring claims, NOT compensation for the
    # moon/zodi reversals the decomposition used to produce.
    # (Phase E's shared_branch knobs lived here; removed with the other dead
    #  keys below -- the code path was deleted on 2026-08-27.)

    # ------------------------------------------------------------------
    # Removed 2026-09-04: 14 config keys the Trainer does not read.  It
    # warned about them on every run ("NOT read by the trainer and will have
    # no effect"); they were left behind by the 2026-08-27 trim, which deleted
    # the code paths but not the config entries.  Two were active hazards for
    # the post-reversal simplification pass: moon_zodi_mode looked like the
    # coupling off-switch and was not (a real one, moon_zodi_coupling_enabled,
    # had to be added), and alpha_ctx_groups looked like it restricted
    # ctx-alpha but the model hardcodes _ALPHA_CTX_GROUPS = ("moon", "zodi",
    # "continuum") -- same value, so behaviour is unchanged.  An A/B driven
    # through any of these would have measured nothing.  The list lives in
    # mlp_predictor.ablations.DEAD_CONFIG_KEYS; ablations.apply() now refuses
    # any override the trainer does not consume.
    #   alpha_ctx_groups, block_cov_loss_groups, flux_mse_eps_frac,
    #   head_extra_dims, high_airmass_boost, moon_down_ecliptic_beta_deg,
    #   moon_down_ecliptic_boost, moon_zodi_branch_dims, moon_zodi_mode,
    #   moon_zodi_moon_head_extra_dims, moon_zodi_zodi_head_extra_dims,
    #   relative_mse_eps_frac, relative_mse_groups, use_coef_err_weights
    # NOTE use_coef_err_weights was hardcoded True and coef_err_sigma_floor_rel
    # (below) is live, so COEF_ERR weighting remains active.
    # ------------------------------------------------------------------
}
print(f"train_cfg: {len(train_cfg)} knobs, {len(train_cfg['ensemble_seeds'])}-seed ensemble, "
      f"flux_mse_groups={train_cfg['flux_mse_groups']}, "
      f"flux_amp_lambda={train_cfg['flux_amp_lambda']}")


In [ ]:
# --- Fit the ensemble, or restore a previously saved one ---------------------
# Gate: when a saved ensemble exists at SAVED_ENSEMBLE_PATH it is deserialised
# and training is skipped entirely; otherwise the ensemble is trained from
# train_cfg above and the next cell writes it to disk.  Set FORCE_RETRAIN=True
# to retrain even when a saved file is present (e.g. after editing train_cfg).
from pathlib import Path as _Path

from mlp_predictor import serialization

# SAVED_ENSEMBLE_PATH is defined in the imports cell next to DECOMP_DATA_ROOT,
# so the model always lands beside the corpus it was trained on.
# TRUE for this run: flux_amp_lambda changed in the cell above, and that
# changes only the LOSS, not the architecture -- so the saved ensemble would
# deserialise cleanly and you would silently evaluate the OLD weights while
# the new config printed above.  Set back to False once the retrain is saved.
# TRUE for this run: flux_mse_groups changed above.  That changes only the
# LOSS, not the architecture, so the saved ensemble would deserialise
# cleanly and you would silently evaluate the OLD weights against the new
# config.  Set back to False once the retrain is saved.
# TRUE for this run: flux_pixel_weighting was added above (2026-09-08).  Like
# every other flux-loss knob it changes only the LOSS, not the architecture, so
# the saved ensemble would deserialise cleanly and you would silently evaluate
# the OLD weights while the new config printed above.  Set back to False once
# the retrain is saved.
FORCE_RETRAIN = False

ENSEMBLE_WAS_LOADED = (
    _Path(SAVED_ENSEMBLE_PATH).expanduser().exists() and not FORCE_RETRAIN
)

if ENSEMBLE_WAS_LOADED:
    mlp_artifacts = serialization.load_ensemble(SAVED_ENSEMBLE_PATH)
    _ensemble_members = mlp_artifacts["members"]
    # Training-time only; save_ensemble deliberately does not persist it.
    per_seed_test_metrics = None

    # Refuse a model built against a different corpus or ctx layout -- the
    # network's inputs are positional, so a silent mismatch would produce
    # plausible-looking nonsense rather than an error.
    _saved_coef = list(mlp_artifacts["coef_names"])
    _live_coef = [str(n) for n in filtered_triplet["coef_names"]]
    if _saved_coef != _live_coef:
        raise RuntimeError(
            f"Saved ensemble was trained on a different coefficient basis: "
            f"{len(_saved_coef)} names vs {len(_live_coef)} in filtered_triplet. "
            f"Set FORCE_RETRAIN=True or point SAVED_ENSEMBLE_PATH elsewhere.")
    if list(mlp_artifacts["ctx_names"]) != list(filtered_triplet["ctx_names"]):
        raise RuntimeError(
            "Saved ensemble ctx_names differ from filtered_triplet['ctx_names'] "
            "(context augment changed since training). Set FORCE_RETRAIN=True.")

    # The saved payload omits the train/val/test split by design, but the split
    # is exactly reproducible: Trainer.run_ensemble trains every member on
    # filtered_triplet['compress_*_idx'], which cell 8 derives deterministically
    # from (obstime_mjd, moon_phase, seed=42).
    train_idx = np.asarray(filtered_triplet["compress_train_idx"], dtype=int)
    val_idx = np.asarray(filtered_triplet["compress_val_idx"], dtype=int)
    test_idx = np.asarray(filtered_triplet["compress_test_idx"], dtype=int)
    mlp_artifacts["train_idx"] = train_idx
    mlp_artifacts["val_idx"] = val_idx
    mlp_artifacts["test_idx"] = test_idx

    # Prefer the compressor / geometry / grouping the model was TRAINED with over
    # the freshly fitted ones from cell 8: the network's scores are only
    # meaningful under its own compressor.
    group_compressors = mlp_artifacts["compressors"]
    compress_geom_kwargs = mlp_artifacts["geom_kwargs"]
    group_indices = mlp_artifacts["group_indices"]

    print(f"Restored {len(_ensemble_members)}-member ensemble from "
          f"{SAVED_ENSEMBLE_PATH}; training skipped.")
    print(f"  seeds={list(mlp_artifacts['seeds'])}  "
          f"split: train={train_idx.size} val={val_idx.size} test={test_idx.size}")
    print("  using the saved compressors / geom_kwargs / group_indices "
          "(cell 8's fitted copies are overridden).")
    print("  NB per-seed test metrics and training history are not persisted, so "
          "the training-log")
    print("     tables in this cell are unavailable on the load path.")
else:
    _trainer = trainer.Trainer(cfg=train_cfg)
    artifacts = _trainer.run_ensemble(
        filtered_triplet, group_compressors, group_indices, compress_geom_kwargs,
        input_fits_for_basis=cfg.data.input_fits_for_basis,
        # Full-corpus stack: the only file with FLUX_SCI for the training rows,
        # needed by the photon-noise pixel weights on the flux-space loss.
        input_fits_flux=cfg.data.input_fits_flux,
        n_moon_knots=N_MOON_KNOTS, split_zodi=SPLIT_ZODI, n_zodi_knots=N_ZODI_KNOTS,
        # Build the flux basis the corpus was FITTED with. Without this the
        # trainer falls back to a split-zodi basis at lsf_sigma=1.0 A, which
        # matches no deployed corpus: against this corpus's own fitted LSF
        # surface its OH template integrals are 2.2x off and ATOM/Orc 1.9x.
        decomp_suffix=cfg.data.decomp_suffix,
        verbose=True,
    )
    mlp_artifacts = artifacts.mlp_artifacts
    _ensemble_members = artifacts.members
    per_seed_test_metrics = artifacts.per_seed_test_metrics
    train_idx = np.asarray(mlp_artifacts["train_idx"], dtype=int)
    val_idx = np.asarray(mlp_artifacts["val_idx"], dtype=int)
    test_idx = np.asarray(mlp_artifacts["test_idx"], dtype=int)

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all  = np.asarray(filtered_triplet["coef_far"],  dtype=np.float32)
coef_sci_all  = np.asarray(filtered_triplet["coef_sci"],  dtype=np.float32)
ctx_near_all  = np.asarray(filtered_triplet["ctx_near"],  dtype=np.float32)
ctx_far_all   = np.asarray(filtered_triplet["ctx_far"],   dtype=np.float32)
ctx_sci_all   = np.asarray(filtered_triplet["ctx_sci"],   dtype=np.float32)
coef_err_near_all = np.asarray(
    filtered_triplet.get("coef_err_near", np.full_like(coef_near_all, np.nan)), dtype=np.float32)
coef_err_far_all = np.asarray(
    filtered_triplet.get("coef_err_far", np.full_like(coef_far_all, np.nan)), dtype=np.float32)
coef_err_sci_all = np.asarray(
    filtered_triplet.get("coef_err_sci", np.full_like(coef_sci_all, np.nan)), dtype=np.float32)

coef_pred_det = trainer.predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all[test_idx],
    coef_far_phys=coef_far_all[test_idx],
    ctx_near_phys=ctx_near_all[test_idx],
    ctx_far_phys=ctx_far_all[test_idx],
    ctx_sci_phys=ctx_sci_all[test_idx],
).astype(np.float32)


In [ ]:
# --- Persist the trained ensemble to disk for inference ---
# See mlp_predictor.serialization + mlp_predictor.inference for the loader
# and the minimal-input predict API; consumed by
# notebook_example_predict_sky.ipynb.
# SAVED_ENSEMBLE_PATH is defined by the gate in the cell above.  Skipped when
# that gate restored an existing ensemble: there is nothing new to write, and
# re-saving would rewrite the file we just read.
if ENSEMBLE_WAS_LOADED:
    print(f"[save_ensemble] skipped: ensemble was restored from "
          f"{SAVED_ENSEMBLE_PATH} (set FORCE_RETRAIN=True above to retrain).")
else:
    serialization.save_ensemble(mlp_artifacts, SAVED_ENSEMBLE_PATH)


In [ ]:
# --- Training progress against epoch -----------------------------------------
# Targets: per-seed train/val loss, and the near-arm blend alpha per group.
# Look for: val still falling at the last epoch => n_epochs is the binding
#           constraint, not patience; best-epoch markers far from the end =>
#           patience fired inside plateau noise and early weights were kept;
#           an alpha that never leaves its init => that group is not learning
#           its near/far blend (see alpha_lr_mult in train_cfg).
# Recomputes nothing: it reads the history the trainer recorded, which
# save_ensemble has persisted since 2026-09-23, so it behaves identically on a
# freshly trained ensemble and on a restored one.  An ensemble saved before
# that date carries no history; the cell says so and skips.
# Placed after the save cell on purpose, so a retrain is always written to disk
# before any diagnostic can fail.
from mlp_predictor import diagnostics as _diag_th

_tmp = _diag_th.Diagnostics(_diag_th.DiagnosticsContext(
    mlp_artifacts=mlp_artifacts,
    ensemble_members=_ensemble_members,
)).training_history()

In [ ]:
# --- Build the diagnostics context ---
from mlp_predictor import diagnostics  # local import: survives kernel-restart edge cases

diag_ctx = diagnostics.DiagnosticsContext(
    filtered_triplet=filtered_triplet,
    mlp_artifacts=mlp_artifacts,
    group_compressors=group_compressors,
    group_indices=group_indices,
    geom_kwargs=compress_geom_kwargs,
    ensemble_members=_ensemble_members,
    coef_pred_det=coef_pred_det,
    coef_near_all=coef_near_all,
    coef_far_all=coef_far_all,
    coef_sci_all=coef_sci_all,
    ctx_near_all=ctx_near_all,
    ctx_far_all=ctx_far_all,
    ctx_sci_all=ctx_sci_all,
    coef_err_near_all=coef_err_near_all,
    coef_err_far_all=coef_err_far_all,
    coef_err_sci_all=coef_err_sci_all,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    # Pre-filter triplet + notebook-only bare names the extracted cell bodies read.
    extras={
        "triplet": triplet,
        "coef_wavelengths_a": ext.coef_wavelengths_a,
        "context_cols": list(cfg.data.context_columns),
        "FACTOR": FACTOR,
        "DECOMP_DATA_ROOT": cfg.data.decomp_data_root,
        "DECOMP_STEM": cfg.data.decomp_stem,
        # Moon-model ctx feature: the diagnostic cells rebuild their own
        # triplets, so they must apply the SAME augments or ctx_names will not
        # match the trained ensemble.
        "USE_MOON_MODEL_FEATURE": USE_MOON_MODEL_FEATURE,
        "MOON_MODEL_CORPUS_PREFIX": cfg.data.decomp_prefix,
        "_DECOMP_SUFFIX": cfg.data.decomp_suffix,
        # Per-row telluric transmission for the reconstruction, or None for the
        # production split-zodi variant.  Bound to the EVERY10 stack because
        # that is the file every reconstructing diagnostic indexes.  Built here
        # rather than per cell so the ~100 MB transmission model is loaded once.
        "TELLURIC_ROW_FOR": (
            data.make_telluric_row_lookup(cfg.data.input_fits_for_basis,
                                          decomp_suffix=cfg.data.decomp_suffix)
            if USE_TELLURIC_RECON else None),
        # One representative-row telluric bundle for the things that need a
        # SINGLE basis for the whole corpus -- the integrated-amplitude weights
        # v_g and the flux-space mesospheric companion.  Moon/zodi/diffuse are
        # identical between variants, but OH is not, so these must not be built
        # on the split-zodi basis when the corpus is telluric.
        "TELLURIC_BASIS_KW": (
            data.make_telluric_row_lookup(
                cfg.data.input_fits_for_basis, verbose=False,
                decomp_suffix=cfg.data.decomp_suffix)(
                    'sci', data.telluric_representative_row(
                        cfg.data.input_fits_for_basis))
            if USE_TELLURIC_RECON else None),
        # Spline-knot counts read by full_spectrum_* cell bodies.
        "N_MOON_KNOTS": N_MOON_KNOTS,
        "SPLIT_ZODI": SPLIT_ZODI,
        "N_ZODI_KNOTS": N_ZODI_KNOTS,
        # Per-seed test metrics DataFrame consumed by naive_baseline.
        # None on the load path: per-seed metrics are not persisted.
        "cmp_df": per_seed_test_metrics,
    },
)
diag = diagnostics.Diagnostics(diag_ctx)


## Diagnostics

Ten cells, down from twenty-eight.  `headline_summary` reads the
persisted results of the three cells above it rather than
recomputing, so it cannot disagree with them.  The cells removed on
2026-09-09 were the uncertainty-calibration family (five cells whose
conclusion was always "ensemble spread under-estimates, use
`ensemble_spread_calibration`"), the per-seed and worst-row-stability
checks, three WRMSE-vs-context cells that were never run, and
`residual_ctx_attribution`, whose finding is closed and recorded in
the changelog.  They live on in `mlp_predictor.diagnostics` and can be
called from a scratch cell if a specific question needs them.


In [ ]:
# Targets: ML vs copy_near / near_geo / mean_geo baselines, per group per regime,
#          in TWO spaces: coefficient sRMSE and integrated FLUX amplitude |dlog10|.
# Look for: ML should beat B2_mean_geo on moon+zodi; look at moon_up / moon_down / close_zodi.
# When the two spaces DISAGREE, believe the flux one -- the coefficient sRMSE is an
# unweighted sum over a group, so a 358-coefficient block is owned by whichever few
# coefficients are numerically largest (measured: 93-95% of the ML's mesospheric MSE
# comes from 5 of 358 sticks).  That is why mesospheric reads -12.3% in coefficients
# and +7.5% in flux on gaia-stars-mask-telluric-chi2, and why the group-equal verdict
# is printed in both spaces at the bottom of the cell.
_tmp = diag.naive_baseline()


In [ ]:
# Targets: per-family AMPLITUDE error vs the context that drives that family.
# Amplitude A = sum_lambda f(lambda) = c . B.sum(axis=1), a linear functional of
# the coefficients; error Delta = log10(A_pred/A_true), because the miss is
# multiplicative (83-89% of the tail MSE is removed by a single per-row rescale).
# Four grids: moon vs moon geometry, zodi vs ecliptic geometry, the diffuse
# continuum (HO2/FeO/O2Ac, coloured) and OH (by wavelength band, plus O2_b01)
# vs the airglow and solar-activity terms.
# Look for: a trace departing from zero in some regime = a bias the head has not
# learned.  Flat at zero = that family's brightness is transferred correctly.
_tmp = diag.amplitude_error_vs_ctx()


In [ ]:
# Targets: per-row flux-space RMSE (near/far/sci) + absolute photon chi2.
# Look for: sci pRMSE distribution; per-component residual per row.
# size    = rows evaluated.  None (the default) takes EVERY held-out row
#           that passes the gates -- 2900 on the 1.3.2 corpus.  All of them
#           feed every statistic.  An integer caps it, drawn phase-stratified.
# split   = which rows: 'heldout' (validation + test, the default), 'test',
#           'val', or 'all'.  Before 2026-09-23 this cell drew from every
#           gated every10 row -- ~79% training data, and only a tenth of
#           the split even so; 'all' restores the training rows.
# n_workers = processes for the reconstruction loop.  2900 rows take ~5 min
#           on 8 workers against ~30 min serial; the pool falls back to
#           serial on its own if forking fails.  Reads the CORPUS stack,
#           not every10, so it sees the whole split.
# stroked = rows DRAWN as lines; the memory knob, ~0.8 MB each.  Uncapped
#           at size=500 the figure is ~758 MB of JSON and kills the kernel.
# NB do NOT call diag._init_globals() here: it REPLACES the shared
# exec-globals, discarding naive_baseline_result and
# amplitude_error_vs_ctx_result from the cells above, which is exactly what
# headline_summary reads.  Diagnostics.__init__ already initialises them,
# and _run self-heals after an %autoreload.
_tmp = diag.full_spectrum_batch_rmse(stroked=200, n_workers=8)

In [ ]:
# Targets: the 10 worst reconstructions from the batch above.
# Look for: which regimes (moon-up, bright-moon, high-|beta_ecl|, twilight) dominate the tail.
_tmp = diag.worst_recon()


In [ ]:
# Targets: one consolidated table; run after the three cells above it.
# Look for: any group with gain <= 0 (ML losing to a naive baseline); moon-down
#           MAD far above moon-up; reduced chi2 moved beyond seed noise.
# Recomputes nothing -- reads naive_baseline_result, amplitude_error_vs_ctx_result
# and rmse_subset_results, so it cannot disagree with the cells it summarises.
_tmp = diag.headline_summary()

## Full-spectrum verification


In [ ]:
# Targets: reconstruct one every10 row from pred vs true coefs.
# Look for: quality of the fit; residual panels break out moon / zodi / lines / diffuse.
REQUESTED_ROW = 1368
_tmp = diag.full_spectrum_single_row(row=REQUESTED_ROW, show_moon_zodi_model=True)


## Regime breakdowns: lunation, moon state, zodi


In [ ]:
# Targets: ML per-lunation mean/max error trend.
# Look for: monotone drift across lunations => solar-activity / seasonal missing feature.
_tmp = diag.per_lunation_drift()


In [ ]:
# Targets: zodi mean bias per moon-state regime, separately for near and far arms.
# Look for: asymmetric bias => zodi head is being pushed by the wrong arm.
_tmp = diag.sky_arm_zodi_bias()


## Wavelength systematics and uncertainty


In [ ]:
# Targets: aggregated (pred - true)(lambda), split moon-down / moon-up,
#          absolute + fractional, each against ITS OWN 1-fibre photon noise.
# Look for: median line leaving the green noise band = real error; coherent
#          bumps flag specific bands; fractional bias signals colour miscalibration.
# size = rows reconstructed; the split halves what each panel sees, so 500
#        leaves ~250 per regime.  Cost ~0.7 MB and 2 LSF recons per row.
_tmp = diag.wavelength_residual_atlas(size=500, seed=42)

In [ ]:
# Targets: ensemble std vs actual |pred - true| per group + reliability curve.
# Look for: RMS ratio ~ 1 + Kendall tau > 0.3 => trustworthy sigma; ratio >> 1 => under-diverse ensemble.
# Uncertainty trust: ensemble std vs actual |pred - true| per group.
_tmp = diag.ensemble_spread_calibration()
